# Pengembangan dan Evaluasi Algoritma Pencarian WisataFinder

## Deskripsi Proyek

WisataFinder merupakan mesin pencari destinasi wisata di Indonesia yang dikembangkan menggunakan Python dan Streamlit.

Proyek ini bertujuan membantu pengguna menemukan destinasi wisata berdasarkan kata kunci, kategori, kota, dan provinsi.

## Tujuan Pengembangan

1. Mengembangkan sistem pencarian destinasi wisata berbasis TF-IDF dan Cosine Similarity.
2. Meningkatkan relevansi hasil pencarian berdasarkan kategori dan lokasi.
3. Mengembangkan pengenalan kota dan provinsi secara otomatis.
4. Mengevaluasi performa beberapa versi algoritma pencarian.
5. Menentukan algoritma yang paling sesuai untuk diintegrasikan ke aplikasi WisataFinder.

## Dataset

Dataset nasional berisi 1.025 destinasi wisata dengan cakupan 37 label provinsi.

## Metode

- Text Preprocessing
- TF-IDF
- Cosine Similarity
- Filter Kategori
- Pengenalan Lokasi Otomatis
- Precision@10

In [1]:
import sys
import os
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import re

print("Python executable:", sys.executable)
print("Versi Python:", sys.version)
print("Folder kerja:", os.getcwd())

Python executable: D:\Anaconda3\envs\ds_siska\python.exe
Versi Python: 3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]
Folder kerja: D:\WisataFinder\notebooks


In [2]:
BASE_DIR = Path.cwd()

if BASE_DIR.name.lower() == "notebooks":
    BASE_DIR = BASE_DIR.parent

print("Folder proyek:", BASE_DIR)

file_dataset = BASE_DIR / "data" / "processed" / "destinations_nasional.csv"

file_perbandingan = BASE_DIR / "data" / "processed" / "perbandingan_algoritma_v2_v7.csv"

file_provinsi = BASE_DIR / "data" / "processed" / "evaluasi_provinsi_v7.csv"

print("\nPemeriksaan file:")
print("Dataset nasional:", file_dataset.exists())
print("Perbandingan algoritma:", file_perbandingan.exists())
print("Evaluasi provinsi:", file_provinsi.exists())

Folder proyek: D:\WisataFinder

Pemeriksaan file:
Dataset nasional: True
Perbandingan algoritma: True
Evaluasi provinsi: True


In [3]:
df_nasional = pd.read_csv(file_dataset)

print("Ukuran dataset:", df_nasional.shape)
print("\nDaftar kolom:")
print(df_nasional.columns.tolist())

display(df_nasional.head())

Ukuran dataset: (1025, 23)

Daftar kolom:
['Place_Id', 'Place_Name', 'Description', 'Category', 'City', 'Province', 'Address', 'Price', 'Rating', 'Time_Minutes', 'Lat', 'Long', 'Image_URL', 'Location_Label', 'Location_Verification', 'Description_Quality', 'City_Verification', 'Province_Verification', 'Coordinate_Verification', 'Source', 'Original_Name', 'Original_Image_Path', 'search_text']


,Place_Id,Place_Name,Description,Category,City,Province,Address,Price,Rating,Time_Minutes,...,Location_Label,Location_Verification,Description_Quality,City_Verification,Province_Verification,Coordinate_Verification,Source,Original_Name,Original_Image_Path,search_text
0,IDN-0001,Air Terjun Aling-Aling,Deskripsi tidak ditemukan,air terjun,Buleleng,Bali,"Air Terjun Aling-Aling, Panji Anom, Buleleng, ...",NaN,NaN,NaN,...,Buleleng,Belum diverifikasi,Tidak informatif,Tercatat pada dataset,Perlu pemeriksaan lanjutan,Koordinat unik,Kaggle - Indonesia Tourism,"Air Terjun Aling-Aling, Buleleng, Bali",gambar_wisata/Air_Terjun_Aling-Aling,air terjun aling-aling air terjun deskripsi ti...
1,IDN-0002,Air Terjun Bantimurung,Air Terjun Bantimurung adalah salah satu air t...,air terjun,Sulawesi,Sulawesi Selatan,"Air Terjun Bantimurung, Jalan Poros Bantimurun...",NaN,NaN,NaN,...,Sulawesi,Perlu verifikasi: label wilayah umum,Informatif,Perlu verifikasi,Perlu pemeriksaan lanjutan,Koordinat unik,Kaggle - Indonesia Tourism,"Air Terjun Bantimurung, Sulawesi, Sulawesi Sel...",gambar_wisata/Air_Terjun_Bantimurung,air terjun bantimurung air terjun air terjun b...
2,IDN-0003,Air Terjun Benang Kelambu,Deskripsi tidak ditemukan,air terjun,Lombok Tengah,Nusa Tenggara Barat,"Air Terjun Benang Kelambu, Pemotoh, Lombok Ten...",NaN,NaN,NaN,...,Lombok Tengah,Belum diverifikasi,Tidak informatif,Tercatat pada dataset,Perlu pemeriksaan lanjutan,Koordinat unik,Kaggle - Indonesia Tourism,"Air Terjun Benang Kelambu, Lombok Tengah, Nusa...",gambar_wisata/Air_Terjun_Benang_Kelambu,air terjun benang kelambu air terjun deskripsi...
3,IDN-0004,Air Terjun Benang Stokel,Deskripsi tidak ditemukan,air terjun,Lombok Tengah,Nusa Tenggara Barat,"Air Terjun Benang Stokel, Pemotoh, Lombok Teng...",NaN,NaN,NaN,...,Lombok Tengah,Belum diverifikasi,Tidak informatif,Tercatat pada dataset,Perlu pemeriksaan lanjutan,Koordinat unik,Kaggle - Indonesia Tourism,"Air Terjun Benang Stokel, Lombok Tengah, Nusa ...",gambar_wisata/Air_Terjun_Benang_Stokel,air terjun benang stokel air terjun deskripsi ...
4,IDN-0005,Air Terjun Bidadari,Daftar ini tidak lengkap dan hanya dapat dijad...,air terjun,Belitung Timur,Kepulauan Bangka Belitung,"Air Terjun Bidadari, Nyuruk, Belitung Timur, K...",NaN,NaN,NaN,...,Belitung Timur,Belum diverifikasi,Tidak informatif,Tercatat pada dataset,Perlu pemeriksaan lanjutan,Koordinat unik,Kaggle - Indonesia Tourism,"Air Terjun Bidadari, Belitung Timur, Kepulauan...",gambar_wisata/Air_Terjun_Bidadari,air terjun bidadari air terjun daftar ini tida...


## 2. Eksplorasi dan Karakteristik Dataset Nasional

Dataset yang digunakan dalam pengembangan WisataFinder merupakan dataset Indonesia Tourism yang diperoleh dari Kaggle.

Dataset ini berisi 1.025 destinasi wisata dengan 23 kolom yang mencakup nama destinasi, deskripsi, kategori, kota, provinsi, alamat, koordinat, dan informasi gambar.

Dataset digunakan sebagai dasar pengembangan mesin pencari destinasi wisata nasional.

In [4]:
# Ringkasan karakteristik dataset

ringkasan_dataset = pd.DataFrame({
    "Indikator": [
        "Jumlah destinasi wisata",
        "Jumlah kolom",
        "Jumlah ID duplikat",
        "Jumlah provinsi",
        "Jumlah kategori",
        "Kota kosong",
        "Deskripsi kosong",
        "Deskripsi tidak informatif",
        "Data Price tersedia",
        "Data Rating tersedia",
        "Data Time_Minutes tersedia"
    ],
    "Nilai": [
        len(df_nasional),
        len(df_nasional.columns),
        df_nasional["Place_Id"].duplicated().sum(),
        df_nasional["Province"].nunique(),
        df_nasional["Category"].nunique(),
        df_nasional["City"].isna().sum(),
        df_nasional["Description"].isna().sum(),
        df_nasional["Description_Quality"].eq("Tidak informatif").sum(),
        df_nasional["Price"].notna().sum(),
        df_nasional["Rating"].notna().sum(),
        df_nasional["Time_Minutes"].notna().sum()
    ]
})

display(ringkasan_dataset)

,Indikator,Nilai
0,Jumlah destinasi wisata,1025
1,Jumlah kolom,23
2,Jumlah ID duplikat,0
3,Jumlah provinsi,37
4,Jumlah kategori,22
5,Kota kosong,18
6,Deskripsi kosong,27
7,Deskripsi tidak informatif,177
8,Data Price tersedia,0
9,Data Rating tersedia,0


In [5]:
distribusi_kategori = (
    df_nasional["Category"]
    .value_counts()
    .rename_axis("Kategori")
    .reset_index(name="Jumlah Destinasi")
)

display(distribusi_kategori)

,Kategori,Jumlah Destinasi
0,wisata alam,179
1,pantai,104
2,museum,90
3,mall,77
4,gunung,74
5,wisata kuliner,67
6,taman,65
7,alun-alun,55
8,wisata religi,49
9,bukit,47


In [6]:
distribusi_provinsi = (
    df_nasional["Province"]
    .value_counts()
    .rename_axis("Provinsi")
    .reset_index(name="Jumlah Destinasi")
)

display(distribusi_provinsi)

,Provinsi,Jumlah Destinasi
0,Jawa Timur,120
1,Jawa Barat,105
2,Jawa Tengah,99
3,Bali,98
4,Daerah Istimewa Yogyakarta,80
5,DKI Jakarta,79
6,Nusa Tenggara Barat,52
7,Kalimantan Timur,42
8,Sulawesi Selatan,38
9,Nusa Tenggara Timur,32


In [7]:
df_perbandingan = pd.read_csv(file_perbandingan)

df_evaluasi_provinsi = pd.read_csv(file_provinsi)

print("Perbandingan algoritma:")
display(df_perbandingan)

print("Evaluasi 37 provinsi:")
display(df_evaluasi_provinsi)

Perbandingan algoritma:


,Versi,Hasil_Bandung_Total,Hasil_Kota_Bandung,Hasil_Provinsi_Jawa_Barat,Hasil_Papua_Total,Hasil_Papua_Barat,Hasil_Atlantis,Hasil_Kepri
0,V2,10,3,4,10,0,10,10
1,V3,10,3,4,10,0,10,10
2,V4,10,3,4,10,0,10,10
3,V5,4,4,4,10,0,10,10
4,V6,4,4,4,3,3,10,0
5,V7,4,4,4,3,3,0,0


Evaluasi 37 provinsi:


,Provinsi,Jumlah_Hasil,Hasil_Sesuai,Akurasi_Lokasi_Persen
0,Daerah Istimewa Yogyakarta,10,10,100.0
1,Kepulauan Bangka Belitung,10,10,100.0
2,Nusa Tenggara Barat,10,10,100.0
3,Nusa Tenggara Timur,10,10,100.0
4,Kalimantan Selatan,7,7,100.0
5,Sulawesi Tenggara,10,10,100.0
6,Kalimantan Tengah,10,10,100.0
7,Sulawesi Selatan,10,10,100.0
8,Kalimantan Barat,10,10,100.0
9,Sumatera Selatan,10,10,100.0



## 3. Analisis Hasil Eksplorasi Dataset

Berdasarkan hasil eksplorasi, dataset nasional WisataFinder memiliki 1.025 data destinasi wisata dengan 23 atribut dan 37 label provinsi. Dataset tidak memiliki ID destinasi yang duplikat dan memiliki 22 kategori wisata.

### 3.1 Kualitas Data

Hasil pemeriksaan menunjukkan terdapat 18 data kota kosong, 27 deskripsi kosong, dan 177 deskripsi yang tidak informatif. Selain itu, atribut Price, Rating, dan Time_Minutes belum memiliki nilai yang tersedia.

Kondisi tersebut menunjukkan bahwa dataset masih memiliki keterbatasan kualitas informasi. Oleh karena itu, sistem pencarian difokuskan pada atribut nama destinasi, kategori, deskripsi, kota, dan provinsi.

### 3.2 Distribusi Kategori

Kategori wisata alam memiliki jumlah terbanyak, yaitu 179 destinasi. Sementara itu, wisata lampion memiliki jumlah paling sedikit, yaitu 2 destinasi.

Distribusi kategori yang tidak merata dapat memengaruhi variasi hasil pencarian karena beberapa kategori memiliki jumlah data yang lebih sedikit.

### 3.3 Distribusi Provinsi

Dataset mencakup 37 label provinsi dengan jumlah destinasi yang berbeda-beda. Jawa Timur memiliki jumlah terbanyak, yaitu 120 destinasi, sedangkan Papua memiliki jumlah paling sedikit, yaitu 1 destinasi.

Hal ini menunjukkan bahwa cakupan dataset nasional belum merata. Hasil pencarian pada provinsi tertentu dapat lebih terbatas dibandingkan provinsi dengan jumlah data yang lebih banyak.

### 3.4 Kesimpulan Eksplorasi

Dataset nasional dapat digunakan sebagai dasar pengembangan mesin pencari wisata berbasis teks. Namun, keterbatasan deskripsi, data lokasi, dan ketimpangan distribusi destinasi perlu diperhatikan dalam proses pengembangan dan evaluasi algoritma.



## 4. Metode Pencarian Destinasi Wisata

WisataFinder menggunakan pendekatan Information Retrieval untuk menemukan destinasi wisata berdasarkan kemiripan antara kata kunci pencarian pengguna dan informasi teks pada dataset.

Metode utama yang digunakan adalah TF-IDF (Term Frequency-Inverse Document Frequency) dan Cosine Similarity.

### 4.1 Text Preprocessing

Tahap preprocessing bertujuan mempersiapkan informasi teks sebelum digunakan dalam proses pencarian.

Pada dataset WisataFinder, kolom `search_text` telah disiapkan dengan menggabungkan:
- Nama destinasi wisata.
- Kategori wisata.
- Deskripsi destinasi.

Teks disimpan dalam bentuk huruf kecil agar pencarian tidak membedakan penggunaan huruf kapital.

### 4.2 TF-IDF

TF-IDF digunakan untuk mengubah teks menjadi representasi numerik berdasarkan bobot setiap kata.

Metode ini memperhitungkan frekuensi kata dalam suatu dokumen dan tingkat keunikan kata tersebut dalam keseluruhan dokumen.

Kata yang lebih informatif dapat memperoleh bobot yang lebih tinggi.

### 4.3 Cosine Similarity

Cosine Similarity digunakan untuk mengukur kemiripan antara vektor kata kunci pencarian pengguna dan vektor dokumen destinasi wisata.

Nilai kemiripan digunakan untuk mengurutkan hasil pencarian. Semakin tinggi nilai kemiripan, semakin dekat representasi teks destinasi dengan kata kunci pencarian.

### 4.4 Pengembangan Algoritma

Setelah metode dasar diterapkan, algoritma dikembangkan secara bertahap dengan menambahkan filter kategori, pengenalan kota, pengenalan provinsi, dan validasi lokasi.

Pengembangan tersebut menghasilkan beberapa versi algoritma, mulai dari V2 hingga V7.


In [8]:
# Memastikan kolom search_text tersedia

print("Kolom search_text tersedia:", "search_text" in df_nasional.columns)

print("\nContoh teks pencarian:")
display(
    df_nasional[
        ["Place_Name", "Category", "Description", "search_text"]
    ].head(5)
)

print(
    "\nJumlah search_text kosong:",
    df_nasional["search_text"].fillna("").eq("").sum()
)

Kolom search_text tersedia: True

Contoh teks pencarian:


,Place_Name,Category,Description,search_text
0,Air Terjun Aling-Aling,air terjun,Deskripsi tidak ditemukan,air terjun aling-aling air terjun deskripsi ti...
1,Air Terjun Bantimurung,air terjun,Air Terjun Bantimurung adalah salah satu air t...,air terjun bantimurung air terjun air terjun b...
2,Air Terjun Benang Kelambu,air terjun,Deskripsi tidak ditemukan,air terjun benang kelambu air terjun deskripsi...
3,Air Terjun Benang Stokel,air terjun,Deskripsi tidak ditemukan,air terjun benang stokel air terjun deskripsi ...
4,Air Terjun Bidadari,air terjun,Daftar ini tidak lengkap dan hanya dapat dijad...,air terjun bidadari air terjun daftar ini tida...



Jumlah search_text kosong: 0


In [9]:
# Membuat salinan khusus pengembangan algoritma

df_model = df_nasional.copy().reset_index(drop=True)

# Menginisialisasi TF-IDF

vectorizer = TfidfVectorizer()

# Mengubah teks destinasi menjadi matriks TF-IDF

matriks_tfidf = vectorizer.fit_transform(
    df_model["search_text"].fillna("").astype(str)
)

print("Jumlah destinasi:", len(df_model))
print("Ukuran matriks TF-IDF:", matriks_tfidf.shape)
print("Jumlah kosakata:", len(vectorizer.get_feature_names_out()))

Jumlah destinasi: 1025
Ukuran matriks TF-IDF: (1025, 3525)
Jumlah kosakata: 3525


In [10]:
# Kata kunci pencarian pengguna

query = "pantai di bali"

# Mengubah query menjadi vektor TF-IDF

query_vector = vectorizer.transform([query])

# Menghitung kemiripan query dengan seluruh destinasi

skor_similarity = cosine_similarity(
    query_vector,
    matriks_tfidf
).flatten()

# Mengambil 10 destinasi dengan skor tertinggi

indeks_teratas = np.argsort(skor_similarity)[::-1][:10]

hasil_pencarian_dasar = df_model.iloc[indeks_teratas][
    ["Place_Name", "Category", "City", "Province"]
].copy()

hasil_pencarian_dasar["Skor_Cosine_Similarity"] = (
    skor_similarity[indeks_teratas]
)

display(hasil_pencarian_dasar)

,Place_Name,Category,City,Province,Skor_Cosine_Similarity
440,Pantai Amed,pantai,Karangasem,Bali,0.599351
455,Pantai Candidasa,pantai,Karangasem,Bali,0.599351
242,Bali Zoo,kebun binatang,Gianyar,Bali,0.503165
443,Pantai Balian,pantai,Tabanan,Bali,0.486939
498,Pantai Nusa Dua,pantai,Badung,Bali,0.482892
241,Bali Bird Park,kebun binatang,Gianyar,Bali,0.481920
608,Taman Nusa,taman,Gianyar,Bali,0.447238
520,Pantai Seminyak,pantai,Badung,Bali,0.431646
463,Pantai Gerupuk,pantai,Mataram,Nusa Tenggara Barat,0.428082
452,Pantai Bontang,pantai,Bontang,Kalimantan Timur,0.411912



### 4.5 Pengujian TF-IDF dan Cosine Similarity

Pengujian awal dilakukan menggunakan kata kunci "pantai di bali".

Sistem mengubah kata kunci menjadi vektor TF-IDF, kemudian menghitung Cosine Similarity terhadap seluruh dokumen destinasi wisata.

Hasil pencarian diurutkan berdasarkan nilai kemiripan tertinggi.

Pada tahap ini, sistem masih menggunakan pencocokan kemiripan teks. Oleh karena itu, destinasi yang memiliki kemiripan kata tetapi tidak sesuai kategori atau lokasi masih berpotensi muncul.

Keterbatasan tersebut menjadi dasar pengembangan algoritma berikutnya melalui penambahan filter kategori dan lokasi.



### 4.6 Analisis Hasil Pencarian Dasar

Pengujian dilakukan menggunakan kata kunci "pantai di bali" dengan mengambil 10 destinasi yang memiliki nilai Cosine Similarity tertinggi.

Hasil pengujian menunjukkan bahwa sistem mampu menemukan beberapa destinasi pantai di Bali, seperti Pantai Amed, Pantai Candidasa, Pantai Balian, Pantai Nusa Dua, dan Pantai Seminyak.

Namun, ditemukan beberapa hasil yang kurang sesuai. Bali Zoo, Bali Bird Park, dan Taman Nusa muncul karena memiliki kemiripan teks dengan kata kunci pencarian, meskipun kategorinya bukan pantai. Selain itu, Pantai Gerupuk dan Pantai Bontang muncul meskipun lokasinya berada di luar Bali.

Dari 10 hasil teratas, terdapat 8 destinasi yang tercatat berada di Bali dan 7 destinasi dengan kategori pantai. Jika kategori dan lokasi harus sesuai secara bersamaan, hanya 5 hasil yang memenuhi kedua kriteria tersebut.

Temuan ini menunjukkan bahwa penggunaan TF-IDF dan Cosine Similarity saja belum cukup untuk memastikan kesesuaian kategori dan lokasi destinasi.

Oleh karena itu, diperlukan pengembangan algoritma dengan menambahkan mekanisme filter kategori, pengenalan lokasi, dan validasi lokasi agar hasil pencarian lebih sesuai dengan kebutuhan pengguna.


## 5. Pengembangan Algoritma Pencarian

Pengembangan algoritma WisataFinder dilakukan secara bertahap untuk meningkatkan relevansi hasil pencarian destinasi wisata.

Pengembangan dimulai dari metode TF-IDF dan Cosine Similarity, kemudian ditingkatkan dengan penambahan skor kategori, pencocokan lokasi, pengenalan kota dan provinsi secara otomatis, serta validasi lokasi yang tidak dikenal.

Versi algoritma yang dikembangkan meliputi:

1. V2: Penambahan skor kategori dan lokasi.
2. V3: Filter kategori dan lokasi.
3. V4: Pencocokan kategori berdasarkan nama destinasi.
4. V5: Pengenalan kota secara otomatis.
5. V6: Pengenalan provinsi secara otomatis.
6. V7: Validasi lokasi yang tidak dikenal atau belum tersedia.

In [12]:
# Kamus alias kategori wisata

alias_kategori = {
    "pantai": ["pantai", "laut", "bahari"],
    "gunung": ["gunung", "pendakian"],
    "wisata alam": ["alam", "hutan", "taman nasional"],
    "museum": ["museum", "sejarah"],
    "candi": ["candi", "pura", "situs sejarah"],
    "wisata kuliner": ["kuliner", "makanan", "tempat makan"],
    "wisata religi": ["religi", "ziarah", "tempat ibadah"],
    "kebun binatang": ["kebun binatang", "zoo"],
    "air terjun": ["air terjun", "curug"],
    "bukit": ["bukit", "perbukitan"],
    "wisata edukasi": ["edukasi", "pendidikan"],
    "mall": ["mall", "mal", "pusat perbelanjaan"],
    "budaya": ["budaya", "wisata budaya"]
}

# Kamus alias lokasi yang digunakan pada pengembangan awal

alias_lokasi = {
    "Daerah Istimewa Yogyakarta": ["jogja", "yogyakarta", "diy"],
    "DKI Jakarta": ["jakarta", "dki jakarta"],
    "Jawa Tengah": ["jawa tengah", "jateng"],
    "Jawa Barat": ["jawa barat", "jabar"],
    "Jawa Timur": ["jawa timur", "jatim"],
    "Nusa Tenggara Barat": ["ntb", "lombok"],
    "Bali": ["bali"],
    "Kota Semarang": ["semarang"]
}

# DataFrame khusus pengembangan algoritma

df_pengembangan = df_model.copy()

# Menggunakan matriks TF-IDF yang sudah dibuat sebelumnya

vectorizer_v2 = vectorizer
matriks_v2 = matriks_tfidf

print("Jumlah data:", len(df_pengembangan))
print("Jumlah alias kategori:", len(alias_kategori))
print("Jumlah alias lokasi:", len(alias_lokasi))

Jumlah data: 1025
Jumlah alias kategori: 13
Jumlah alias lokasi: 8


### 5.1 Algoritma V2 — Penambahan Skor Kategori dan Lokasi

Pada versi V2, sistem menggunakan TF-IDF dan Cosine Similarity untuk menghitung kemiripan teks.

Selanjutnya, sistem memberikan tambahan skor sebesar 0,30 untuk destinasi yang sesuai dengan kategori dan lokasi yang terdeteksi dalam kata kunci pencarian.

Tujuan pengembangan ini adalah meningkatkan posisi destinasi yang memiliki kesesuaian kategori dan lokasi dalam hasil pencarian.

In [14]:
def cari_wisata_v2(query, jumlah=10):
    query_bersih = query.lower().strip()

    query_vector = vectorizer_v2.transform([query_bersih])

    skor_teks = cosine_similarity(
        query_vector,
        matriks_v2
    ).flatten()

    skor_akhir = skor_teks.copy()

    kategori_cocok = [
        kategori
        for kategori, daftar_alias in alias_kategori.items()
        if any(
            re.search(
                r"(?<!\w)" + re.escape(alias) + r"(?!\w)",
                query_bersih
            )
            for alias in daftar_alias
        )
    ]

    lokasi_cocok = [
        (provinsi, alias)
        for provinsi, daftar_alias in alias_lokasi.items()
        for alias in daftar_alias
        if re.search(
            r"(?<!\w)" + re.escape(alias) + r"(?!\w)",
            query_bersih
        )
    ]

    if kategori_cocok:
        mask_kategori = (
            df_pengembangan["Category"]
            .fillna("")
            .str.lower()
            .isin(kategori_cocok)
        )

        skor_akhir[mask_kategori.to_numpy()] += 0.30

    if lokasi_cocok:
        mask_lokasi = df_pengembangan.apply(
            lambda row: any(
                str(row["Province"]).lower() == provinsi.lower()
                or alias in str(row["City"]).lower()
                for provinsi, alias in lokasi_cocok
            ),
            axis=1
        )

        skor_akhir[mask_lokasi.to_numpy()] += 0.30

    indeks = skor_akhir.argsort()[::-1][:jumlah]

    hasil = df_pengembangan.iloc[indeks][
        ["Place_Name", "Category", "City", "Province"]
    ].copy()

    hasil["Skor_TF_IDF"] = skor_teks[indeks]
    hasil["Skor_Akhir"] = skor_akhir[indeks]

    return hasil

In [15]:
uji_v2 = cari_wisata_v2("pantai di bali", 10)

display(uji_v2)

,Place_Name,Category,City,Province,Skor_TF_IDF,Skor_Akhir
440,Pantai Amed,pantai,Karangasem,Bali,0.599351,1.199351
455,Pantai Candidasa,pantai,Karangasem,Bali,0.599351,1.199351
443,Pantai Balian,pantai,Tabanan,Bali,0.486939,1.086939
498,Pantai Nusa Dua,pantai,Badung,Bali,0.482892,1.082892
520,Pantai Seminyak,pantai,Badung,Bali,0.431646,1.031646
469,Pantai Jimbaran,pantai,Badung,Bali,0.394676,0.994676
456,Pantai Canggu,pantai,Badung,Bali,0.394676,0.994676
461,Pantai Dreamland,pantai,Badung,Bali,0.394676,0.994676
441,Pantai Balangan,pantai,Badung,Bali,0.384800,0.984800
522,Pantai Seseh,pantai,Badung,Bali,0.384800,0.984800


### 5.2 Hasil Pengujian Algoritma V2

Pengujian V2 menggunakan kata kunci "pantai di bali". Algoritma menghitung kemiripan teks menggunakan TF-IDF dan Cosine Similarity, kemudian memberikan tambahan skor sebesar 0,30 untuk kesesuaian kategori dan lokasi yang terdeteksi.

Hasil pengujian menunjukkan bahwa 10 destinasi teratas berasal dari Bali dan berkategori pantai.

Penambahan skor kategori dan lokasi membantu meningkatkan posisi destinasi yang sesuai dengan kebutuhan pencarian.

Namun, mekanisme V2 masih menggunakan penambahan skor, bukan filter yang membatasi hasil secara ketat. Oleh karena itu, destinasi yang tidak sesuai masih berpotensi muncul pada query atau kondisi tertentu.

### 5.3 Algoritma V3 — Filter Kategori dan Lokasi

Pada versi V3, mekanisme pencarian dikembangkan dengan menerapkan filter kategori dan lokasi secara langsung.

Destinasi yang ditampilkan dipilih berdasarkan kesesuaian kategori dan lokasi yang terdeteksi dalam kata kunci pencarian.

Apabila pengguna memasukkan kategori dan lokasi sekaligus, sistem menggunakan kombinasi kedua filter tersebut.

Tujuan pengembangan ini adalah mengurangi hasil pencarian yang tidak sesuai dengan kategori maupun lokasi yang diminta pengguna.

In [16]:
def cari_wisata_v3(query, jumlah=10):
    query_bersih = query.lower().strip()

    query_vector = vectorizer_v2.transform([query_bersih])

    skor_teks = cosine_similarity(
        query_vector,
        matriks_v2
    ).flatten()

    # Mendeteksi kategori dari query
    kategori_cocok = [
        k for k, aliases in alias_kategori.items()
        if any(
            re.search(
                r"(?<!\w)" + re.escape(a) + r"(?!\w)",
                query_bersih
            )
            for a in aliases
        )
    ]

    # Kelompok kategori wisata budaya
    kelompok_budaya = [
        "candi",
        "museum",
        "monumen",
        "rumah adat",
        "wisata kerajaan"
    ]

    kategori_filter = []

    for k in kategori_cocok:
        if k == "budaya":
            kategori_filter.extend(kelompok_budaya)
        else:
            kategori_filter.append(k)

    # Mendeteksi lokasi dari alias
    lokasi_cocok = [
        (p, a)
        for p, aliases in alias_lokasi.items()
        for a in aliases
        if re.search(
            r"(?<!\w)" + re.escape(a) + r"(?!\w)",
            query_bersih
        )
    ]

    mask_kategori = (
        df_pengembangan["Category"]
        .fillna("")
        .str.lower()
        .isin(kategori_filter)
        if kategori_filter else None
    )

    mask_lokasi = (
        df_pengembangan.apply(
            lambda row: any(
                str(row["Province"]).lower() == p.lower()
                or a in str(row["City"]).lower()
                for p, a in lokasi_cocok
            ),
            axis=1
        )
        if lokasi_cocok else None
    )

    # Menggabungkan filter
    if mask_kategori is not None and mask_lokasi is not None:
        mask_hasil = mask_kategori & mask_lokasi
    elif mask_kategori is not None:
        mask_hasil = mask_kategori
    elif mask_lokasi is not None:
        mask_hasil = mask_lokasi
    else:
        mask_hasil = None

    # Perilaku fallback V3
    if mask_hasil is not None and mask_hasil.any():
        indeks_awal = mask_hasil.to_numpy().nonzero()[0]
    else:
        indeks_awal = np.arange(len(df_pengembangan))

    indeks = sorted(
        indeks_awal,
        key=lambda i: skor_teks[i],
        reverse=True
    )[:jumlah]

    hasil = df_pengembangan.iloc[indeks][
        ["Place_Name", "Category", "City", "Province"]
    ].copy()

    hasil["Skor_TF_IDF"] = skor_teks[indeks]

    return hasil

In [17]:
uji_v3 = cari_wisata_v3("pantai di bali", 10)

display(uji_v3)

,Place_Name,Category,City,Province,Skor_TF_IDF
440,Pantai Amed,pantai,Karangasem,Bali,0.599351
455,Pantai Candidasa,pantai,Karangasem,Bali,0.599351
443,Pantai Balian,pantai,Tabanan,Bali,0.486939
498,Pantai Nusa Dua,pantai,Badung,Bali,0.482892
520,Pantai Seminyak,pantai,Badung,Bali,0.431646
456,Pantai Canggu,pantai,Badung,Bali,0.394676
461,Pantai Dreamland,pantai,Badung,Bali,0.394676
469,Pantai Jimbaran,pantai,Badung,Bali,0.394676
441,Pantai Balangan,pantai,Badung,Bali,0.384800
522,Pantai Seseh,pantai,Badung,Bali,0.384800


In [18]:
uji_v3_jakarta = cari_wisata_v3(
    "kebun binatang di jakarta", 10
)

display(uji_v3_jakarta)

,Place_Name,Category,City,Province,Skor_TF_IDF
244,Kebun Binatang Cikembulan Garut,kebun binatang,Neglasari,Jawa Barat,0.667667
247,Kebun Binatang Kasang Kulim Pekanbaru,kebun binatang,Pekanbaru,Riau,0.610568
250,Kebun Binatang Surabaya,kebun binatang,Surabaya,Jawa Timur,0.602726
249,Kebun Binatang Medan Zoo,kebun binatang,Deli Serdang,Sumatera Utara,0.550311
826,Kebun Binatang Ragunan,wisata edukasi,Daerah Khusus ibukota Jakarta,DKI Jakarta,0.526092
254,Taman Margasatwa dan Budaya Kinantan,kebun binatang,Bukittinggi,Sumatera Barat,0.516490
256,Taman Safari Indonesia Bogor,kebun binatang,Kab Bogor,Jawa Barat,0.508042
251,Medan Zoo,kebun binatang,Deli Serdang,Sumatera Utara,0.499199
243,Kebun Binatang Bandung,kebun binatang,Kota Bandung,Jawa Barat,0.478812
246,Kebun Binatang Kaltim Samarinda,kebun binatang,Samarinda,Kalimantan Timur,0.402240



### 5.4 Hasil Pengujian Algoritma V3

Pengujian V3 dilakukan menggunakan dua kata kunci, yaitu "pantai di bali" dan "kebun binatang di jakarta".

Pada pencarian "pantai di bali", sistem menghasilkan 10 destinasi yang seluruhnya berkategori pantai dan berlokasi di Bali. Hal ini menunjukkan bahwa filter kategori dan lokasi dapat bekerja ketika kedua kriteria memiliki data yang cocok.

Namun, pada pencarian "kebun binatang di jakarta", sistem masih menampilkan destinasi dari berbagai provinsi. Salah satu penyebabnya adalah Kebun Binatang Ragunan tercatat dalam dataset dengan kategori wisata edukasi, bukan kebun binatang.

Ketika kombinasi filter tidak menemukan hasil, mekanisme fallback V3 mengambil kembali data dari seluruh dataset.

Temuan tersebut menunjukkan bahwa sistem perlu mempertimbangkan nama destinasi sebagai informasi tambahan dalam mengenali kategori wisata.



### 5.5 Algoritma V4 — Pencocokan Kategori Berdasarkan Nama Destinasi

Pada versi V4, sistem dikembangkan dengan menambahkan pencocokan kata kunci kategori terhadap nama destinasi.

Pengembangan ini bertujuan menangani perbedaan antara kategori yang tercatat dalam dataset dan jenis destinasi yang disebutkan pengguna.

Sebagai contoh, Kebun Binatang Ragunan tercatat dengan kategori wisata edukasi. Namun, nama destinasi tersebut mengandung frasa "Kebun Binatang".

Dengan demikian, sistem dapat mempertimbangkan nama destinasi sebagai informasi tambahan dalam proses pencarian.


In [19]:
def cari_wisata_v4(query, jumlah=10):
    query_bersih = query.lower().strip()

    query_vector = vectorizer_v2.transform([query_bersih])

    skor_teks = cosine_similarity(
        query_vector,
        matriks_v2
    ).flatten()

    # Mengenali kategori
    kategori_cocok = [
        k for k, aliases in alias_kategori.items()
        if any(
            re.search(
                r"(?<!\w)" + re.escape(a) + r"(?!\w)",
                query_bersih
            )
            for a in aliases
        )
    ]

    kelompok_budaya = [
        "candi", "museum", "monumen",
        "rumah adat", "wisata kerajaan"
    ]

    kategori_filter = []

    for k in kategori_cocok:
        if k == "budaya":
            kategori_filter.extend(kelompok_budaya)
        else:
            kategori_filter.append(k)

    # Mengenali alias kategori dalam query
    alias_terdeteksi = [
        a for aliases in alias_kategori.values()
        for a in aliases
        if re.search(
            r"(?<!\w)" + re.escape(a) + r"(?!\w)",
            query_bersih
        )
    ]

    # Filter berdasarkan kategori ATAU nama destinasi
    mask_kategori = None

    if kategori_filter or alias_terdeteksi:
        kategori_data = (
            df_pengembangan["Category"]
            .fillna("")
            .astype(str)
            .str.lower()
        )

        nama_data = (
            df_pengembangan["Place_Name"]
            .fillna("")
            .astype(str)
            .str.lower()
        )

        mask_nama = nama_data.apply(
            lambda nama: any(
                re.search(
                    r"(?<!\w)" + re.escape(a) + r"(?!\w)",
                    nama
                )
                for a in alias_terdeteksi
            )
        )

        mask_kategori = (
            kategori_data.isin(kategori_filter) | mask_nama
        )

    # Filter lokasi berdasarkan alias yang tersedia
    lokasi_cocok = [
        (p, a)
        for p, aliases in alias_lokasi.items()
        for a in aliases
        if re.search(
            r"(?<!\w)" + re.escape(a) + r"(?!\w)",
            query_bersih
        )
    ]

    mask_lokasi = None

    if lokasi_cocok:
        mask_lokasi = df_pengembangan.apply(
            lambda row: any(
                str(row["Province"]).lower() == p.lower()
                or a in str(row["City"]).lower()
                for p, a in lokasi_cocok
            ),
            axis=1
        )

    # Menggabungkan filter
    if mask_kategori is not None and mask_lokasi is not None:
        mask_hasil = mask_kategori & mask_lokasi
    elif mask_kategori is not None:
        mask_hasil = mask_kategori
    elif mask_lokasi is not None:
        mask_hasil = mask_lokasi
    else:
        mask_hasil = None

    # Mekanisme fallback V4
    if mask_hasil is not None and mask_hasil.any():
        indeks_awal = mask_hasil.to_numpy().nonzero()[0]
    else:
        indeks_awal = np.arange(len(df_pengembangan))

    indeks = sorted(
        indeks_awal,
        key=lambda i: skor_teks[i],
        reverse=True
    )[:jumlah]

    hasil = df_pengembangan.iloc[indeks][
        ["Place_Name", "Category", "City", "Province"]
    ].copy()

    hasil["Skor_TF_IDF"] = skor_teks[indeks]

    return hasil

In [20]:
uji_v4_zoo = cari_wisata_v4(
    "kebun binatang di jakarta", 10
)

display(uji_v4_zoo)

,Place_Name,Category,City,Province,Skor_TF_IDF
826,Kebun Binatang Ragunan,wisata edukasi,Daerah Khusus ibukota Jakarta,DKI Jakarta,0.526092


In [21]:
uji_v4_bali = cari_wisata_v4(
    "pantai di bali", 10
)

display(uji_v4_bali)

,Place_Name,Category,City,Province,Skor_TF_IDF
440,Pantai Amed,pantai,Karangasem,Bali,0.599351
455,Pantai Candidasa,pantai,Karangasem,Bali,0.599351
443,Pantai Balian,pantai,Tabanan,Bali,0.486939
498,Pantai Nusa Dua,pantai,Badung,Bali,0.482892
520,Pantai Seminyak,pantai,Badung,Bali,0.431646
456,Pantai Canggu,pantai,Badung,Bali,0.394676
461,Pantai Dreamland,pantai,Badung,Bali,0.394676
469,Pantai Jimbaran,pantai,Badung,Bali,0.394676
441,Pantai Balangan,pantai,Badung,Bali,0.384800
522,Pantai Seseh,pantai,Badung,Bali,0.384800


In [22]:
uji_v4_air_terjun = cari_wisata_v4(
    "air terjun di jakarta", 10
)

display(uji_v4_air_terjun)

,Place_Name,Category,City,Province,Skor_TF_IDF
9,Air Terjun Jumog,air terjun,Karanganyar,Jawa Tengah,0.683643
18,Air Terjun Tegenungan,air terjun,Gianyar,Bali,0.668580
7,Air Terjun Curug Dengdeng,air terjun,Tasikmalaya,Jawa Barat,0.645601
5,Air Terjun Coban Rondo,air terjun,Kabupaten Malang,Jawa Timur,0.639113
6,Air Terjun Coban Talun,air terjun,Kota Batu,Jawa Timur,0.627874
12,Air Terjun Mangku Sakti,air terjun,Lombok Utara,Nusa Tenggara Barat,0.627874
2,Air Terjun Benang Kelambu,air terjun,Lombok Tengah,Nusa Tenggara Barat,0.623508
3,Air Terjun Benang Stokel,air terjun,Lombok Tengah,Nusa Tenggara Barat,0.623508
1,Air Terjun Bantimurung,air terjun,Sulawesi,Sulawesi Selatan,0.617590
10,Air Terjun Kanto Lampo,air terjun,Gianyar,Bali,0.617200



### 5.6 Hasil Pengujian Algoritma V4

Pengujian V4 dilakukan menggunakan tiga kata kunci, yaitu "kebun binatang di jakarta", "pantai di bali", dan "air terjun di jakarta".

Pada pencarian "kebun binatang di jakarta", sistem berhasil menemukan Kebun Binatang Ragunan. Hal ini menunjukkan bahwa pencocokan kategori berdasarkan nama destinasi dapat membantu mengatasi perbedaan label kategori pada dataset.

Pada pencarian "pantai di bali", sistem menghasilkan 10 destinasi yang seluruhnya sesuai dengan kategori pantai dan lokasi Bali.

Namun, pada pencarian "air terjun di jakarta", sistem masih menampilkan destinasi dari berbagai provinsi. Hal ini terjadi karena mekanisme fallback masih mengambil seluruh dataset ketika kombinasi filter kategori dan lokasi tidak menemukan hasil.

Dengan demikian, V4 berhasil meningkatkan pencocokan kategori, tetapi masih membutuhkan pengembangan pengenalan kota secara otomatis dan perbaikan penanganan lokasi.



### 5.7 Algoritma V5 — Pengenalan Kota Otomatis

Pada versi V5, algoritma dikembangkan dengan menambahkan mekanisme pengenalan kota secara otomatis berdasarkan kolom City pada dataset.

Sistem mengidentifikasi nama kota yang terdapat dalam kata kunci pencarian, kemudian menggunakan informasi tersebut untuk membatasi hasil pencarian.

Pengembangan ini bertujuan meningkatkan ketepatan hasil berdasarkan lokasi kota tanpa harus menambahkan seluruh nama kota secara manual ke dalam kamus alias.

Namun, V5 masih memiliki keterbatasan dalam mengenali seluruh provinsi dan menangani lokasi yang belum dikenal.


In [23]:
def cari_wisata_v5(query, jumlah=10):
    query_bersih = query.lower().strip()

    query_vector = vectorizer_v2.transform([query_bersih])
    skor_teks = cosine_similarity(
        query_vector, matriks_v2
    ).flatten()

    # 1. Mengenali kategori
    kategori_cocok = [
        k for k, aliases in alias_kategori.items()
        if any(
            re.search(
                r"(?<!\w)" + re.escape(a) + r"(?!\w)",
                query_bersih
            )
            for a in aliases
        )
    ]

    kelompok_budaya = [
        "candi", "museum", "monumen",
        "rumah adat", "wisata kerajaan"
    ]

    kategori_filter = []
    for k in kategori_cocok:
        if k == "budaya":
            kategori_filter.extend(kelompok_budaya)
        else:
            kategori_filter.append(k)

    alias_terdeteksi = [
        a for aliases in alias_kategori.values()
        for a in aliases
        if re.search(
            r"(?<!\w)" + re.escape(a) + r"(?!\w)",
            query_bersih
        )
    ]

    mask_kategori = None

    if kategori_filter or alias_terdeteksi:
        kategori_data = (
            df_pengembangan["Category"]
            .fillna("")
            .astype(str)
            .str.lower()
        )

        nama_data = (
            df_pengembangan["Place_Name"]
            .fillna("")
            .astype(str)
            .str.lower()
        )

        mask_kategori = (
            kategori_data.isin(kategori_filter)
            |
            nama_data.apply(
                lambda nama: any(
                    a in nama for a in alias_terdeteksi
                )
            )
        )

    # 2. Mengenali lokasi berdasarkan alias
    lokasi_cocok = [
        (p, a)
        for p, aliases in alias_lokasi.items()
        for a in aliases
        if re.search(
            r"(?<!\w)" + re.escape(a) + r"(?!\w)",
            query_bersih
        )
    ]

    mask_lokasi_alias = None

    if lokasi_cocok:
        mask_lokasi_alias = df_pengembangan.apply(
            lambda row: any(
                str(row["Province"]).lower() == p.lower()
                or a in str(row["City"]).lower()
                for p, a in lokasi_cocok
            ),
            axis=1
        )

    # 3. Menyiapkan nama kota otomatis dari dataset
    kota_data = (
        df_pengembangan["City"]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.strip()
        .str.replace(
            r"^(?:kota administrasi|kabupaten|kab\.?|kota)\s+",
            "",
            regex=True
        )
    )

    wilayah_umum = [
        "jawa", "sumatera", "sulawesi",
        "kalimantan", "papua", "maluku",
        "western new guinea"
    ]

    kota_unik = sorted(
        {
            kota for kota in kota_data.unique()
            if kota and kota not in wilayah_umum
        },
        key=len,
        reverse=True
    )

    # 4. Mendeteksi kota dalam query
    mask_kota = kota_data.apply(
        lambda kota: (
            bool(kota)
            and kota not in wilayah_umum
            and re.search(
                r"(?<!\w)" + re.escape(kota) + r"(?!\w)",
                query_bersih
            ) is not None
        )
    )

    # 5. Menggabungkan pengenalan lokasi
    if mask_lokasi_alias is not None and mask_kota.any():
        mask_lokasi = mask_lokasi_alias | mask_kota
    elif mask_lokasi_alias is not None:
        mask_lokasi = mask_lokasi_alias
    elif mask_kota.any():
        mask_lokasi = mask_kota
    else:
        mask_lokasi = None

    # 6. Menggabungkan filter kategori dan lokasi
    if mask_kategori is not None and mask_lokasi is not None:
        mask_hasil = mask_kategori & mask_lokasi
    elif mask_kategori is not None:
        mask_hasil = mask_kategori
    elif mask_lokasi is not None:
        mask_hasil = mask_lokasi
    else:
        mask_hasil = None

    # 7. Fallback V5
    if mask_hasil is not None and mask_hasil.any():
        indeks_awal = mask_hasil.to_numpy().nonzero()[0]
    else:
        indeks_awal = np.arange(len(df_pengembangan))

    indeks = sorted(
        indeks_awal,
        key=lambda i: skor_teks[i],
        reverse=True
    )[:jumlah]

    hasil = df_pengembangan.iloc[indeks][
        ["Place_Name", "Category", "City", "Province"]
    ].copy()

    hasil["Skor_TF_IDF"] = skor_teks[indeks]

    return hasil

In [24]:
uji_v5_bandung = cari_wisata_v5(
    "museum di bandung", 10
)

display(uji_v5_bandung)

,Place_Name,Category,City,Province,Skor_TF_IDF
368,Museum Geologi Bandung,museum,Kota Bandung,Jawa Barat,0.591834
406,Museum Pendidikan Nasional,museum,Kota Bandung,Jawa Barat,0.503622
379,Museum Konferensi Asia Afrika,museum,Kota Bandung,Jawa Barat,0.458231
387,Museum Mandala Wangsit Siliwangi,museum,Kota Bandung,Jawa Barat,0.251646


In [25]:
uji_v5_semarang = cari_wisata_v5(
    "museum di semarang", 10
)

display(uji_v5_semarang)

,Place_Name,Category,City,Province,Skor_TF_IDF
413,Museum Ranggawarsita,museum,Kota Semarang,Jawa Tengah,0.464744


In [26]:
uji_v5_air_terjun = cari_wisata_v5(
    "air terjun di jakarta", 10
)

display(uji_v5_air_terjun)

,Place_Name,Category,City,Province,Skor_TF_IDF
9,Air Terjun Jumog,air terjun,Karanganyar,Jawa Tengah,0.683643
18,Air Terjun Tegenungan,air terjun,Gianyar,Bali,0.668580
7,Air Terjun Curug Dengdeng,air terjun,Tasikmalaya,Jawa Barat,0.645601
5,Air Terjun Coban Rondo,air terjun,Kabupaten Malang,Jawa Timur,0.639113
6,Air Terjun Coban Talun,air terjun,Kota Batu,Jawa Timur,0.627874
12,Air Terjun Mangku Sakti,air terjun,Lombok Utara,Nusa Tenggara Barat,0.627874
2,Air Terjun Benang Kelambu,air terjun,Lombok Tengah,Nusa Tenggara Barat,0.623508
3,Air Terjun Benang Stokel,air terjun,Lombok Tengah,Nusa Tenggara Barat,0.623508
1,Air Terjun Bantimurung,air terjun,Sulawesi,Sulawesi Selatan,0.617590
10,Air Terjun Kanto Lampo,air terjun,Gianyar,Bali,0.617200


In [27]:
uji_v5_bali = cari_wisata_v5(
    "pantai di bali", 10
)

display(uji_v5_bali)

,Place_Name,Category,City,Province,Skor_TF_IDF
440,Pantai Amed,pantai,Karangasem,Bali,0.599351
455,Pantai Candidasa,pantai,Karangasem,Bali,0.599351
443,Pantai Balian,pantai,Tabanan,Bali,0.486939
498,Pantai Nusa Dua,pantai,Badung,Bali,0.482892
520,Pantai Seminyak,pantai,Badung,Bali,0.431646
456,Pantai Canggu,pantai,Badung,Bali,0.394676
461,Pantai Dreamland,pantai,Badung,Bali,0.394676
469,Pantai Jimbaran,pantai,Badung,Bali,0.394676
441,Pantai Balangan,pantai,Badung,Bali,0.384800
522,Pantai Seseh,pantai,Badung,Bali,0.384800



### 5.8 Hasil Pengujian Algoritma V5

Pengujian V5 dilakukan menggunakan empat kata kunci, yaitu "museum di Bandung", "museum di Semarang", "air terjun di Jakarta", dan "pantai di Bali".

Hasil pengujian menunjukkan bahwa pengenalan kota otomatis berhasil dilakukan. Pencarian museum di Bandung menghasilkan empat destinasi yang tercatat di Kota Bandung, sedangkan pencarian museum di Semarang menghasilkan Museum Ranggawarsita.

Namun, pencarian air terjun di Jakarta masih menghasilkan destinasi dari berbagai provinsi karena mekanisme fallback belum diperbaiki.

Pencarian pantai di Bali menghasilkan 10 destinasi yang seluruhnya sesuai dengan kategori dan lokasi.

Dengan demikian, V5 meningkatkan ketepatan pencarian berdasarkan kota, tetapi masih memerlukan pengembangan pengenalan provinsi dan validasi lokasi.



### 5.9 Algoritma V6 — Pengenalan Provinsi Otomatis

Pada versi V6, algoritma dikembangkan untuk mengenali nama provinsi secara otomatis berdasarkan kolom Province pada dataset.

Sistem menggunakan daftar nama provinsi yang tersedia dalam dataset serta alias provinsi yang telah ditentukan.

Pengenalan lokasi dilakukan dengan mempertimbangkan nama provinsi dan nama kota. Apabila pengguna memasukkan nama provinsi yang memiliki kemiripan dengan nama provinsi lain, sistem menggunakan pencocokan frasa yang lebih spesifik untuk menghindari kesalahan pengenalan.

Pengembangan ini bertujuan memperluas kemampuan pencarian nasional tanpa harus memasukkan seluruh provinsi secara manual ke dalam kamus alias.


In [28]:
# Mengambil nama provinsi dari dataset
provinsi_dataset_v6 = sorted(
    df_pengembangan["Province"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique(),
    key=len,
    reverse=True
)

# Menambahkan Kepulauan Riau sebagai lokasi yang dikenal,
# meskipun belum tersedia dalam dataset
provinsi_dikenali_v6 = provinsi_dataset_v6 + [
    "Kepulauan Riau"
]

print("Provinsi dalam dataset:", len(provinsi_dataset_v6))
print("Provinsi yang dikenali:", len(provinsi_dikenali_v6))

Provinsi dalam dataset: 37
Provinsi yang dikenali: 38


In [29]:
def cari_wisata_v6(query, jumlah=10):
    query_bersih = str(query).lower().strip()

    query_vector = vectorizer_v2.transform([query_bersih])
    skor_teks = cosine_similarity(
        query_vector, matriks_v2
    ).flatten()

    # 1. Mendeteksi kategori
    kategori_cocok = [
        k for k, aliases in alias_kategori.items()
        if any(
            re.search(
                r"(?<!\w)" + re.escape(a) + r"(?!\w)",
                query_bersih
            )
            for a in aliases
        )
    ]

    kelompok_budaya = [
        "candi", "museum", "monumen",
        "rumah adat", "wisata kerajaan"
    ]

    kategori_filter = []
    for k in kategori_cocok:
        if k == "budaya":
            kategori_filter.extend(kelompok_budaya)
        else:
            kategori_filter.append(k)

    alias_terdeteksi = [
        a for aliases in alias_kategori.values()
        for a in aliases
        if re.search(
            r"(?<!\w)" + re.escape(a) + r"(?!\w)",
            query_bersih
        )
    ]

    # Filter kategori dan nama destinasi
    mask_kategori = None

    if kategori_filter or alias_terdeteksi:
        kategori_data = (
            df_pengembangan["Category"]
            .fillna("")
            .astype(str)
            .str.lower()
        )

        nama_data = (
            df_pengembangan["Place_Name"]
            .fillna("")
            .astype(str)
            .str.lower()
        )

        mask_nama = nama_data.apply(
            lambda nama: any(
                re.search(
                    r"(?<!\w)" + re.escape(a) + r"(?!\w)",
                    nama
                )
                for a in alias_terdeteksi
            )
        )

        mask_kategori = (
            kategori_data.isin(kategori_filter) | mask_nama
        )

    # 2. Menyiapkan data kota
    kota_data = (
        df_pengembangan["City"]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.strip()
        .str.replace(
            r"^(?:kota administrasi|kabupaten|kab\.?|kota)\s+",
            "",
            regex=True
        )
    )

    wilayah_umum = {
        "jawa", "sumatera", "sulawesi",
        "kalimantan", "papua", "maluku",
        "western new guinea"
    }

    kota_unik = sorted(
        {
            kota for kota in kota_data.unique()
            if kota and kota not in wilayah_umum
        },
        key=len,
        reverse=True
    )

    # 3. Menyiapkan daftar nama provinsi dan aliasnya
    kandidat_lokasi = [
        (p.lower(), "provinsi", p)
        for p in provinsi_dikenali_v6
    ]

    # Hanya menggunakan alias untuk nama provinsi
    # yang benar-benar tercatat dalam dataset
    for provinsi in provinsi_dataset_v6:
        for alias in alias_lokasi.get(provinsi, []):
            kandidat_lokasi.append(
                (str(alias).lower(), "provinsi", provinsi)
            )

    # Menambahkan nama kota otomatis
    for kota in kota_unik:
        kandidat_lokasi.append(
            (kota, "kota", kota)
        )

    # Frasa lebih panjang diperiksa terlebih dahulu
    kandidat_lokasi.sort(
        key=lambda item: len(item[0]),
        reverse=True
    )

    # 4. Mendeteksi lokasi dalam query
    provinsi_terdeteksi = set()
    kota_terdeteksi = set()
    rentang_terpakai = []

    for istilah, jenis, nilai in kandidat_lokasi:
        pola = (
            r"(?<!\w)" + re.escape(istilah) + r"(?!\w)"
        )

        for kecocokan in re.finditer(pola, query_bersih):
            rentang = (
                kecocokan.start(),
                kecocokan.end()
            )

            tumpang_tindih = any(
                rentang[0] < akhir and rentang[1] > awal
                for awal, akhir in rentang_terpakai
            )

            if tumpang_tindih:
                continue

            rentang_terpakai.append(rentang)

            if jenis == "provinsi":
                provinsi_terdeteksi.add(nilai)
            else:
                kota_terdeteksi.add(nilai)

            break

    # 5. Membuat filter lokasi
    bagian_lokasi = []

    if provinsi_terdeteksi:
        provinsi_data = (
            df_pengembangan["Province"]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        bagian_lokasi.append(
            provinsi_data.isin(provinsi_terdeteksi)
        )

    if kota_terdeteksi:
        bagian_lokasi.append(
            kota_data.isin(kota_terdeteksi)
        )

    mask_lokasi = None

    if bagian_lokasi:
        mask_lokasi = bagian_lokasi[0]

        for bagian in bagian_lokasi[1:]:
            mask_lokasi = mask_lokasi & bagian

    # 6. Menggabungkan filter kategori dan lokasi
    if mask_kategori is not None and mask_lokasi is not None:
        mask_hasil = mask_kategori & mask_lokasi
    elif mask_kategori is not None:
        mask_hasil = mask_kategori
    elif mask_lokasi is not None:
        mask_hasil = mask_lokasi
    else:
        mask_hasil = pd.Series(
            True,
            index=df_pengembangan.index
        )

    # 7. Mengambil hasil tanpa fallback ke seluruh dataset
    indeks_awal = mask_hasil.to_numpy().nonzero()[0]

    indeks = sorted(
        indeks_awal,
        key=lambda i: skor_teks[i],
        reverse=True
    )[:jumlah]

    hasil = df_pengembangan.iloc[indeks][
        ["Place_Name", "Category", "City", "Province"]
    ].copy()

    hasil["Skor_TF_IDF"] = skor_teks[indeks]

    return hasil

In [30]:
uji_v6_sultra = cari_wisata_v6(
    "museum di sulawesi tenggara", 10
)

display(uji_v6_sultra)

,Place_Name,Category,City,Province,Skor_TF_IDF
424,Museum Sulawesi Tenggara,museum,Kendari,Sulawesi Tenggara,0.734619


In [31]:
uji_v6_papua = cari_wisata_v6(
    "tempat wisata di papua barat", 10
)

display(uji_v6_papua)

,Place_Name,Category,City,Province,Skor_TF_IDF
168,Gunung Arfak,gunung,Manokwari,Papua Barat,0.130588
531,Pantai Tanjung,pantai,Fakfak,Papua Barat,0.122447
754,Pulau Mioswaar,wisata alam,Western New Guinea,Papua Barat,0.051778


In [32]:
uji_v6_semarang = cari_wisata_v6(
    "museum di semarang", 10
)

display(uji_v6_semarang)

,Place_Name,Category,City,Province,Skor_TF_IDF
413,Museum Ranggawarsita,museum,Kota Semarang,Jawa Tengah,0.464744


In [33]:
uji_v6_air_terjun = cari_wisata_v6(
    "air terjun di jakarta", 10
)

display(uji_v6_air_terjun)

,Place_Name,Category,City,Province,Skor_TF_IDF


### 5.10 Hasil Pengujian Algoritma V6

Pengujian V6 dilakukan menggunakan empat kata kunci, yaitu "museum di Sulawesi Tenggara", "tempat wisata di Papua Barat", "museum di Semarang", dan "air terjun di Jakarta".

Hasil pengujian menunjukkan bahwa sistem berhasil mengenali nama provinsi secara otomatis berdasarkan data pada dataset.

Pada pencarian "tempat wisata di Papua Barat", sistem menghasilkan tiga destinasi yang seluruhnya tercatat di Provinsi Papua Barat. Sistem juga berhasil membedakan Papua Barat dari provinsi lain yang memiliki nama serupa.

Pada pencarian "air terjun di Jakarta", sistem menghasilkan data kosong karena tidak ditemukan destinasi yang memenuhi kombinasi kategori dan lokasi.

Dengan demikian, V6 berhasil meningkatkan kemampuan pengenalan provinsi dan memperketat filter pencarian lokasi.

### 5.11 Algoritma V7 — Validasi Lokasi

Pada versi V7, algoritma dikembangkan dengan menambahkan mekanisme validasi lokasi sebelum proses pencarian dilakukan.

Sistem memeriksa apakah lokasi yang disebutkan pengguna termasuk dalam daftar provinsi yang dikenali atau nama kota yang tersedia dalam dataset.

Apabila pengguna memasukkan lokasi yang tidak dikenal, sistem mengembalikan hasil kosong.

Apabila lokasi dikenal tetapi belum memiliki destinasi dalam dataset, sistem juga mengembalikan hasil kosong.

Apabila lokasi dikenal dan memiliki data, sistem melanjutkan proses pencarian menggunakan algoritma V6.

Pengembangan ini bertujuan mencegah sistem menampilkan destinasi dari wilayah yang tidak sesuai dengan permintaan pengguna.

In [35]:
def cari_wisata_v7(query, jumlah=10):
    query_bersih = str(query).lower().strip()

    # Memeriksa apakah pengguna menyebutkan lokasi dengan kata "di"
    pola_di = r"(?<!\w)di(?!\w)"

    if re.search(pola_di, query_bersih):

        bagian_lokasi = re.split(
            pola_di, query_bersih
        )[-1].strip()

        # Daftar provinsi dan kota yang dikenali
        lokasi_dikenal = [
            p.lower() for p in provinsi_dikenali_v6
        ]

        # Menambahkan alias provinsi
        for provinsi, aliases in alias_lokasi.items():
            lokasi_dikenal.append(provinsi.lower())
            lokasi_dikenal.extend([
                str(a).lower() for a in aliases
            ])

        # Menyiapkan nama kota dari dataset
        kota_data = (
            df_pengembangan["City"]
            .fillna("")
            .astype(str)
            .str.lower()
            .str.strip()
            .str.replace(
                r"^(?:kota administrasi|kabupaten|kab\.?|kota)\s+",
                "",
                regex=True
            )
        )

        wilayah_umum = {
            "jawa", "sumatera", "sulawesi",
            "kalimantan", "papua", "maluku",
            "western new guinea"
        }

        lokasi_dikenal.extend([
            kota for kota in kota_data.unique()
            if kota and kota not in wilayah_umum
        ])

        # Memeriksa kecocokan lokasi
        lokasi_valid = any(
            re.search(
                r"(?<!\w)" + re.escape(lokasi) + r"(?!\w)",
                bagian_lokasi
            )
            for lokasi in lokasi_dikenal
            if lokasi
        )

        # Jika lokasi tidak dikenali, kembalikan hasil kosong
        if not lokasi_valid:
            return pd.DataFrame(columns=[
                "Place_Name",
                "Category",
                "City",
                "Province",
                "Skor_TF_IDF"
            ])

    # Jika lokasi valid atau query tidak menyebut lokasi,
    # lanjutkan menggunakan algoritma V6
    return cari_wisata_v6(query, jumlah)

In [36]:
uji_v7_atlantis = cari_wisata_v7(
    "tempat wisata di atlantis", 10
)

print("Jumlah hasil:", len(uji_v7_atlantis))
display(uji_v7_atlantis)

Jumlah hasil: 0


,Place_Name,Category,City,Province,Skor_TF_IDF


In [37]:
uji_v7_kepri = cari_wisata_v7(
    "tempat wisata di kepulauan riau", 10
)

print("Jumlah hasil:", len(uji_v7_kepri))
display(uji_v7_kepri)

Jumlah hasil: 0


,Place_Name,Category,City,Province,Skor_TF_IDF


In [38]:
uji_v7_semarang = cari_wisata_v7(
    "museum di semarang", 10
)

display(uji_v7_semarang)

,Place_Name,Category,City,Province,Skor_TF_IDF
413,Museum Ranggawarsita,museum,Kota Semarang,Jawa Tengah,0.464744


In [39]:
uji_v7_museum = cari_wisata_v7("museum", 10)

display(uji_v7_museum)

,Place_Name,Category,City,Province,Skor_TF_IDF
399,Museum NTB,museum,Mataram,Nusa Tenggara Barat,0.708428
430,Museum Tekstil,museum,Daerah Khusus ibukota Jakarta,DKI Jakarta,0.700956
367,Museum Fatahillah,museum,Daerah Khusus ibukota Jakarta,DKI Jakarta,0.680142
392,Museum Nasional Indonesia,museum,Bogor,Jawa Barat,0.677783
391,Museum Musik Indonesia Malang,museum,Kota Malang,Jawa Timur,0.671011
428,Museum Tani Jawa Indonesia,museum,Kebonagung,Daerah Istimewa Yogyakarta,0.656354
395,Museum Negeri Nusa Tenggara Barat,museum,Mataram,Nusa Tenggara Barat,0.639080
424,Museum Sulawesi Tenggara,museum,Kendari,Sulawesi Tenggara,0.637136
413,Museum Ranggawarsita,museum,Kota Semarang,Jawa Tengah,0.629049
397,Museum Negeri Sumatera Utara,museum,Kota Medan,Sumatera Utara,0.618959



### 5.12 Hasil Pengujian Algoritma V7

Pengujian V7 dilakukan menggunakan empat skenario, yaitu lokasi tidak dikenal, lokasi dikenal tetapi belum memiliki data, lokasi yang memiliki data, dan pencarian kategori tanpa lokasi.

Pada pencarian "tempat wisata di Atlantis", sistem menghasilkan data kosong karena lokasi tersebut tidak ditemukan dalam daftar lokasi yang dikenali.

Pada pencarian "tempat wisata di Kepulauan Riau", sistem juga menghasilkan data kosong karena provinsi tersebut telah dikenali, tetapi belum memiliki destinasi dalam dataset.

Pada pencarian "museum di Semarang", sistem berhasil menemukan Museum Ranggawarsita yang tercatat di Kota Semarang.

Sementara itu, pencarian "museum" tanpa lokasi menghasilkan 10 museum dari berbagai provinsi.

Hasil pengujian menunjukkan bahwa V7 mampu membedakan lokasi yang tidak dikenal, lokasi yang belum memiliki data, dan lokasi yang tersedia dalam dataset.


In [40]:
# Perbandingan berdasarkan empat skenario pengujian

hasil_bandung_per_versi = {
    "V2": cari_wisata_v2("museum di bandung", 10),
    "V3": cari_wisata_v3("museum di bandung", 10),
    "V4": cari_wisata_v4("museum di bandung", 10),
    "V5": cari_wisata_v5("museum di bandung", 10),
    "V6": cari_wisata_v6("museum di bandung", 10),
    "V7": cari_wisata_v7("museum di bandung", 10)
}

hasil_papua_per_versi = {
    "V2": cari_wisata_v2("tempat wisata di papua barat", 10),
    "V3": cari_wisata_v3("tempat wisata di papua barat", 10),
    "V4": cari_wisata_v4("tempat wisata di papua barat", 10),
    "V5": cari_wisata_v5("tempat wisata di papua barat", 10),
    "V6": cari_wisata_v6("tempat wisata di papua barat", 10),
    "V7": cari_wisata_v7("tempat wisata di papua barat", 10)
}

hasil_atlantis_per_versi = {
    "V2": cari_wisata_v2("tempat wisata di atlantis", 10),
    "V3": cari_wisata_v3("tempat wisata di atlantis", 10),
    "V4": cari_wisata_v4("tempat wisata di atlantis", 10),
    "V5": cari_wisata_v5("tempat wisata di atlantis", 10),
    "V6": cari_wisata_v6("tempat wisata di atlantis", 10),
    "V7": cari_wisata_v7("tempat wisata di atlantis", 10)
}

hasil_kepri_per_versi = {
    "V2": cari_wisata_v2("tempat wisata di kepulauan riau", 10),
    "V3": cari_wisata_v3("tempat wisata di kepulauan riau", 10),
    "V4": cari_wisata_v4("tempat wisata di kepulauan riau", 10),
    "V5": cari_wisata_v5("tempat wisata di kepulauan riau", 10),
    "V6": cari_wisata_v6("tempat wisata di kepulauan riau", 10),
    "V7": cari_wisata_v7("tempat wisata di kepulauan riau", 10)
}

In [41]:
perbandingan_akhir = pd.DataFrame([
    {
        "Versi": versi,
        "Hasil_Bandung": len(hasil_bandung_per_versi[versi]),
        "Bandung_Sesuai": int(
            hasil_bandung_per_versi[versi]["City"]
            .fillna("")
            .astype(str)
            .str.lower()
            .str.contains("bandung")
            .sum()
        ),
        "Hasil_Papua_Barat": len(hasil_papua_per_versi[versi]),
        "Papua_Barat_Sesuai": int(
            hasil_papua_per_versi[versi]["Province"]
            .fillna("")
            .astype(str)
            .str.lower()
            .eq("papua barat")
            .sum()
        ),
        "Hasil_Atlantis": len(hasil_atlantis_per_versi[versi]),
        "Hasil_Kepri": len(hasil_kepri_per_versi[versi])
    }
    for versi in ["V2", "V3", "V4", "V5", "V6", "V7"]
])

display(perbandingan_akhir)

,Versi,Hasil_Bandung,Bandung_Sesuai,Hasil_Papua_Barat,Papua_Barat_Sesuai,Hasil_Atlantis,Hasil_Kepri
0,V2,10,3,10,0,10,10
1,V3,10,3,10,0,10,10
2,V4,10,3,10,0,10,10
3,V5,4,4,10,0,10,10
4,V6,4,4,3,3,10,0
5,V7,4,4,3,3,0,0


In [42]:
file_perbandingan_akhir = (
    BASE_DIR / "data" / "processed"
    / "perbandingan_akhir_v2_v7.csv"
)

perbandingan_akhir.to_csv(
    file_perbandingan_akhir,
    index=False,
    encoding="utf-8-sig"
)

print("File tersimpan:", file_perbandingan_akhir)

File tersimpan: D:\WisataFinder\data\processed\perbandingan_akhir_v2_v7.csv


In [43]:
cek_perbandingan_akhir = pd.read_csv(
    file_perbandingan_akhir
)

print("Ukuran tabel:", cek_perbandingan_akhir.shape)
display(cek_perbandingan_akhir)

Ukuran tabel: (6, 7)


,Versi,Hasil_Bandung,Bandung_Sesuai,Hasil_Papua_Barat,Papua_Barat_Sesuai,Hasil_Atlantis,Hasil_Kepri
0,V2,10,3,10,0,10,10
1,V3,10,3,10,0,10,10
2,V4,10,3,10,0,10,10
3,V5,4,4,10,0,10,10
4,V6,4,4,3,3,10,0
5,V7,4,4,3,3,0,0



## 6. Evaluasi Perbandingan Algoritma V2–V7

Pengujian perbandingan dilakukan menggunakan empat skenario, yaitu pencarian museum di Bandung, tempat wisata di Papua Barat, tempat wisata di Atlantis, dan tempat wisata di Kepulauan Riau.

Hasil pengujian menunjukkan adanya peningkatan kemampuan pencarian secara bertahap.

V2 hingga V4 belum mampu mengenali lokasi kota dan provinsi secara menyeluruh. V5 meningkatkan ketepatan pencarian berdasarkan kota melalui pengenalan nama kota otomatis.

V6 selanjutnya mengembangkan pengenalan provinsi secara otomatis dan berhasil menampilkan hasil yang sesuai pada pengujian Papua Barat.

V7 menambahkan validasi lokasi sehingga sistem dapat membedakan lokasi yang tidak dikenal dan lokasi yang dikenal tetapi belum memiliki data.

Berdasarkan hasil pengujian tersebut, V7 dipilih sebagai kandidat algoritma utama untuk pengembangan WisataFinder.

Evaluasi ini masih terbatas pada skenario pengujian yang telah ditentukan dan belum menggambarkan seluruh aspek relevansi hasil pencarian.


In [45]:
data_label_selesai = pd.read_csv(
    file_label_csv,
    sep=None,
    engine="python",
    encoding="utf-8-sig"
)

print("Ukuran data:", data_label_selesai.shape)
print("\nNama kolom:")
print(data_label_selesai.columns.tolist())

display(data_label_selesai.head())

Ukuran data: (56, 8)

Nama kolom:
['Query', 'Posisi', 'Place_Name', 'Category', 'City', 'Province', 'Relevan', 'Catatan']


,Query,Posisi,Place_Name,Category,City,Province,Relevan,Catatan
0,Pantai di Bali,1,Pantai Amed,pantai,Karangasem,Bali,1,NaN
1,Pantai di Bali,2,Pantai Candidasa,pantai,Karangasem,Bali,1,NaN
2,Pantai di Bali,3,Pantai Balian,pantai,Tabanan,Bali,1,NaN
3,Pantai di Bali,4,Pantai Nusa Dua,pantai,Badung,Bali,1,NaN
4,Pantai di Bali,5,Pantai Seminyak,pantai,Badung,Bali,1,NaN


In [46]:
print("Distribusi label:")
print(data_label_selesai["Relevan"].value_counts(dropna=False))

print("\nJumlah label kosong:")
print(data_label_selesai["Relevan"].isna().sum())

Distribusi label:
Relevan
1    55
0     1
Name: count, dtype: int64

Jumlah label kosong:
0


In [47]:
display(
    data_label_selesai.loc[
        data_label_selesai["Relevan"] == 0,
        ["Query", "Place_Name", "Category",
         "City", "Province", "Relevan", "Catatan"]
    ]
)

,Query,Place_Name,Category,City,Province,Relevan,Catatan
12,Wisata budaya di Jawa Tengah,Candi Cetho,candi,Ngawi,Jawa Tengah,0,NaN


In [48]:
ringkasan_label = (
    data_label_selesai
    .groupby("Query")
    .agg(
        Jumlah_Hasil=("Relevan", "size"),
        Jumlah_Relevan=("Relevan", "sum"),
        Jumlah_Tidak_Relevan=(
            "Relevan",
            lambda x: (x == 0).sum()
        )
    )
    .reset_index()
)

display(ringkasan_label)

,Query,Jumlah_Hasil,Jumlah_Relevan,Jumlah_Tidak_Relevan
0,Gunung dan wisata alam,10,10,0
1,Kebun binatang di Jakarta,1,1,0
2,Museum,10,10,0
3,Museum di Bandung,4,4,0
4,Museum di Semarang,1,1,0
5,Pantai di Bali,10,10,0
6,Tempat wisata di Bali,10,10,0
7,Wisata budaya di Jawa Tengah,10,9,1


In [49]:
data_label_selesai = pd.read_csv(
    file_label_csv,
    sep=None,
    engine="python",
    encoding="utf-8-sig"
)

data_label_selesai["Relevan"] = pd.to_numeric(
    data_label_selesai["Relevan"],
    errors="coerce"
)

print(data_label_selesai["Relevan"].value_counts(dropna=False))

Relevan
1    55
0     1
Name: count, dtype: int64


In [50]:
evaluasi_precision = (
    data_label_selesai
    .groupby("Query")
    .agg(
        Jumlah_Hasil=("Relevan", "size"),
        Jumlah_Relevan=("Relevan", "sum"),
        Jumlah_Tidak_Relevan=(
            "Relevan",
            lambda x: (x == 0).sum()
        )
    )
    .reset_index()
)

# Precision@10 menggunakan penyebut tetap 10
evaluasi_precision["Precision@10"] = (
    evaluasi_precision["Jumlah_Relevan"] / 10
)

# Precision berdasarkan jumlah hasil yang ditemukan
evaluasi_precision["Precision_Hasil_Ditemukan"] = (
    evaluasi_precision["Jumlah_Relevan"]
    / evaluasi_precision["Jumlah_Hasil"]
)

display(evaluasi_precision)

,Query,Jumlah_Hasil,Jumlah_Relevan,Jumlah_Tidak_Relevan,Precision@10,Precision_Hasil_Ditemukan
0,Gunung dan wisata alam,10,10,0,1.0,1.0
1,Kebun binatang di Jakarta,1,1,0,0.1,1.0
2,Museum,10,10,0,1.0,1.0
3,Museum di Bandung,4,4,0,0.4,1.0
4,Museum di Semarang,1,1,0,0.1,1.0
5,Pantai di Bali,10,10,0,1.0,1.0
6,Tempat wisata di Bali,10,10,0,1.0,1.0
7,Wisata budaya di Jawa Tengah,10,9,1,0.9,0.9


In [51]:
print(
    "Rata-rata Precision@10:",
    round(evaluasi_precision["Precision@10"].mean() * 100, 2),
    "%"
)

print(
    "Rata-rata Precision hasil ditemukan:",
    round(evaluasi_precision["Precision_Hasil_Ditemukan"].mean() * 100, 2),
    "%"
)

Rata-rata Precision@10: 68.75 %
Rata-rata Precision hasil ditemukan: 98.75 %


In [53]:
# Mengubah kolom Catatan menjadi tipe teks
data_label_selesai["Catatan"] = (
    data_label_selesai["Catatan"].astype("string")
)

# Memastikan kolom Relevan bertipe numerik
data_label_selesai["Relevan"] = pd.to_numeric(
    data_label_selesai["Relevan"],
    errors="coerce"
)

print(data_label_selesai.dtypes)

Query                 object
Posisi                 int64
Place_Name            object
Category              object
City                  object
Province              object
Relevan                int64
Catatan       string[python]
dtype: object


In [54]:
display(
    data_label_selesai.loc[
        data_label_selesai["Place_Name"] == "Candi Cetho",
        ["Place_Name", "City", "Province", "Relevan", "Catatan"]
    ]
)

,Place_Name,City,Province,Relevan,Catatan
12,Candi Cetho,Ngawi,Jawa Tengah,1,Relevan. Lokasi sebenarnya Kabupaten Karangany...


In [55]:
evaluasi_precision = (
    data_label_selesai
    .groupby("Query")
    .agg(
        Jumlah_Hasil=("Relevan", "size"),
        Jumlah_Relevan=("Relevan", "sum"),
        Jumlah_Tidak_Relevan=(
            "Relevan",
            lambda x: (x == 0).sum()
        )
    )
    .reset_index()
)

evaluasi_precision["Precision@10"] = (
    evaluasi_precision["Jumlah_Relevan"] / 10
)

evaluasi_precision["Precision_Hasil_Ditemukan"] = (
    evaluasi_precision["Jumlah_Relevan"]
    / evaluasi_precision["Jumlah_Hasil"]
)

display(evaluasi_precision)

,Query,Jumlah_Hasil,Jumlah_Relevan,Jumlah_Tidak_Relevan,Precision@10,Precision_Hasil_Ditemukan
0,Gunung dan wisata alam,10,10,0,1.0,1.0
1,Kebun binatang di Jakarta,1,1,0,0.1,1.0
2,Museum,10,10,0,1.0,1.0
3,Museum di Bandung,4,4,0,0.4,1.0
4,Museum di Semarang,1,1,0,0.1,1.0
5,Pantai di Bali,10,10,0,1.0,1.0
6,Tempat wisata di Bali,10,10,0,1.0,1.0
7,Wisata budaya di Jawa Tengah,10,10,0,1.0,1.0


In [56]:
print(
    "Rata-rata Precision@10:",
    round(evaluasi_precision["Precision@10"].mean() * 100, 2),
    "%"
)

print(
    "Rata-rata Precision hasil ditemukan:",
    round(
        evaluasi_precision["Precision_Hasil_Ditemukan"].mean() * 100,
        2
    ),
    "%"
)

Rata-rata Precision@10: 70.0 %
Rata-rata Precision hasil ditemukan: 100.0 %



## 7. Evaluasi Relevansi Hasil Pencarian V7

Evaluasi relevansi dilakukan secara manual terhadap 56 hasil pencarian yang berasal dari delapan query pengujian.

Setiap hasil diberikan label relevansi, yaitu 1 untuk hasil yang sesuai dengan kebutuhan pencarian dan 0 untuk hasil yang tidak sesuai.

Hasil evaluasi menunjukkan bahwa sebanyak 56 destinasi dinilai relevan dan tidak ditemukan hasil yang dinilai tidak relevan setelah pemeriksaan metadata Candi Cetho.

Berdasarkan perhitungan, diperoleh rata-rata Precision@10 sebesar 70% dan rata-rata Precision hasil ditemukan sebesar 100%.

Perbedaan kedua nilai tersebut dipengaruhi oleh jumlah hasil yang ditemukan pada setiap query. Beberapa pencarian hanya menghasilkan satu atau empat destinasi sehingga tidak memenuhi sepuluh posisi hasil yang dievaluasi.

Hasil ini menunjukkan bahwa destinasi yang berhasil ditampilkan pada skenario pengujian telah memenuhi kriteria relevansi yang ditentukan. Namun, jumlah data yang terbatas pada beberapa wilayah masih menjadi kendala dalam menyediakan sepuluh hasil pencarian.

Evaluasi ini bersifat terbatas pada delapan query dan 56 hasil yang dinilai secara manual. Oleh karena itu, hasil belum dapat digeneralisasikan untuk seluruh kemungkinan pencarian pengguna.


In [57]:
data_label_selesai.to_csv(
    BASE_DIR / "data" / "processed"
    / "label_relevansi_manual_v7_terverifikasi.csv",
    index=False,
    encoding="utf-8-sig"
)

evaluasi_precision.to_csv(
    BASE_DIR / "data" / "processed"
    / "evaluasi_precision_v7.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Label:", len(data_label_selesai), "baris")
print("Evaluasi:", len(evaluasi_precision), "query")
print(
    "File label tersimpan:",
    (BASE_DIR / "data" / "processed" /
     "label_relevansi_manual_v7_terverifikasi.csv").exists()
)
print(
    "File evaluasi tersimpan:",
    (BASE_DIR / "data" / "processed" /
     "evaluasi_precision_v7.csv").exists()
)

Label: 56 baris
Evaluasi: 8 query
File label tersimpan: True
File evaluasi tersimpan: True


In [58]:
# Memeriksa file hasil evaluasi

file_yang_diperiksa = [
    BASE_DIR / "data" / "processed" / "perbandingan_akhir_v2_v7.csv",
    BASE_DIR / "data" / "processed" / "evaluasi_provinsi_v7.csv",
    BASE_DIR / "data" / "processed" / "label_relevansi_manual_v7_terverifikasi.csv",
    BASE_DIR / "data" / "processed" / "evaluasi_precision_v7.csv"
]

for file in file_yang_diperiksa:
    print(file.name, ":", file.exists())

perbandingan_akhir_v2_v7.csv : True
evaluasi_provinsi_v7.csv : True
label_relevansi_manual_v7_terverifikasi.csv : True
evaluasi_precision_v7.csv : True


In [59]:
# Lokasi aplikasi utama
file_app = BASE_DIR / "app.py"

print("Lokasi app.py:", file_app)
print("Apakah app.py tersedia:", file_app.exists())

Lokasi app.py: D:\WisataFinder\app.py
Apakah app.py tersedia: True


In [60]:
# Membaca kode aplikasi tanpa mengubahnya
kode_app = file_app.read_text(encoding="utf-8")
baris_app = kode_app.splitlines()

print("Jumlah baris app.py:", len(baris_app))

print("\n=== BAGIAN PENTING APP.PY ===")

for nomor, baris in enumerate(baris_app, start=1):
    if re.search(
        r"DATA_PATH|def load_data|TfidfVectorizer|cosine_similarity|def search|Location_Label|Location_Verification|search_text",
        baris,
        flags=re.IGNORECASE
    ):
        print(f"Baris {nomor}: {baris.strip()}")

Jumlah baris app.py: 1662

=== BAGIAN PENTING APP.PY ===
Baris 9: from sklearn.feature_extraction.text import TfidfVectorizer
Baris 10: from sklearn.metrics.pairwise import cosine_similarity
Baris 24: DATA_PATH = BASE_DIR / "data" / "processed" / "destinations_clean.csv"
Baris 670: def load_data():
Baris 671: if not DATA_PATH.exists():
Baris 673: f"Dataset tidak ditemukan: {DATA_PATH}"
Baris 676: data = pd.read_csv(DATA_PATH)
Baris 695: data["search_text"] = (
Baris 706: vectorizer = TfidfVectorizer(
Baris 719: tuple(df["search_text"].tolist())
Baris 999: if "Location_Label" in df.columns:
Baris 1000: location_column = "Location_Label"
Baris 1189: and "Location_Verification" in df.columns
Baris 1198: & df["Location_Verification"]
Baris 1231: scores = cosine_similarity(
Baris 1251: if "Location_Verification" in results.columns:
Baris 1253: ~results["Location_Verification"]
Baris 1270: and "Location_Verification" in df.columns
Baris 1279: & df["Location_Verification"]


In [61]:
print("\n".join(baris_app[:90]))


from pathlib import Path

import pandas as pd
import streamlit as st
import requests
import re
from urllib.parse import quote
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# =====================================================
# CONFIGURATION
# =====================================================
st.set_page_config(
    page_title="WisataFinder | Explore Indonesia",
    page_icon="🌿",
    layout="wide",
    initial_sidebar_state="collapsed",
)

BASE_DIR = Path(__file__).resolve().parent
DATA_PATH = BASE_DIR / "data" / "processed" / "destinations_clean.csv"


# =====================================================
# CUSTOM DESIGN
# =====================================================
st.markdown(
    """
    <style>
    @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=Playfair+Display:wght@500;600;700&display=swap');

    :root {
        --forest: #214E3D;
        --forest

In [62]:
import shutil

# Membuat folder cadangan
folder_backup = BASE_DIR / "backups"
folder_backup.mkdir(exist_ok=True)

# Lokasi file cadangan
file_backup = folder_backup / "app_sebelum_integrasi_v7.py"

# Jangan menimpa cadangan yang sudah ada
if not file_backup.exists():
    shutil.copy2(file_app, file_backup)
    print("Backup berhasil dibuat.")
else:
    print("File backup sudah tersedia. Tidak ditimpa.")

print("Lokasi backup:", file_backup)
print("Ukuran file:", file_backup.stat().st_size, "bytes")

Backup berhasil dibuat.
Lokasi backup: D:\WisataFinder\backups\app_sebelum_integrasi_v7.py
Ukuran file: 47892 bytes


In [63]:
# Menampilkan daftar fungsi dalam app.py

for nomor, baris in enumerate(baris_app, start=1):
    if re.match(r"^\s*def\s+", baris):
        print(f"Baris {nomor}: {baris.strip()}")

Baris 670: def load_data():
Baris 705: def build_index(texts):
Baris 730: def find_destination_image(place_name, city):
Baris 746: def normalize(text):
Baris 819: def match_score(title):
Baris 1034: def filter_worship_results(results, query):


In [64]:
print("\n".join(baris_app[660:735]))

    """,
    unsafe_allow_html=True,
)


# =====================================================
# DATA AND SEARCH INDEX
# =====================================================
@st.cache_data
def load_data():
    if not DATA_PATH.exists():
        raise FileNotFoundError(
            f"Dataset tidak ditemukan: {DATA_PATH}"
        )

    data = pd.read_csv(DATA_PATH)
    data.columns = data.columns.str.strip()

    required = ["Place_Name", "Category", "City", "Description"]
    missing = [col for col in required if col not in data.columns]

    if missing:
        raise ValueError(
            f"Kolom berikut belum tersedia: {missing}. "
            f"Kolom dataset: {data.columns.tolist()}"
        )

    for col in required:
        data[col] = data[col].fillna("").astype(str).str.strip()

    for col in ["Price", "Rating"]:
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors="coerce")

    data["search_text"] = (
        data["Place_Name"] + " "
 

In [65]:
print("\n".join(baris_app[975:1030]))

# SEARCH PANEL
# =====================================================
st.markdown(
    """
    <div class="section-eyebrow">MULAI MENJELAJAH</div>
    <div class="section-title">Ke mana perjalananmu?</div>
    <div class="section-subtitle">
        Masukkan kata kunci untuk menemukan destinasi
        berdasarkan kemiripan informasi.
    </div>
    """,
    unsafe_allow_html=True,
)


with st.form("search_form"):
    query = st.text_input(
        "Kata kunci",
        placeholder="Contoh: wisata alam di Semarang",
    )

    f1, f2 = st.columns(2)

    if "Location_Label" in df.columns:
        location_column = "Location_Label"
    else:
        location_column = "City"

    city_options = sorted(
        df[location_column].dropna().unique()
    )

    category_options = sorted(
        category
        for category in df["Category"].dropna().unique()
        if category
    )

    with f1:
        selected_city = st.selectbox(
            "Pilih kota",
            ["Semua kota"] +

In [66]:
print("\n".join(baris_app[1160:1300]))

# =====================================================
# RESULTS
# =====================================================
if submitted:
    if not query.strip():
        st.warning("Masukkan kata kunci terlebih dahulu.")
    else:
        
        # Deteksi nama kota yang disebutkan dalam kata kunci
        query_lower = query.casefold()
        detected_city = None

        for city_name in city_options:
            if city_name.casefold() in query_lower:
                detected_city = city_name
                break

        # Kota dalam kata kunci diprioritaskan.
        # Jika tidak ada, gunakan pilihan kota pada dropdown.
        effective_city = (
            detected_city
            if detected_city is not None
            else selected_city
        )
                # Beri peringatan jika destinasi bermasalah dicari
        # dengan filter kota yang tidak sesuai.
        if (
            effective_city != "Semua kota"
            and "Location_Verification" in df.columns
    

In [67]:
print("\n".join(baris_app[1180:1320]))

            detected_city
            if detected_city is not None
            else selected_city
        )
                # Beri peringatan jika destinasi bermasalah dicari
        # dengan filter kota yang tidak sesuai.
        if (
            effective_city != "Semua kota"
            and "Location_Verification" in df.columns
        ):
            excluded_place = df[
                df["Place_Name"].str.contains(
                    query.strip(),
                    case=False,
                    na=False,
                    regex=False,
                )
                & df["Location_Verification"]
                .fillna("")
                .str.startswith("Perlu verifikasi")
            ]

            if not excluded_place.empty:
                st.warning(
                    "Benteng Pendem ditemukan, tetapi tidak ditampilkan "
                    "karena metadata lokasinya perlu diverifikasi. "
                    "Pilih 'Semua kota' untuk mencari tanpa filter kota."
 

In [68]:
folder_backup = BASE_DIR / "backups"
file_backup = folder_backup / "app_sebelum_integrasi_v7.py"

print("Folder backup tersedia:", folder_backup.exists())
print("File backup tersedia:", file_backup.exists())

Folder backup tersedia: True
File backup tersedia: True


In [69]:
print("\n".join(baris_app[1280:1430]))

                .str.startswith("Perlu verifikasi")
            ]

            if not excluded_place.empty:
                st.markdown(
    """
    <p style="color: black; font-size: 16px;">
        <b>Lokasi destinasi belum terverifikasi</b><br>
        Benteng Pendem ditemukan, tetapi tidak ditampilkan karena
        metadata lokasinya perlu diverifikasi. Pilih
        <b>Semua kota</b> untuk mencari tanpa filter kota.
    </p>
    """,
    unsafe_allow_html=True,
)
                st.stop()
        # Hanya tampilkan hasil dengan skor kemiripan > 0
        results = results[
            results["similarity_score"] > 0
        ].copy()

        

        
        
        # Prioritaskan kecocokan nama destinasi untuk kueri jenis tempat.
        # TF-IDF + Cosine Similarity tetap menjadi metode utama.

        query_lower = query.casefold()

        # Pertahankan filter khusus tempat ibadah yang sudah ada.
        results = filter_worship_results(results, query)

       
        # Pol

In [70]:
print("\n".join(baris_app[1430:1580]))



        st.write("")

        st.markdown(
            '<div class="section-eyebrow">HASIL PENCARIAN</div>',
            unsafe_allow_html=True
        )

        st.markdown(
            '<div class="section-title">Destinasi untukmu</div>',
            unsafe_allow_html=True
        )

        if results.empty:
            st.info(
                "Belum ditemukan destinasi yang cocok. "
                "Coba kata kunci lain atau ubah filter kota dan kategori."
            )
        else:
            st.success(
                f"Ditemukan {len(results)} destinasi yang relevan."
            )

            top_results = results.head(20)

            for rank, (_, item) in enumerate(
                top_results.iterrows(),
                start=1
            ):
                name = item["Place_Name"]
                city = item[location_column]
                category = item["Category"]
                description = item["Description"]
                if pd.notna(description):
    

In [71]:
print("\n".join(baris_app[1025:1185]))

    submitted = st.form_submit_button(
        "⌕  Temukan Destinasi",
        use_container_width=True,
    )

# =====================================================
# FILTER MAKSUD PENCARIAN TEMPAT IBADAH
# =====================================================
def filter_worship_results(results, query):
    q = query.casefold().strip()

    # Kueri spesifik klenteng/kelenteng:
    # identitas destinasi harus cocok melalui nama
    # atau kategori, bukan hanya penyebutan di deskripsi.
    if re.search(r"\b(klenteng|kelenteng)\b", q):
        place_name = (
            results["Place_Name"]
            .fillna("")
            .astype(str)
            .str.casefold()
        )

        category = (
            results["Category"]
            .fillna("")
            .astype(str)
            .str.casefold()
        )

        identity = place_name + " " + category

        return results[
            identity.str.contains(
                r"\b(klenteng|kelenteng)\b",
                rege

In [72]:
print("\n".join(baris_app[1570:1662]))

                mime="text/csv"
            )
# =====================================================
# INSPIRATION SECTION
# =====================================================
else:
    st.write("")
    st.markdown(
        """
        <div class="section-eyebrow">INSPIRASI PERJALANAN</div>
        <div class="section-title">Mulai dari yang kamu suka.</div>
        <div class="section-subtitle">
            Pilih ide perjalanan, lalu gunakan kata kunci tersebut
            untuk memulai pencarian.
        </div>
        """,
        unsafe_allow_html=True,
    )

    c1, c2, c3 = st.columns(3)

    with c1:
        st.markdown(
            """
            <div class="inspiration-card">
                <div class="inspiration-icon">🌿</div>
                <h3>Menjelajah alam</h3>
                <p>
                    Temukan taman, kawasan alam, dan destinasi
                    dengan pemandangan yang menenangkan.
                </p>
            </div>
            """,
         

In [73]:
import shutil

file_app_v7 = BASE_DIR / "app_nasional_v7.py"

if not file_app_v7.exists():
    shutil.copy2(file_app, file_app_v7)
    print("Salinan aplikasi berhasil dibuat.")
else:
    print("Salinan aplikasi sudah tersedia.")

print("Lokasi:", file_app_v7)

Salinan aplikasi berhasil dibuat.
Lokasi: D:\WisataFinder\app_nasional_v7.py


In [74]:
DATA_PATH = BASE_DIR / "data" / "processed" / "destinations_clean.csv"

In [76]:
for nomor, baris in enumerate(kode_test.splitlines(), start=1):
    if "DATA_PATH" in baris:
        print(f"Baris {nomor}: {baris!r}")

Baris 24: 'DATA_PATH = BASE_DIR / "data" / "processed" / "destinations_clean.csv"'
Baris 671: '    if not DATA_PATH.exists():'
Baris 673: '            f"Dataset tidak ditemukan: {DATA_PATH}"'
Baris 676: '    data = pd.read_csv(DATA_PATH)'


In [77]:
from pathlib import Path

# Mencari folder utama proyek
folder_sekarang = Path.cwd()

if (folder_sekarang / "app.py").exists():
    folder_proyek = folder_sekarang
elif (folder_sekarang.parent / "app.py").exists():
    folder_proyek = folder_sekarang.parent
else:
    raise FileNotFoundError("File app.py tidak ditemukan.")

# Membaca app.py asli
file_app = folder_proyek / "app.py"
kode_test = file_app.read_text(encoding="utf-8")

# Baris lokasi dataset lama dan baru
path_lama = 'DATA_PATH = BASE_DIR / "data" / "processed" / "destinations_clean.csv"'
path_baru = 'DATA_PATH = BASE_DIR / "data" / "processed" / "destinations_nasional.csv"'

# Pastikan baris lama ditemukan tepat satu kali
jumlah = kode_test.count(path_lama)

if jumlah != 1:
    raise ValueError(f"Baris DATA_PATH ditemukan {jumlah} kali, bukan satu kali.")

# Mengganti lokasi dataset
kode_test_v7 = kode_test.replace(path_lama, path_baru, 1)

# Memeriksa sintaks Python
compile(kode_test_v7, "app_nasional_v7.py", "exec")

# Menyimpan sebagai file aplikasi baru
file_app_v7 = folder_proyek / "app_nasional_v7.py"
file_app_v7.write_text(kode_test_v7, encoding="utf-8")

print("Berhasil membuat:", file_app_v7)
print("Dataset yang digunakan:", path_baru)
print("Pemeriksaan sintaks: LULUS")

Berhasil membuat: D:\WisataFinder\app_nasional_v7.py
Dataset yang digunakan: DATA_PATH = BASE_DIR / "data" / "processed" / "destinations_nasional.csv"
Pemeriksaan sintaks: LULUS


In [1]:
from pathlib import Path

file_app_v7 = Path(r"D:\WisataFinder\app_nasional_v7.py")
baris = file_app_v7.read_text(encoding="utf-8").splitlines()

for i, baris_kode in enumerate(baris):
    if "location_column" in baris_kode:
        awal = max(0, i - 8)
        akhir = min(len(baris), i + 75)

        for nomor in range(awal, akhir):
            print(f"{nomor + 1}: {baris[nomor]}")
        break
else:
    print("Kode location_column tidak ditemukan.")

992:     query = st.text_input(
993:         "Kata kunci",
994:         placeholder="Contoh: wisata alam di Semarang",
995:     )
996: 
997:     f1, f2 = st.columns(2)
998: 
999:     if "Location_Label" in df.columns:
1000:         location_column = "Location_Label"
1001:     else:
1002:         location_column = "City"
1003: 
1004:     city_options = sorted(
1005:         df[location_column].dropna().unique()
1006:     )
1007: 
1008:     category_options = sorted(
1009:         category
1010:         for category in df["Category"].dropna().unique()
1011:         if category
1012:     )
1013: 
1014:     with f1:
1015:         selected_city = st.selectbox(
1016:             "Pilih kota",
1017:             ["Semua kota"] + city_options,
1018:         )
1019: 
1020:     with f2:
1021:         selected_category = st.selectbox(
1022:             "Pilih kategori",
1023:             ["Semua kategori"] + category_options,
1024:         )
1025: 
1026:     submitted = st.form_submit_button(
1027

In [2]:
from pathlib import Path

file_app_v7 = Path(r"D:\WisataFinder\app_nasional_v7.py")
baris = file_app_v7.read_text(encoding="utf-8").splitlines()

for nomor in range(1074, min(len(baris), 1270)):
    print(f"{nomor + 1}: {baris[nomor]}")

1075:         "buddha": [
1076:             "buddha", "buddhis", "vihara", "wihara",
1077:             "tridharma"
1078:         ],
1079:     }
1080: 
1081:     detected_type = None
1082: 
1083:     for religion, keywords in worship_types.items():
1084:         if any(word in q for word in keywords):
1085:             detected_type = religion
1086:             break
1087: 
1088:     # Kueri umum tidak memerlukan filter agama.
1089:     if detected_type is None:
1090:         return results
1091: 
1092:     place_name = (
1093:         results["Place_Name"]
1094:         .fillna("")
1095:         .astype(str)
1096:         .str.casefold()
1097:     )
1098: 
1099:     category = (
1100:         results["Category"]
1101:         .fillna("")
1102:         .astype(str)
1103:         .str.casefold()
1104:     )
1105: 
1106:     description = (
1107:         results["Description"]
1108:         .fillna("")
1109:         .astype(str)
1110:         .str.casefold()
1111:     )
1112: 
1113:     i

In [3]:
for nomor in range(1266, min(len(baris), 1410)):
    print(f"{nomor + 1}: {baris[nomor]}")

1267:         # Tangani destinasi yang dikecualikan karena masalah lokasi.
1268:         if (
1269:             effective_city != "Semua kota"
1270:             and "Location_Verification" in df.columns
1271:         ):
1272:             excluded_place = df[
1273:                 df["Place_Name"].str.contains(
1274:                     query.strip(),
1275:                     case=False,
1276:                     na=False,
1277:                     regex=False,
1278:                 )
1279:                 & df["Location_Verification"]
1280:                 .fillna("")
1281:                 .str.startswith("Perlu verifikasi")
1282:             ]
1283: 
1284:             if not excluded_place.empty:
1285:                 st.markdown(
1286:     """
1287:     <p style="color: black; font-size: 16px;">
1288:         <b>Lokasi destinasi belum terverifikasi</b><br>
1289:         Benteng Pendem ditemukan, tetapi tidak ditampilkan karena
1290:         metadata lokasinya perlu diverifikasi. Pil

In [4]:
from pathlib import Path
import pandas as pd
import re

# =====================================================
# 1. PERSIAPAN DAN PEMERIKSAAN DATA
# =====================================================

file_app_v7 = Path(r"D:\WisataFinder\app_nasional_v7.py")
file_dataset = Path(
    r"D:\WisataFinder\data\processed\destinations_nasional.csv"
)

if not file_app_v7.exists():
    raise FileNotFoundError("app_nasional_v7.py tidak ditemukan.")

if not file_dataset.exists():
    raise FileNotFoundError("Dataset nasional tidak ditemukan.")

df_pemeriksaan = pd.read_csv(file_dataset)

if "Province" not in df_pemeriksaan.columns:
    raise ValueError("Kolom Province tidak ditemukan dalam dataset.")

if not df_pemeriksaan["Province"].fillna("").astype(str).str.strip().str.casefold().eq("bali").any():
    raise ValueError("Provinsi Bali tidak ditemukan dalam dataset.")

kode = file_app_v7.read_text(encoding="utf-8")


# =====================================================
# 2. FUNGSI PENGGANTIAN BAGIAN KODE
# =====================================================

def ganti_bagian(teks, penanda_awal, penanda_akhir, pengganti, nama):
    if teks.count(penanda_awal) != 1:
        raise ValueError(f"Penanda awal bagian {nama} tidak ditemukan tepat satu kali.")

    if teks.count(penanda_akhir) != 1:
        raise ValueError(f"Penanda akhir bagian {nama} tidak ditemukan tepat satu kali.")

    posisi_awal = teks.index(penanda_awal)
    posisi_akhir = teks.index(penanda_akhir, posisi_awal)

    if posisi_akhir <= posisi_awal:
        raise ValueError(f"Urutan penanda bagian {nama} tidak sesuai.")

    return teks[:posisi_awal] + pengganti + teks[posisi_akhir:]


# =====================================================
# 3. MENAMBAHKAN FILTER PROVINSI
# =====================================================

bagian_filter = '''    f1, f2, f3 = st.columns(3)

    if "Location_Label" in df.columns:
        location_column = "Location_Label"
    else:
        location_column = "City"

    city_options = sorted(
        {
            str(value).strip()
            for value in df[location_column].dropna().unique()
            if str(value).strip()
        },
        key=str.casefold,
    )

    province_options = sorted(
        {
            str(value).strip()
            for value in df["Province"].dropna().unique()
            if str(value).strip()
        },
        key=str.casefold,
    ) if "Province" in df.columns else []

    category_options = sorted(
        {
            str(value).strip()
            for value in df["Category"].dropna().unique()
            if str(value).strip()
        },
        key=str.casefold,
    )

    with f1:
        selected_province = st.selectbox(
            "Pilih provinsi",
            ["Semua provinsi"] + province_options,
        )

    with f2:
        selected_city = st.selectbox(
            "Pilih kota/kabupaten",
            ["Semua kota"] + city_options,
        )

    with f3:
        selected_category = st.selectbox(
            "Pilih kategori",
            ["Semua kategori"] + category_options,
        )

'''

kode = ganti_bagian(
    kode,
    "    f1, f2 = st.columns(2)",
    "    submitted = st.form_submit_button(",
    bagian_filter,
    "dropdown lokasi dan kategori",
)


# =====================================================
# 4. DETEKSI PROVINSI DAN KOTA DARI KATA KUNCI
# =====================================================

bagian_deteksi = '''        query_lower = query.casefold()

        # Deteksi provinsi dari kata kunci.
        detected_province = None
        detected_province_term = None

        for province_name in sorted(
            province_options,
            key=len,
            reverse=True,
        ):
            pola = rf"(?<!\\\\w){re.escape(province_name.casefold())}(?!\\\\w)"
            cocok = re.search(pola, query_lower)

            if cocok:
                detected_province = province_name
                detected_province_term = cocok.group(0)
                break

        # Alias untuk beberapa nama provinsi.
        province_aliases = {
            "Daerah Istimewa Yogyakarta": [
                "diy", "jogja", "yogya"
            ],
            "DKI Jakarta": [
                "dki", "jakarta"
            ],
            "Jawa Barat": [
                "jabar"
            ],
            "Jawa Tengah": [
                "jateng"
            ],
            "Jawa Timur": [
                "jatim"
            ],
            "Nusa Tenggara Barat": [
                "ntb", "lombok"
            ],
            "Nusa Tenggara Timur": [
                "ntt"
            ],
        }

        if detected_province is None:
            province_lookup = {
                province.casefold(): province
                for province in province_options
            }

            for nama_provinsi, daftar_alias in province_aliases.items():
                provinsi_asli = province_lookup.get(
                    nama_provinsi.casefold()
                )

                if provinsi_asli is None:
                    continue

                for alias in daftar_alias:
                    pola = rf"(?<!\\\\w){re.escape(alias)}(?!\\\\w)"
                    cocok = re.search(pola, query_lower)

                    if cocok:
                        detected_province = provinsi_asli
                        detected_province_term = alias
                        break

                if detected_province is not None:
                    break

        # Deteksi kota/kabupaten dari kata kunci.
        detected_city = None

        for city_name in sorted(
            city_options,
            key=len,
            reverse=True,
        ):
            pola = rf"(?<!\\\\w){re.escape(city_name.casefold())}(?!\\\\w)"

            if re.search(pola, query_lower):
                detected_city = city_name
                break

        # Nama lokasi dari kata kunci diprioritaskan
        # dibanding pilihan dropdown.
        effective_province = (
            detected_province
            if detected_province is not None
            else selected_province
        )

        effective_city = (
            detected_city
            if detected_city is not None
            else selected_city
        )

        location_filter_active = (
            effective_province != "Semua provinsi"
            or effective_city != "Semua kota"
        )

        # Hilangkan nama provinsi/kota dari teks pencarian
        # agar TF-IDF lebih fokus pada jenis destinasi.
        search_query = query

        lokasi_dihapus = []

        if detected_province_term:
            lokasi_dihapus.append(detected_province_term)

        if detected_city:
            lokasi_dihapus.append(detected_city)

        for lokasi in sorted(
            set(lokasi_dihapus),
            key=len,
            reverse=True,
        ):
            search_query = re.sub(
                rf"\\\\b(?:di\\\\s+)?{re.escape(lokasi)}\\\\b",
                " ",
                search_query,
                flags=re.IGNORECASE,
            )

        search_query = re.sub(r"\\\\s+", " ", search_query).strip(" ,.-")

        if not search_query:
            search_query = query

'''

kode = ganti_bagian(
    kode,
    "        # Deteksi nama kota yang disebutkan dalam kata kunci",
    "        # Hitung TF-IDF dan cosine similarity",
    bagian_deteksi,
    "deteksi lokasi",
)


# =====================================================
# 5. MENERAPKAN FILTER PROVINSI DAN KOTA
# =====================================================

bagian_penerapan = '''        # Filter berdasarkan provinsi.
        if effective_province != "Semua provinsi":
            results = results[
                results["Province"]
                .fillna("")
                .astype(str)
                .str.strip()
                .str.casefold()
                == effective_province.casefold()
            ]

        # Filter berdasarkan kota/kabupaten.
        if effective_city != "Semua kota":
            results = results[
                results[location_column]
                .fillna("")
                .astype(str)
                .str.strip()
                .str.casefold()
                == effective_city.casefold()
            ]

        # Jika lokasi dibatasi, jangan tampilkan data
        # yang status verifikasi lokasinya masih bermasalah.
        if location_filter_active and "Location_Verification" in results.columns:
            results = results[
                ~results["Location_Verification"]
                .fillna("")
                .astype(str)
                .str.startswith("Perlu verifikasi")
            ]

'''

kode = ganti_bagian(
    kode,
    "        # Terapkan filter berdasarkan label lokasi",
    "        # Filter berdasarkan kategori",
    bagian_penerapan,
    "filter provinsi dan kota",
)


# =====================================================
# 6. MEMPERBAIKI PESAN PERINGATAN LOKASI
# =====================================================

bagian_peringatan = '''        # Beri informasi jika ada data yang belum terverifikasi.
        if location_filter_active and "Location_Verification" in df.columns:
            unverified_matches = df[
                df["Place_Name"]
                .fillna("")
                .astype(str)
                .str.contains(
                    search_query,
                    case=False,
                    na=False,
                    regex=False,
                )
                & df["Location_Verification"]
                .fillna("")
                .astype(str)
                .str.startswith("Perlu verifikasi")
            ]

            if not unverified_matches.empty:
                st.info(
                    "Sebagian destinasi yang cocok tidak ditampilkan "
                    "karena informasi lokasinya belum terverifikasi."
                )

'''

kode = ganti_bagian(
    kode,
    "        # Tangani destinasi yang dikecualikan karena masalah lokasi.",
    "        # Hanya tampilkan hasil dengan skor kemiripan > 0",
    bagian_peringatan,
    "peringatan verifikasi lokasi",
)


# =====================================================
# 7. PEMERIKSAAN SINTAKS DAN PENYIMPANAN
# =====================================================

# Pastikan file masih valid secara sintaks Python.
compile(kode, str(file_app_v7), "exec")

# Buat cadangan file sebelum perubahan.
file_backup = file_app_v7.with_name("app_nasional_v7.py.bak")

if not file_backup.exists():
    file_backup.write_text(
        file_app_v7.read_text(encoding="utf-8"),
        encoding="utf-8",
    )

# Simpan aplikasi yang sudah diperbaiki.
file_app_v7.write_text(kode, encoding="utf-8")

print("Berhasil memperbarui filter aplikasi!")
print("File aplikasi :", file_app_v7)
print("File cadangan :", file_backup)
print("Provinsi Bali :", "tersedia dalam dataset")
print("Pemeriksaan sintaks: LULUS")

Berhasil memperbarui filter aplikasi!
File aplikasi : D:\WisataFinder\app_nasional_v7.py
File cadangan : D:\WisataFinder\app_nasional_v7.py.bak
Provinsi Bali : tersedia dalam dataset
Pemeriksaan sintaks: LULUS


In [5]:
from pathlib import Path
import re

file_app_v7 = Path(r"D:\WisataFinder\app_nasional_v7.py")

kode = file_app_v7.read_text(encoding="utf-8")

awal = "        # Deteksi provinsi dari kata kunci."
akhir = "        # Hitung TF-IDF dan cosine similarity"

if kode.count(awal) != 1 or kode.count(akhir) != 1:
    raise ValueError("Bagian deteksi lokasi tidak ditemukan dengan tepat.")

pos_awal = kode.index(awal)
pos_akhir = kode.index(akhir, pos_awal)

blok = kode[pos_awal:pos_akhir]

# Memperbaiki karakter escape regex pada blok yang baru ditambahkan.
blok_baru = blok.replace("\\\\", "\\")

kode_baru = kode[:pos_awal] + blok_baru + kode[pos_akhir:]

# Periksa sintaks sebelum menyimpan.
compile(kode_baru, str(file_app_v7), "exec")

file_app_v7.write_text(kode_baru, encoding="utf-8")

# Tes pola lokasi Bali.
hasil = re.sub(
    r"\b(?:di\s+)?bali\b",
    " ",
    "pantai di bali",
    flags=re.IGNORECASE
).strip()

print("Pemeriksaan sintaks: LULUS")
print("Hasil penghapusan lokasi:", repr(hasil))
print("Hasil yang diharapkan: 'pantai'")

Pemeriksaan sintaks: LULUS
Hasil penghapusan lokasi: 'pantai'
Hasil yang diharapkan: 'pantai'


In [1]:
import pandas as pd

df_periksa = pd.read_csv(
    r"D:\WisataFinder\data\processed\destinations_nasional.csv"
)

for kolom in ["City", "Location_Label"]:
    if kolom in df_periksa.columns:
        print(f"\n--- Isi kolom {kolom} ---")

        nilai = (
            df_periksa[kolom]
            .dropna()
            .astype(str)
            .unique()
        )

        for nama in sorted(nilai):
            if "semarang" in nama.casefold() or "bandung" in nama.casefold():
                print(repr(nama))


--- Isi kolom City ---
'Bandung Barat'
'Kabupaten Bandung'
'Kabupaten Semarang'
'Kota Bandung'
'Kota Semarang'

--- Isi kolom Location_Label ---
'Bandung Barat'
'Kabupaten Bandung'
'Kabupaten Semarang'
'Kota Bandung'
'Kota Semarang'


In [2]:
from pathlib import Path
import pandas as pd
import re

file_app_v7 = Path(r"D:\WisataFinder\app_nasional_v7.py")
file_dataset = Path(
    r"D:\WisataFinder\data\processed\destinations_nasional.csv"
)

kode = file_app_v7.read_text(encoding="utf-8")
df_cek = pd.read_csv(file_dataset)

city_column = (
    "Location_Label"
    if "Location_Label" in df_cek.columns
    else "City"
)

city_options = sorted(
    df_cek[city_column].dropna().astype(str).unique()
)

# Bagian deteksi kota yang akan diperbaiki
bagian_kota = r'''        # Deteksi kota/kabupaten dari kata kunci.
        detected_city = None
        detected_city_term = None

        # Prioritaskan nama lengkap, misalnya Kabupaten Semarang.
        for city_name in sorted(
            city_options,
            key=len,
            reverse=True,
        ):
            pola = rf"(?<!\w){re.escape(city_name.casefold())}(?!\w)"
            cocok = re.search(pola, query_lower)

            if cocok:
                detected_city = city_name
                detected_city_term = cocok.group(0)
                break

        # Jika nama lengkap tidak ditemukan, gunakan nama pendek.
        if detected_city is None:
            city_candidates = []

            for city_name in city_options:
                alias = re.sub(
                    r"^(?:kota administrasi|kota|kabupaten|kab\.?)\s+",
                    "",
                    city_name,
                    flags=re.IGNORECASE,
                ).strip()

                if not alias:
                    continue

                pola = rf"(?<!\w){re.escape(alias.casefold())}(?!\w)"
                cocok = re.search(pola, query_lower)

                if cocok:
                    nama_lower = city_name.casefold()

                    # Untuk nama pendek yang ambigu,
                    # prioritaskan Kota dibanding Kabupaten.
                    if nama_lower.startswith(("kota ", "kota administrasi ")):
                        prioritas = 0
                    elif nama_lower.startswith(("kabupaten ", "kab. ")):
                        prioritas = 2
                    else:
                        prioritas = 1

                    city_candidates.append(
                        (
                            prioritas,
                            -len(alias),
                            city_name,
                            cocok.group(0),
                        )
                    )

            if city_candidates:
                city_candidates.sort()

                (
                    _,
                    _,
                    detected_city,
                    detected_city_term,
                ) = city_candidates[0]

        # Nama lokasi dari kata kunci diprioritaskan
        # dibanding pilihan dropdown.
        effective_province = (
            detected_province
            if detected_province is not None
            else selected_province
        )

        effective_city = (
            detected_city
            if detected_city is not None
            else selected_city
        )

        location_filter_active = (
            effective_province != "Semua provinsi"
            or effective_city != "Semua kota"
        )

        # Hilangkan nama provinsi/kota dari teks pencarian.
        search_query = query

        lokasi_dihapus = []

        if detected_province_term:
            lokasi_dihapus.append(detected_province_term)

        if detected_city_term:
            lokasi_dihapus.append(detected_city_term)

        for lokasi in sorted(
            set(lokasi_dihapus),
            key=len,
            reverse=True,
        ):
            search_query = re.sub(
                rf"\b(?:di\s+)?{re.escape(lokasi)}\b",
                " ",
                search_query,
                flags=re.IGNORECASE,
            )

        search_query = re.sub(
            r"\s+",
            " ",
            search_query,
        ).strip(" ,.-")

        if not search_query:
            search_query = query

'''

# Ganti bagian deteksi kota
penanda_awal = "        # Deteksi kota/kabupaten dari kata kunci."
penanda_akhir = "        # Nama lokasi dari kata kunci diprioritaskan"

if kode.count(penanda_awal) != 1 or kode.count(penanda_akhir) != 1:
    raise ValueError("Bagian deteksi kota tidak ditemukan dengan tepat.")

awal = kode.index(penanda_awal)
akhir = kode.index(penanda_akhir, awal)

kode_baru = (
    kode[:awal]
    + bagian_kota
    + kode[akhir:]
)

# Ganti penggunaan nama kota lengkap menjadi nama yang terdeteksi
lama = '''        if detected_city:
            lokasi_dihapus.append(detected_city)'''

baru = '''        if detected_city_term:
            lokasi_dihapus.append(detected_city_term)'''

if kode_baru.count(lama) != 1:
    raise ValueError("Bagian penghapusan nama kota tidak ditemukan.")

kode_baru = kode_baru.replace(lama, baru, 1)

# Periksa sintaks
compile(kode_baru, str(file_app_v7), "exec")

# Simpan cadangan sebelum perubahan terbaru
backup = file_app_v7.with_name(
    "app_nasional_v7_sebelum_perbaikan_kota.py.bak"
)

if not backup.exists():
    backup.write_text(kode, encoding="utf-8")

# Simpan hasil perbaikan
file_app_v7.write_text(kode_baru, encoding="utf-8")

print("Perbaikan deteksi kota berhasil disimpan.")
print("Pemeriksaan sintaks: LULUS")
print("Cadangan:", backup)

Perbaikan deteksi kota berhasil disimpan.
Pemeriksaan sintaks: LULUS
Cadangan: D:\WisataFinder\app_nasional_v7_sebelum_perbaikan_kota.py.bak


In [3]:
def uji_deteksi_kota(query):
    query_lower = query.casefold()

    # Cocokkan nama lengkap terlebih dahulu
    for city_name in sorted(city_options, key=len, reverse=True):
        pola = rf"(?<!\w){re.escape(city_name.casefold())}(?!\w)"
        cocok = re.search(pola, query_lower)

        if cocok:
            return city_name

    # Jika nama lengkap tidak ditemukan, cocokkan alias
    kandidat = []

    for city_name in city_options:
        alias = re.sub(
            r"^(?:kota administrasi|kota|kabupaten|kab\.?)\s+",
            "",
            city_name,
            flags=re.IGNORECASE,
        ).strip()

        pola = rf"(?<!\w){re.escape(alias.casefold())}(?!\w)"
        cocok = re.search(pola, query_lower)

        if cocok:
            nama_lower = city_name.casefold()

            if nama_lower.startswith(("kota ", "kota administrasi ")):
                prioritas = 0
            elif nama_lower.startswith(("kabupaten ", "kab. ")):
                prioritas = 2
            else:
                prioritas = 1

            kandidat.append((prioritas, -len(alias), city_name))

    if kandidat:
        kandidat.sort()
        return kandidat[0][2]

    return None


daftar_uji = [
    "museum di Semarang",
    "museum di Bandung",
    "museum di Kabupaten Semarang",
    "museum di Bandung Barat",
]

for query_uji in daftar_uji:
    print(query_uji, "→", uji_deteksi_kota(query_uji))

museum di Semarang → Kota Semarang
museum di Bandung → Kota Bandung
museum di Kabupaten Semarang → Kabupaten Semarang
museum di Bandung Barat → Bandung Barat


In [1]:
from pathlib import Path
import pandas as pd

folder_proyek = Path(r"D:\WisataFinder")

folder_data = [
    folder_proyek / "data" / "raw",
    folder_proyek / "data" / "processed"
]

daftar_file = []

for folder in folder_data:
    for file in sorted(folder.glob("*.csv")):
        try:
            data = pd.read_csv(
                file,
                sep=None,
                engine="python",
                encoding="utf-8-sig"
            )

            daftar_file.append({
                "Nama File": file.name,
                "Folder": folder.name,
                "Ukuran (KB)": round(file.stat().st_size / 1024, 2),
                "Jumlah Baris": data.shape[0],
                "Jumlah Kolom": data.shape[1],
                "Nama Kolom": ", ".join(data.columns.astype(str))
            })

        except Exception as e:
            daftar_file.append({
                "Nama File": file.name,
                "Folder": folder.name,
                "Ukuran (KB)": round(file.stat().st_size / 1024, 2),
                "Jumlah Baris": "Gagal dibaca",
                "Jumlah Kolom": "-",
                "Nama Kolom": str(e)
            })

df_inventaris = pd.DataFrame(daftar_file)

display(df_inventaris)

print("Jumlah file CSV:", len(df_inventaris))

,Nama File,Folder,Ukuran (KB),Jumlah Baris,Jumlah Kolom,Nama Kolom
0,tourism_with_id.csv,raw,357.22,437,13,"Place_Id, Place_Name, Description, Category, C..."
1,wisata_indonesia_new.csv,raw,563.51,1025,11,"kategori, nama_wisata, latitude, longitude, al..."
2,audit_baseline.csv,processed,23.25,40,11,"Place_Name, Category, City, Price, Rating, sim..."
3,audit_relevansi_independen.csv,processed,22.64,38,10,"query, peringkat, Place_Name, Category, City, ..."
4,audit_relevansi_manual.csv,processed,3.10,38,9,"query, rank, Place_Name, Category, City, Price..."
5,audit_relevansi_semiotomatis.csv,processed,3.14,38,9,"query, rank, Place_Name, Category, City, Price..."
6,audit_sampel_awal.csv,processed,1.40,16,10,"query, rank, Place_Name, Category, City, Price..."
7,destinations_clean.csv,processed,686.99,437,14,"Place_Id, Place_Name, Description, Category, C..."
8,destinations_nasional.csv,processed,946.34,1025,23,"Place_Id, Place_Name, Description, Category, C..."
9,evaluasi_awal.csv,processed,0.32,8,5,"query, jumlah_hasil, k, jumlah_relevan, precision"


Jumlah file CSV: 26


In [1]:
from pathlib import Path
import pandas as pd

folder = Path(r"D:\WisataFinder")

def baca_csv(nama_file):
    return pd.read_csv(
        folder / nama_file,
        sep=None,
        engine="python",
        encoding="utf-8-sig"
    )

pasangan = [
    (
        "Dataset mentah dan audit",
        "data/raw/wisata_indonesia_new.csv",
        "data/processed/wisata_indonesia_audit.csv"
    ),
    (
        "Label manual dan terverifikasi",
        "data/processed/label_relevansi_manual_v7.csv",
        "data/processed/label_relevansi_manual_v7_terverifikasi.csv"
    ),
    (
        "Perbandingan algoritma",
        "data/processed/perbandingan_algoritma_v2_v7.csv",
        "data/processed/perbandingan_akhir_v2_v7.csv"
    ),
]

for judul, file_a, file_b in pasangan:
    df_a = baca_csv(file_a)
    df_b = baca_csv(file_b)

    kolom_sama = list(df_a.columns) == list(df_b.columns)
    isi_sama = (
        df_a.equals(df_b)
        if kolom_sama else False
    )

    print("\n" + "=" * 55)
    print(judul)
    print("=" * 55)
    print("File A:", file_a)
    print("File B:", file_b)
    print("Shape A:", df_a.shape)
    print("Shape B:", df_b.shape)
    print("Kolom sama:", kolom_sama)
    print("Isi sama persis:", isi_sama)

    if not kolom_sama:
        print("Kolom A:", list(df_a.columns))
        print("Kolom B:", list(df_b.columns))


Dataset mentah dan audit
File A: data/raw/wisata_indonesia_new.csv
File B: data/processed/wisata_indonesia_audit.csv
Shape A: (1025, 11)
Shape B: (1025, 11)
Kolom sama: True
Isi sama persis: True

Label manual dan terverifikasi
File A: data/processed/label_relevansi_manual_v7.csv
File B: data/processed/label_relevansi_manual_v7_terverifikasi.csv
Shape A: (56, 8)
Shape B: (56, 8)
Kolom sama: True
Isi sama persis: False

Perbandingan algoritma
File A: data/processed/perbandingan_algoritma_v2_v7.csv
File B: data/processed/perbandingan_akhir_v2_v7.csv
Shape A: (6, 8)
Shape B: (6, 7)
Kolom sama: False
Isi sama persis: False
Kolom A: ['Versi', 'Hasil_Bandung_Total', 'Hasil_Kota_Bandung', 'Hasil_Provinsi_Jawa_Barat', 'Hasil_Papua_Total', 'Hasil_Papua_Barat', 'Hasil_Atlantis', 'Hasil_Kepri']
Kolom B: ['Versi', 'Hasil_Bandung', 'Bandung_Sesuai', 'Hasil_Papua_Barat', 'Papua_Barat_Sesuai', 'Hasil_Atlantis', 'Hasil_Kepri']


In [3]:
from pathlib import Path
import pandas as pd

folder = Path(r"D:\WisataFinder\data\processed")

# Baca CSV dengan deteksi pemisah otomatis
label_awal = pd.read_csv(
    folder / "label_relevansi_manual_v7.csv",
    sep=None,
    engine="python",
    encoding="utf-8-sig"
)

label_final = pd.read_csv(
    folder / "label_relevansi_manual_v7_terverifikasi.csv",
    sep=None,
    engine="python",
    encoding="utf-8-sig"
)

print("=== LABEL AWAL ===")
print("Ukuran:", label_awal.shape)
print("Kolom:", list(label_awal.columns))

print("\n=== LABEL TERVERIFIKASI ===")
print("Ukuran:", label_final.shape)
print("Kolom:", list(label_final.columns))

print("\nApakah nama dan urutan kolom sama?")
print(list(label_awal.columns) == list(label_final.columns))

=== LABEL AWAL ===
Ukuran: (56, 8)
Kolom: ['Query', 'Posisi', 'Place_Name', 'Category', 'City', 'Province', 'Relevan', 'Catatan']

=== LABEL TERVERIFIKASI ===
Ukuran: (56, 8)
Kolom: ['Query', 'Posisi', 'Place_Name', 'Category', 'City', 'Province', 'Relevan', 'Catatan']

Apakah nama dan urutan kolom sama?
True


In [4]:
# Kolom untuk mengenali setiap data
kolom_kunci = ["Query", "Posisi", "Place_Name"]

# Salin data agar file asli tidak berubah
awal = label_awal.copy()
final = label_final.copy()

# Samakan format data untuk perbandingan
for kolom in awal.columns:
    awal[kolom] = awal[kolom].astype("string").fillna("<KOSONG>")
    final[kolom] = final[kolom].astype("string").fillna("<KOSONG>")

# Gunakan identitas data sebagai indeks
awal = awal.set_index(kolom_kunci).sort_index()
final = final.set_index(kolom_kunci).sort_index()

# Periksa apakah seluruh data memiliki identitas yang sama
print("Identitas data sama:", awal.index.equals(final.index))

if not awal.index.equals(final.index):
    print("\nAda perbedaan identitas atau urutan data.")
    print("Data yang hanya ada di file awal:")
    display(awal.loc[~awal.index.isin(final.index)])

    print("Data yang hanya ada di file terverifikasi:")
    display(final.loc[~final.index.isin(awal.index)])

else:
    # Tampilkan hanya nilai yang berubah
    perbedaan_label = awal.compare(
        final,
        result_names=("Sebelum", "Terverifikasi")
    )

    print("\nJumlah baris yang mengalami perubahan:",
          len(perbedaan_label))

    if perbedaan_label.empty:
        print("Tidak ditemukan perubahan isi data.")
    else:
        display(perbedaan_label)

Identitas data sama: True

Jumlah baris yang mengalami perubahan: 1


Relevan                \
                                                Sebelum Terverifikasi   
Query                        Posisi Place_Name                          
Wisata budaya di Jawa Tengah 3      Candi Cetho       0             1   

                                                  Catatan  \
                                                  Sebelum   
Query                        Posisi Place_Name              
Wisata budaya di Jawa Tengah 3      Candi Cetho  <KOSONG>   

                                                                                                    
                                                                                     Terverifikasi  
Query                        Posisi Place_Name                                                      
Wisata budaya di Jawa Tengah 3      Candi Cetho  Relevan. Lokasi sebenarnya Kabupaten Karangany...

In [5]:
from pathlib import Path
import pandas as pd

folder = Path(r"D:\WisataFinder\data\processed")

# Membaca kedua file dengan deteksi pemisah otomatis
perbandingan_lama = pd.read_csv(
    folder / "perbandingan_algoritma_v2_v7.csv",
    sep=None,
    engine="python",
    encoding="utf-8-sig"
)

perbandingan_final = pd.read_csv(
    folder / "perbandingan_akhir_v2_v7.csv",
    sep=None,
    engine="python",
    encoding="utf-8-sig"
)

print("=== PERBANDINGAN ALGORITMA V2-V7 (VERSI LAMA) ===")
display(perbandingan_lama)

print("=== PERBANDINGAN ALGORITMA V2-V7 (VERSI AKHIR) ===")
display(perbandingan_final)

print("\nJumlah baris versi lama:", len(perbandingan_lama))
print("Jumlah kolom versi lama:", len(perbandingan_lama.columns))

print("\nJumlah baris versi akhir:", len(perbandingan_final))
print("Jumlah kolom versi akhir:", len(perbandingan_final.columns))

=== PERBANDINGAN ALGORITMA V2-V7 (VERSI LAMA) ===


,Versi,Hasil_Bandung_Total,Hasil_Kota_Bandung,Hasil_Provinsi_Jawa_Barat,Hasil_Papua_Total,Hasil_Papua_Barat,Hasil_Atlantis,Hasil_Kepri
0,V2,10,3,4,10,0,10,10
1,V3,10,3,4,10,0,10,10
2,V4,10,3,4,10,0,10,10
3,V5,4,4,4,10,0,10,10
4,V6,4,4,4,3,3,10,0
5,V7,4,4,4,3,3,0,0


=== PERBANDINGAN ALGORITMA V2-V7 (VERSI AKHIR) ===


,Versi,Hasil_Bandung,Bandung_Sesuai,Hasil_Papua_Barat,Papua_Barat_Sesuai,Hasil_Atlantis,Hasil_Kepri
0,V2,10,3,10,0,10,10
1,V3,10,3,10,0,10,10
2,V4,10,3,10,0,10,10
3,V5,4,4,10,0,10,10
4,V6,4,4,3,3,10,0
5,V7,4,4,3,3,0,0



Jumlah baris versi lama: 6
Jumlah kolom versi lama: 8

Jumlah baris versi akhir: 6
Jumlah kolom versi akhir: 7


In [6]:
from pathlib import Path
import pandas as pd
import shutil

folder = Path(r"D:\WisataFinder\data\processed")

file_lama = folder / "perbandingan_algoritma_v2_v7.csv"
file_final = folder / "perbandingan_akhir_v2_v7.csv"

# Membaca data
lama = pd.read_csv(file_lama, encoding="utf-8-sig")
final_sebelumnya = pd.read_csv(
    file_final,
    encoding="utf-8-sig"
)

# Memastikan enam versi algoritma tersedia
assert lama["Versi"].tolist() == [
    "V2", "V3", "V4", "V5", "V6", "V7"
]

# Memastikan hasil yang beririsan konsisten
pasangan_kolom = {
    "Hasil_Bandung_Total": "Hasil_Bandung",
    "Hasil_Kota_Bandung": "Bandung_Sesuai",
    "Hasil_Papua_Total": "Hasil_Papua_Barat",
    "Hasil_Papua_Barat": "Papua_Barat_Sesuai",
    "Hasil_Atlantis": "Hasil_Atlantis",
    "Hasil_Kepri": "Hasil_Kepri",
}

for kolom_lama, kolom_final in pasangan_kolom.items():
    assert (
        lama[kolom_lama].tolist()
        == final_sebelumnya[kolom_final].tolist()
    ), f"Data berbeda pada {kolom_lama}"

print("Pemeriksaan kesesuaian: LULUS")

# Membuat tabel final dengan nama kolom yang lebih jelas
tabel_final = lama.rename(columns={
    "Hasil_Bandung_Total": "Hasil_Pencarian_Bandung",
    "Hasil_Kota_Bandung": "Sesuai_Kota_Bandung",
    "Hasil_Provinsi_Jawa_Barat": "Sesuai_Provinsi_Jawa_Barat",
    "Hasil_Papua_Total": "Hasil_Pencarian_Papua_Barat",
    "Hasil_Papua_Barat": "Sesuai_Provinsi_Papua_Barat",
})

# Buat cadangan sebelum mengganti tabel akhir
folder_backup = Path(r"D:\WisataFinder\backups")
folder_backup.mkdir(exist_ok=True)

file_backup = (
    folder_backup /
    "perbandingan_akhir_v2_v7_sebelumnya.csv"
)

if not file_backup.exists():
    shutil.copy2(file_final, file_backup)

# Simpan tabel final
tabel_final.to_csv(file_final, index=False)

print("\nTabel perbandingan final berhasil diperbarui.")
print("Jumlah baris:", len(tabel_final))
print("Jumlah kolom:", len(tabel_final.columns))

display(tabel_final)

Pemeriksaan kesesuaian: LULUS

Tabel perbandingan final berhasil diperbarui.
Jumlah baris: 6
Jumlah kolom: 8


,Versi,Hasil_Pencarian_Bandung,Sesuai_Kota_Bandung,Sesuai_Provinsi_Jawa_Barat,Hasil_Pencarian_Papua_Barat,Sesuai_Provinsi_Papua_Barat,Hasil_Atlantis,Hasil_Kepri
0,V2,10,3,4,10,0,10,10
1,V3,10,3,4,10,0,10,10
2,V4,10,3,4,10,0,10,10
3,V5,4,4,4,10,0,10,10
4,V6,4,4,4,3,3,10,0
5,V7,4,4,4,3,3,0,0


In [7]:
from pathlib import Path
import pandas as pd

folder = Path(r"D:\WisataFinder\data\processed")

daftar_evaluasi = [
    "evaluasi_precision_v7.csv",
    "evaluasi_provinsi_v7.csv",
    "ringkasan_audit_final.csv"
]

for nama_file in daftar_evaluasi:
    print("\n" + "=" * 65)
    print("FILE:", nama_file)
    print("=" * 65)

    data = pd.read_csv(
        folder / nama_file,
        sep=None,
        engine="python",
        encoding="utf-8-sig"
    )

    print("Ukuran data:", data.shape)
    display(data)


FILE: evaluasi_precision_v7.csv
Ukuran data: (8, 6)


,Query,Jumlah_Hasil,Jumlah_Relevan,Jumlah_Tidak_Relevan,Precision@10,Precision_Hasil_Ditemukan
0,Gunung dan wisata alam,10,10,0,1.0,1.0
1,Kebun binatang di Jakarta,1,1,0,0.1,1.0
2,Museum,10,10,0,1.0,1.0
3,Museum di Bandung,4,4,0,0.4,1.0
4,Museum di Semarang,1,1,0,0.1,1.0
5,Pantai di Bali,10,10,0,1.0,1.0
6,Tempat wisata di Bali,10,10,0,1.0,1.0
7,Wisata budaya di Jawa Tengah,10,10,0,1.0,1.0



FILE: evaluasi_provinsi_v7.csv
Ukuran data: (37, 4)


,Provinsi,Jumlah_Hasil,Hasil_Sesuai,Akurasi_Lokasi_Persen
0,Daerah Istimewa Yogyakarta,10,10,100.0
1,Kepulauan Bangka Belitung,10,10,100.0
2,Nusa Tenggara Barat,10,10,100.0
3,Nusa Tenggara Timur,10,10,100.0
4,Kalimantan Selatan,7,7,100.0
5,Sulawesi Tenggara,10,10,100.0
6,Kalimantan Tengah,10,10,100.0
7,Sulawesi Selatan,10,10,100.0
8,Kalimantan Barat,10,10,100.0
9,Sumatera Selatan,10,10,100.0



FILE: ringkasan_audit_final.csv
Ukuran data: (18, 2)


,Pemeriksaan,Hasil
0,Jumlah data,1025
1,Jumlah kolom awal,11
2,Nama wisata duplikat,0
3,Alamat kosong,0
4,Latitude kosong,0
5,Longitude kosong,0
6,Koordinat di luar rentang,0
7,Koordinat unik,861
8,Data dengan koordinat berulang,164
9,Kota kosong,18


In [8]:
from pathlib import Path

# Folder utama proyek
folder_proyek = Path(r"D:\WisataFinder")

# Membuat folder dokumentasi jika belum ada
folder_docs = folder_proyek / "docs"
folder_docs.mkdir(exist_ok=True)

# Isi dokumentasi evaluasi
konten_evaluasi = """# Evaluasi Sistem WisataFinder

## 1. Tujuan Evaluasi

Evaluasi dilakukan untuk mengetahui kemampuan algoritma pencarian WisataFinder
dalam menghasilkan destinasi yang relevan berdasarkan kata kunci, kategori,
dan lokasi yang diminta pengguna.

Evaluasi juga dilakukan untuk mengukur konsistensi penyaringan lokasi dan
memeriksa kualitas dataset nasional yang digunakan oleh sistem.

## 2. Evaluasi Relevansi Pencarian V7

Pengujian relevansi dilakukan menggunakan delapan kueri pencarian.
Sebanyak 56 hasil pencarian dinilai secara manual berdasarkan kesesuaiannya
dengan kebutuhan pencarian.

Hasil evaluasi:

| Kueri | Jumlah Hasil | Jumlah Relevan | Precision@10 |
|---|---:|---:|---:|
| Gunung dan wisata alam | 10 | 10 | 100% |
| Kebun binatang di Jakarta | 1 | 1 | 10% |
| Museum | 10 | 10 | 100% |
| Museum di Bandung | 4 | 4 | 40% |
| Museum di Semarang | 1 | 1 | 10% |
| Pantai di Bali | 10 | 10 | 100% |
| Tempat wisata di Bali | 10 | 10 | 100% |
| Wisata budaya di Jawa Tengah | 10 | 10 | 100% |
| **Rata-rata** | | | **70%** |

### Interpretasi

Rata-rata Precision@10 sebesar 70% diperoleh dari delapan kueri pengujian.

Perhitungan menggunakan pembagi tetap 10, termasuk ketika jumlah hasil
yang ditemukan kurang dari 10.

Selain itu, seluruh 56 hasil yang benar-benar dikembalikan oleh sistem
dinilai relevan berdasarkan penilaian manual pada sampel pengujian ini.
Dengan demikian, relevansi di antara hasil yang ditemukan adalah 100%.

Kedua nilai tersebut memiliki definisi pengukuran yang berbeda dan tidak
boleh dianggap sebagai ukuran akurasi umum seluruh sistem.

### Verifikasi label manual

Proses pemeriksaan ulang menemukan satu perubahan label pada destinasi
Candi Cetho untuk kueri "Wisata budaya di Jawa Tengah".

Label awal bernilai 0 dan setelah pemeriksaan ulang diubah menjadi 1
karena destinasi tersebut relevan terhadap kueri. Catatan juga diberikan
mengenai ketidaksesuaian metadata kota pada dataset.

File label yang digunakan sebagai acuan evaluasi akhir adalah:

`data/processed/label_relevansi_manual_v7_terverifikasi.csv`

## 3. Pengujian Penyaringan Provinsi

Pengujian dilakukan terhadap seluruh 37 nama provinsi yang tersedia
dalam dataset nasional.

Hasil pengujian menunjukkan bahwa seluruh provinsi menghasilkan
kecocokan metadata lokasi sebesar 100%.

Terdapat 299 kemunculan hasil dari seluruh pengujian provinsi,
dan seluruh hasil yang dikembalikan memiliki nilai Province yang
sesuai dengan provinsi yang diminta.

Hasil ini menunjukkan konsistensi penyaringan berdasarkan metadata
provinsi pada skenario pengujian.

Nilai tersebut bukan jaminan bahwa seluruh metadata geografis dalam
dataset telah benar secara faktual. Verifikasi lokasi destinasi
tetap diperlukan.

Rincian pengujian tersedia pada:

`data/processed/evaluasi_provinsi_v7.csv`

## 4. Audit Kualitas Dataset Nasional

Dataset nasional diperiksa menggunakan beberapa aspek kualitas data.

| Pemeriksaan | Hasil |
|---|---:|
| Jumlah destinasi | 1.025 |
| Nama wisata duplikat | 0 |
| Alamat kosong | 0 |
| Latitude kosong | 0 |
| Longitude kosong | 0 |
| Koordinat di luar rentang | 0 |
| Koordinat unik | 861 |
| Data dengan koordinat berulang | 164 |
| Kota kosong | 18 |
| Deskripsi kosong | 27 |
| Deskripsi tidak informatif | 177 |
| Deskripsi informatif | 821 |
| URL gambar kosong | 0 |
| URL gambar duplikat | 36 |
| URL gambar berformat HTTP/HTTPS | 1.025 |
| Provinsi dikoreksi berdasarkan alamat | 89 |
| Provinsi perlu verifikasi | 1 |

Pemeriksaan menunjukkan bahwa dataset masih memiliki beberapa keterbatasan,
khususnya deskripsi yang tidak informatif, informasi kota yang kosong,
koordinat yang berulang, serta satu data provinsi yang masih memerlukan
verifikasi.

Informasi yang tidak tersedia tidak dibuat secara sembarangan.
Nilai kosong tetap dipertahankan agar sistem tidak menyajikan
informasi yang tidak memiliki dasar.

Ringkasan audit tersedia pada:

`data/processed/ringkasan_audit_final.csv`

## 5. Keterbatasan Evaluasi

Evaluasi yang dilakukan masih memiliki beberapa keterbatasan:

1. Pengujian relevansi baru menggunakan delapan kueri.
2. Label relevansi dibuat melalui penilaian manual pada sampel hasil.
3. Hasil pengujian 37 provinsi mengukur kesesuaian metadata,
   bukan verifikasi geografis menyeluruh.
4. Kualitas informasi destinasi masih dipengaruhi kelengkapan dataset.
5. Kinerja algoritma pada variasi bahasa dan pertanyaan yang lebih luas
   masih memerlukan pengujian lanjutan.

## 6. Kesimpulan

Berdasarkan pengujian yang dilakukan, algoritma V7 menunjukkan kemampuan
menyaring lokasi secara konsisten pada skenario pengujian yang digunakan.

Rata-rata Precision@10 pada delapan kueri mencapai 70%, sedangkan seluruh
hasil yang ditemukan pada sampel tersebut dinilai relevan secara manual.

Pengujian provinsi menunjukkan kecocokan metadata sebesar 100% pada
37 provinsi.

Hasil ini menjadi dasar untuk pengembangan dan pengujian lebih lanjut
terhadap aplikasi WisataFinder berbasis dataset nasional.

---

**Artefak evaluasi:**
- `evaluasi_precision_v7.csv`
- `evaluasi_provinsi_v7.csv`
- `ringkasan_audit_final.csv`
- `label_relevansi_manual_v7_terverifikasi.csv`
- `perbandingan_akhir_v2_v7.csv`
"""

# Menyimpan dokumentasi
file_evaluasi = folder_docs / "EVALUASI.md"
file_evaluasi.write_text(konten_evaluasi, encoding="utf-8")

print("Dokumentasi berhasil dibuat!")
print("Lokasi:", file_evaluasi)
print("Ukuran:", round(file_evaluasi.stat().st_size / 1024, 2), "KB")

Dokumentasi berhasil dibuat!
Lokasi: D:\WisataFinder\docs\EVALUASI.md
Ukuran: 5.19 KB


In [9]:
from pathlib import Path

folder_proyek = Path(r"D:\WisataFinder")
folder_docs = folder_proyek / "docs"
folder_docs.mkdir(exist_ok=True)

konten_dataset = """# Dokumentasi Dataset WisataFinder

## 1. Gambaran Umum Dataset

WisataFinder menggunakan data destinasi wisata sebagai sumber informasi
utama dalam proses pencarian dan rekomendasi destinasi.

Dalam pengembangannya, proyek ini menggunakan dua dataset pada tahap
yang berbeda. Dataset awal digunakan pada pengembangan versi pertama,
sedangkan dataset nasional digunakan sebagai dataset utama untuk
pengembangan sistem berikutnya.

Kedua dataset tidak langsung digabungkan karena memiliki sumber,
struktur, dan cakupan data yang berbeda.

## 2. Dataset Awal

Dataset awal berasal dari file:

`data/raw/tourism_with_id.csv`

Karakteristik dataset awal:
- Jumlah data: 437 destinasi.
- Jumlah kolom: 13.
- Dataset hasil preprocessing: `data/processed/destinations_clean.csv`.
- Jumlah kolom setelah preprocessing: 14.

Dataset ini digunakan dalam pengembangan awal aplikasi WisataFinder
dan menjadi dasar pembuatan mesin pencari berbasis TF-IDF
dan Cosine Similarity.

Dataset awal tetap dipertahankan sebagai bagian dari riwayat
pengembangan proyek dan pembanding.

## 3. Dataset Nasional

Dataset nasional diperoleh dari Kaggle:

https://www.kaggle.com/datasets/sagitasantia/indonesia-tourism

File sumber:

`data/raw/wisata_indonesia_new.csv`

Karakteristik dataset sumber:
- Jumlah data: 1.025 destinasi.
- Jumlah kolom: 11.

Setelah proses pengolahan, dataset disimpan sebagai:

`data/processed/destinations_nasional.csv`

Dataset hasil pengolahan memiliki:
- Jumlah data: 1.025 destinasi.
- Jumlah kolom: 23.
- Cakupan label provinsi: 37 provinsi.

Dataset nasional digunakan sebagai dataset utama untuk
pengembangan aplikasi WisataFinder versi nasional.

Jumlah provinsi tersebut menunjukkan cakupan data yang tersedia,
bukan klaim bahwa seluruh destinasi wisata Indonesia telah tercakup.

## 4. Struktur Kolom Dataset Nasional

Dataset hasil pengolahan terdiri dari 23 kolom.

| Kolom | Keterangan |
|---|---|
| Place_Id | Identitas destinasi |
| Place_Name | Nama destinasi wisata |
| Description | Deskripsi destinasi |
| Category | Kategori wisata |
| City | Informasi kota atau kabupaten |
| Province | Informasi provinsi |
| Address | Alamat destinasi |
| Price | Informasi harga |
| Rating | Penilaian destinasi |
| Time_Minutes | Perkiraan durasi kunjungan |
| Lat | Koordinat lintang |
| Long | Koordinat bujur |
| Image_URL | Alamat gambar destinasi |
| Location_Label | Label lokasi untuk kebutuhan aplikasi |
| Location_Verification | Status pemeriksaan lokasi |
| Description_Quality | Informasi kualitas deskripsi |
| City_Verification | Informasi pemeriksaan kota |
| Province_Verification | Informasi pemeriksaan provinsi |
| Coordinate_Verification | Informasi pemeriksaan koordinat |
| Source | Informasi sumber data |
| Original_Name | Nama asli dari sumber |
| Original_Image_Path | Referensi lokasi gambar asli |
| search_text | Gabungan teks yang digunakan untuk pencarian |

## 5. Proses Pengolahan Dataset

Dataset nasional melalui proses pemeriksaan dan pengolahan sebelum
digunakan dalam sistem.

Tahapan yang dilakukan meliputi:

1. Memeriksa struktur, jumlah baris, dan jumlah kolom dataset.
2. Memeriksa nilai kosong pada setiap kolom.
3. Memeriksa nama destinasi yang berulang.
4. Memeriksa kelengkapan alamat dan koordinat.
5. Memeriksa kualitas deskripsi destinasi.
6. Memeriksa URL gambar.
7. Memeriksa konsistensi informasi kota dan provinsi.
8. Melakukan koreksi metadata provinsi berdasarkan informasi alamat
   pada data yang diperiksa.
9. Menambahkan informasi pendukung untuk pemeriksaan lokasi.
10. Menyusun kolom search_text untuk mendukung proses pencarian teks.

Nilai yang tidak tersedia tetap dipertahankan sebagai nilai kosong.
Informasi yang tidak memiliki sumber yang dapat dipastikan tidak
dibuat atau diperkirakan secara sembarangan.

## 6. Hasil Audit Kualitas Dataset Nasional

Berdasarkan hasil audit, diperoleh ringkasan berikut:

| Aspek Pemeriksaan | Hasil |
|---|---:|
| Jumlah destinasi | 1.025 |
| Jumlah kolom awal | 11 |
| Nama wisata duplikat | 0 |
| Alamat kosong | 0 |
| Latitude kosong | 0 |
| Longitude kosong | 0 |
| Koordinat di luar rentang | 0 |
| Koordinat unik | 861 |
| Data dengan koordinat berulang | 164 |
| Kota kosong | 18 |
| Deskripsi kosong | 27 |
| Deskripsi tidak informatif | 177 |
| Deskripsi informatif | 821 |
| URL gambar kosong | 0 |
| URL gambar duplikat | 36 |
| URL berformat HTTP/HTTPS | 1.025 |
| Provinsi dikoreksi berdasarkan alamat | 89 |
| Provinsi perlu verifikasi | 1 |

Hasil audit memperlihatkan bahwa dataset sudah memiliki informasi
dasar yang cukup untuk mendukung pengembangan mesin pencari.
Namun, masih terdapat beberapa keterbatasan yang perlu diperhatikan.

## 7. Keterbatasan Dataset

Beberapa keterbatasan yang ditemukan adalah:

1. Sebanyak 18 data belum memiliki informasi kota.
2. Sebanyak 27 data memiliki deskripsi kosong.
3. Sebanyak 177 deskripsi dinilai tidak informatif.
4. Terdapat 164 baris dalam kelompok koordinat yang berulang.
5. Terdapat 36 URL gambar yang berulang.
6. Satu data provinsi masih memerlukan verifikasi lebih lanjut.
7. Kolom Price, Rating, dan Time_Minutes belum memiliki nilai terisi
   dalam dataset nasional yang diperiksa.
8. Jumlah destinasi pada setiap provinsi tidak merata.

Keterbatasan tersebut perlu diperhatikan saat sistem menghasilkan
pencarian dan rekomendasi.

Hasil pencarian akan bergantung pada informasi yang benar-benar tersedia
dalam dataset.

## 8. Penggunaan Dataset dalam Aplikasi

Dataset nasional `destinations_nasional.csv` direncanakan menjadi
sumber data utama untuk:

- Mesin pencari destinasi WisataFinder.
- Penyaringan berdasarkan provinsi.
- Penyaringan berdasarkan kota atau kabupaten.
- Penyaringan berdasarkan kategori wisata.
- Pengembangan rekomendasi melalui WisataBot.

Penggunaan satu dataset utama bertujuan menjaga konsistensi informasi
antara fitur pencarian dan fitur rekomendasi.

Dataset awal tetap disimpan untuk mempertahankan riwayat pengembangan
dan memungkinkan perbandingan dengan versi sebelumnya.

## 9. Lisensi dan Sumber Data

Dataset nasional bersumber dari Kaggle.

Ketentuan lisensi dan izin redistribusi dataset perlu diperiksa
sebelum file mentah maupun hasil pengolahannya dipublikasikan
dalam repository GitHub publik.

Informasi atribusi dan lisensi akan dilengkapi berdasarkan
ketentuan resmi dari pemilik dataset.

## 10. Kesimpulan

Dataset nasional berisi 1.025 destinasi wisata dengan 23 kolom
setelah proses pengolahan.

Dataset ini menjadi dasar pengembangan WisataFinder versi nasional
karena memiliki cakupan wilayah lebih luas dibandingkan dataset awal.

Meskipun demikian, hasil audit menunjukkan masih terdapat data kosong,
deskripsi yang tidak informatif, koordinat berulang, dan metadata
yang perlu diverifikasi.

Oleh karena itu, pemeriksaan kualitas data tetap menjadi bagian
penting dalam pengembangan sistem agar hasil pencarian yang diberikan
kepada pengguna lebih konsisten dan dapat dipertanggungjawabkan.

---

Dokumen pendukung:
- `data/processed/destinations_nasional.csv`
- `data/processed/ringkasan_audit_final.csv`
- `notebooks/pengembangan_dan_evaluasi_algoritma_nasional.ipynb`
"""

file_dataset = folder_docs / "DATASET.md"
file_dataset.write_text(konten_dataset, encoding="utf-8")

print("Dokumentasi dataset berhasil dibuat!")
print("Lokasi:", file_dataset)
print("Ukuran:", round(file_dataset.stat().st_size / 1024, 2), "KB")

Dokumentasi dataset berhasil dibuat!
Lokasi: D:\WisataFinder\docs\DATASET.md
Ukuran: 7.16 KB


In [10]:
from pathlib import Path

folder_proyek = Path(r"D:\WisataFinder")
folder_docs = folder_proyek / "docs"
folder_docs.mkdir(exist_ok=True)

konten_metodologi = r"""# Metodologi Sistem WisataFinder

## 1. Pendekatan Pengembangan

WisataFinder dikembangkan sebagai mesin pencari destinasi wisata
berbasis teks menggunakan Python dan Streamlit.

Sistem menggunakan metode Information Retrieval (IR), yaitu
pendekatan untuk menemukan dokumen atau informasi yang paling
sesuai dengan kebutuhan pencarian pengguna.

Dalam proyek ini, setiap destinasi wisata diperlakukan sebagai
dokumen teks yang berisi informasi nama destinasi, kategori,
dan deskripsi.

Dataset nasional digunakan sebagai sumber data utama dalam
pengembangan sistem pencarian versi nasional.

## 2. Alur Kerja Sistem

Secara umum, proses pencarian WisataFinder terdiri dari beberapa tahap:

1. Memuat dataset destinasi wisata.
2. Melakukan persiapan teks untuk kebutuhan pencarian.
3. Mengubah teks destinasi menjadi representasi numerik menggunakan TF-IDF.
4. Menerima kata kunci dari pengguna.
5. Mengubah kata kunci menjadi representasi TF-IDF.
6. Menghitung kemiripan antara kata kunci dan dokumen destinasi
   menggunakan Cosine Similarity.
7. Menerapkan penyaringan kategori dan lokasi.
8. Mengurutkan hasil berdasarkan tingkat kemiripan.
9. Menampilkan destinasi yang sesuai kepada pengguna.

## 3. Persiapan Data Teks

Sistem menggunakan informasi teks destinasi untuk membangun indeks pencarian.

Informasi yang dimanfaatkan meliputi:

- Nama destinasi (Place_Name).
- Kategori wisata (Category).
- Deskripsi destinasi (Description).

Pada dataset nasional juga disiapkan kolom `search_text`
untuk mendukung proses pencarian.

Persiapan teks bertujuan agar informasi destinasi dapat
direpresentasikan dalam bentuk yang bisa diproses oleh algoritma.

Nilai teks yang kosong ditangani agar tidak menyebabkan
kesalahan pada proses pemuatan dan pengindeksan data.

## 4. Metode TF-IDF

TF-IDF merupakan singkatan dari Term Frequency-Inverse Document Frequency.

Metode ini digunakan untuk memberikan bobot pada kata berdasarkan
frekuensi kemunculannya dalam suatu dokumen dan tingkat kelangkaannya
dalam keseluruhan kumpulan dokumen.

### 4.1. Term Frequency (TF)

Term Frequency menunjukkan seberapa sering suatu kata muncul
dalam sebuah dokumen.

Secara sederhana:

TF(t,d) = jumlah kemunculan kata t dalam dokumen d

### 4.2. Inverse Document Frequency (IDF)

Inverse Document Frequency menunjukkan tingkat kepentingan suatu
kata berdasarkan jumlah dokumen yang memuat kata tersebut.

Secara umum:

IDF(t) = log(N / df(t))

Keterangan:

- N adalah jumlah seluruh dokumen.
- df(t) adalah jumlah dokumen yang mengandung kata t.

### 4.3. Bobot TF-IDF

Bobot TF-IDF diperoleh dari kombinasi TF dan IDF.

TF-IDF(t,d) = TF(t,d) × IDF(t)

Kata yang sering muncul pada dokumen tertentu tetapi relatif jarang
pada dokumen lain dapat memperoleh bobot yang lebih tinggi.

Dalam implementasi eksperimen nasional, digunakan
`TfidfVectorizer` dari library Scikit-learn.

Konfigurasi eksperimen dasar menggunakan pengaturan default
`TfidfVectorizer()` pada dataset yang telah memiliki kolom `search_text`.

## 5. Metode Cosine Similarity

Cosine Similarity digunakan untuk menghitung kemiripan antara
vektor kata kunci pencarian dan vektor dokumen destinasi.

Rumus Cosine Similarity:

similarity(A,B) = (A · B) / (||A|| × ||B||)

Keterangan:

- A adalah vektor kata kunci pengguna.
- B adalah vektor dokumen destinasi.
- A · B adalah hasil perkalian titik kedua vektor.
- ||A|| dan ||B|| adalah panjang masing-masing vektor.

Nilai kemiripan yang lebih tinggi menunjukkan bahwa teks dokumen
memiliki kemiripan yang lebih besar terhadap kata kunci pencarian.

Dalam implementasi, penghitungan dilakukan menggunakan
`cosine_similarity` dari Scikit-learn.

Hasil kemudian diurutkan berdasarkan skor kemiripan.

## 6. Pengembangan Algoritma V2 sampai V7

Pengembangan algoritma dilakukan secara bertahap.
Setiap versi memperbaiki keterbatasan yang ditemukan pada versi
sebelumnya.

### 6.1. V2 – Penambahan Bobot Kategori dan Lokasi

Pada V2 ditambahkan skor tambahan ketika kueri cocok dengan
kategori atau lokasi destinasi.

Tujuannya adalah meningkatkan prioritas destinasi yang memiliki
kesesuaian dengan maksud pencarian pengguna.

Pada eksperimen ini, kecocokan kategori dan lokasi masing-masing
diberikan tambahan skor sebesar 0,30.

Versi ini masih menggunakan pendekatan penambahan skor sehingga
hasil yang tidak sesuai lokasi masih dapat muncul.

### 6.2. V3 – Penyaringan Kategori dan Lokasi

V3 mengembangkan pendekatan penyaringan berdasarkan kategori
dan lokasi.

Sistem mulai menggunakan kondisi kecocokan untuk membatasi
kandidat destinasi.

Pada versi awal, ketika tidak ditemukan hasil yang cocok,
sistem masih dapat kembali menampilkan seluruh dataset.

Perilaku tersebut menjadi salah satu kelemahan yang perlu diperbaiki.

### 6.3. V4 – Pencocokan Nama Destinasi

V4 memperluas pencocokan kata kunci dengan mempertimbangkan
nama destinasi.

Pendekatan ini membantu menemukan tempat berdasarkan identitas
namanya meskipun kategori dataset tidak selalu menggunakan
istilah yang sama dengan kueri pengguna.

Contohnya adalah pencarian kebun binatang di Jakarta,
yang dapat menemukan Kebun Binatang Ragunan meskipun
kategori datanya tidak secara eksplisit bernama kebun binatang.

Pada versi ini, mekanisme fallback dari versi sebelumnya
masih menjadi keterbatasan.

### 6.4. V5 – Deteksi Kota Otomatis

V5 mengembangkan kemampuan sistem dalam mendeteksi kota
yang disebutkan langsung oleh pengguna.

Dengan demikian, sistem tidak selalu bergantung pada pilihan
kota melalui filter.

Pengujian menunjukkan peningkatan pada kueri museum di Bandung.
Versi ini menghasilkan empat hasil yang sesuai dengan Kota Bandung.

Namun, pencarian masih memiliki keterbatasan pada cakupan
pengenalan lokasi dan penanganan wilayah yang belum dikenali.

### 6.5. V6 – Deteksi Provinsi dan Kota

V6 memperluas deteksi lokasi dengan menggunakan informasi
kota dan provinsi yang tersedia pada dataset.

Pengenalan lokasi dilakukan berdasarkan daftar lokasi
yang berasal dari dataset, termasuk nama provinsi
yang telah disusun untuk kebutuhan pencarian.

Peningkatan ini membantu sistem membedakan pencarian
berdasarkan kota dan provinsi.

Pada pengujian Papua Barat, V6 menghasilkan tiga destinasi
yang seluruhnya sesuai dengan metadata provinsi.

Namun, sistem masih memerlukan validasi tambahan untuk
menangani lokasi yang tidak dikenal.

### 6.6. V7 – Validasi Lokasi Pencarian

V7 menambahkan lapisan validasi terhadap lokasi yang
disebutkan oleh pengguna.

Tujuannya adalah mencegah sistem menampilkan destinasi
dari lokasi lain ketika lokasi yang diminta tidak dikenali.

Contohnya:

- `museum di Semarang` harus mengarah pada lokasi Semarang.
- `pantai di Bali` harus mengarah pada provinsi Bali.
- `tempat wisata di Atlantis` harus menghasilkan pencarian kosong
  karena Atlantis tidak tersedia sebagai lokasi yang dikenali.
- `tempat wisata di Kepulauan Riau` tidak menghasilkan destinasi
  apabila lokasi tersebut belum mempunyai data pada dataset.

V7 juga dikembangkan agar tidak menggunakan fallback ke seluruh
dataset ketika lokasi yang disebutkan tidak cocok.

Dengan pendekatan tersebut, sistem lebih menjaga kesesuaian
antara maksud pencarian dan lokasi hasil.

## 7. Penggunaan Alias Kategori dan Lokasi

Pengguna dapat menggunakan istilah yang berbeda untuk
menyatakan kebutuhan pencarian yang sama.

Oleh karena itu, pada eksperimen tertentu digunakan
pemetaan alias kategori dan lokasi.

Contoh alias kategori:

| Istilah pengguna | Kelompok pencarian |
|---|---|
| Laut | Pantai atau bahari |
| Pendakian | Gunung |
| Zoo | Kebun binatang |
| Curug | Air terjun |
| Mal | Mall atau pusat perbelanjaan |

Contoh alias lokasi:

| Istilah pengguna | Lokasi yang dikenali |
|---|---|
| Jateng | Jawa Tengah |
| Jabar | Jawa Barat |
| Jatim | Jawa Timur |
| Jogja | Daerah Istimewa Yogyakarta |
| NTB | Nusa Tenggara Barat |

Pemetaan alias membantu menjembatani perbedaan istilah
antara kata kunci pengguna dan informasi yang tersimpan
dalam dataset.

Penggunaan alias disesuaikan dengan versi eksperimen
yang sedang diuji.

## 8. Penyaringan Hasil

Setelah skor kemiripan diperoleh, sistem menerapkan
penyaringan tambahan.

Penyaringan dapat menggunakan:

- Provinsi.
- Kota atau kabupaten.
- Kategori wisata.
- Kesesuaian identitas destinasi untuk kueri tertentu.
- Status verifikasi lokasi.

Filter lokasi dan kategori bertujuan membatasi hasil
agar sesuai dengan kebutuhan pengguna.

Pencarian tanpa lokasi tetap dapat menampilkan destinasi
dari berbagai wilayah selama hasilnya sesuai dengan
kata kunci dan filter yang digunakan.

## 9. Perbandingan dan Evaluasi Algoritma

Pengembangan V2 sampai V7 diuji dengan beberapa skenario,
antara lain:

- Museum di Bandung.
- Wisata di Papua Barat.
- Lokasi Atlantis.
- Kepulauan Riau.

Pengujian digunakan untuk melihat kemampuan setiap versi
dalam menemukan hasil yang sesuai dan menolak lokasi
yang tidak tersedia.

Selain itu, dilakukan penilaian relevansi secara manual
terhadap delapan kueri untuk mengevaluasi hasil pencarian V7.

Hasil lengkap tersedia pada:

- `data/processed/perbandingan_akhir_v2_v7.csv`
- `data/processed/evaluasi_precision_v7.csv`
- `data/processed/evaluasi_provinsi_v7.csv`

## 10. Batasan Metodologi

Metode TF-IDF dan Cosine Similarity memiliki beberapa keterbatasan.

1. Sistem terutama mengandalkan kemiripan kata dan teks.
2. Sinonim dan istilah yang tidak tercakup dalam alias
   dapat menghasilkan pencarian yang kurang tepat.
3. Kualitas hasil pencarian dipengaruhi kelengkapan
   dan kualitas deskripsi destinasi.
4. Metadata lokasi yang salah dapat menyebabkan hasil
   yang secara geografis tidak tepat.
5. Eksperimen dan evaluasi yang dilakukan belum mencakup
   seluruh kemungkinan kueri pengguna.
6. Pemetaan alias masih perlu diperluas berdasarkan
   variasi bahasa dan kebutuhan pengguna.

Oleh karena itu, pengembangan berikutnya dapat mencakup
perbaikan kualitas data, perluasan variasi kueri,
serta pengujian yang lebih luas.

## 11. Kesimpulan

WisataFinder menggunakan TF-IDF dan Cosine Similarity
sebagai dasar pencarian destinasi wisata.

Pengembangan V2 sampai V7 dilakukan untuk meningkatkan
kemampuan sistem dalam memahami kategori, mengenali lokasi,
menyaring hasil, serta menangani lokasi yang tidak dikenal.

Berdasarkan skenario pengujian yang dilakukan, V7 menunjukkan
perbaikan dalam penyaringan lokasi dibandingkan versi awal.

Algoritma ini menjadi dasar pengembangan aplikasi nasional
WisataFinder dan rencana fitur rekomendasi melalui WisataBot.

---

Dokumen ini menjelaskan metodologi eksperimen algoritma.
Implementasi dan hasil evaluasi aktual tetap perlu dilihat
pada notebook dan file aplikasi yang sesuai.
"""

file_metodologi = folder_docs / "METODOLOGI.md"
file_metodologi.write_text(
    konten_metodologi,
    encoding="utf-8"
)

print("Dokumentasi metodologi berhasil dibuat!")
print("Lokasi:", file_metodologi)
print(
    "Ukuran:",
    round(file_metodologi.stat().st_size / 1024, 2),
    "KB"
)

Dokumentasi metodologi berhasil dibuat!
Lokasi: D:\WisataFinder\docs\METODOLOGI.md
Ukuran: 10.74 KB


In [11]:
from pathlib import Path
import pandas as pd

folder = Path(r"D:\WisataFinder")

# Membaca hasil evaluasi provinsi
df_provinsi = pd.read_csv(
    folder / "data/processed/evaluasi_provinsi_v7.csv"
)

# Menghitung total hasil pengujian
total_hasil = pd.to_numeric(
    df_provinsi["Jumlah_Hasil"]
).sum()

total_sesuai = pd.to_numeric(
    df_provinsi["Hasil_Sesuai"]
).sum()

# Memeriksa jumlah provinsi
assert len(df_provinsi) == 37

# Memeriksa kesesuaian hasil
assert total_hasil == total_sesuai

# Memperbarui dokumentasi evaluasi
file_evaluasi = folder / "docs/EVALUASI.md"
konten = file_evaluasi.read_text(encoding="utf-8")

if "299 kemunculan hasil" not in konten:
    raise ValueError("Angka lama tidak ditemukan. Periksa dokumen.")

konten = konten.replace(
    "299 kemunculan hasil",
    f"{total_hasil} kemunculan hasil"
)

file_evaluasi.write_text(konten, encoding="utf-8")

print("Dokumentasi berhasil diperbarui.")
print("Jumlah provinsi:", len(df_provinsi))
print("Total hasil pengujian:", total_hasil)
print("Total hasil sesuai:", total_sesuai)

Dokumentasi berhasil diperbarui.
Jumlah provinsi: 37
Total hasil pengujian: 289
Total hasil sesuai: 289


In [12]:
from pathlib import Path

# Folder proyek
folder_proyek = Path(r"D:\WisataFinder")
folder_docs = folder_proyek / "docs"
folder_docs.mkdir(exist_ok=True)

konten_pengembangan = """# Riwayat Pengembangan WisataFinder

## 1. Pendahuluan

WisataFinder merupakan aplikasi mesin pencari destinasi wisata
berbasis web yang dikembangkan menggunakan Python dan Streamlit.

Proyek ini dikembangkan secara bertahap, mulai dari aplikasi
pencarian dengan dataset awal hingga pengembangan sistem pencarian
nasional menggunakan TF-IDF dan Cosine Similarity.

Pengembangan dilakukan dengan memperhatikan kualitas data,
kemampuan pencarian, kesesuaian lokasi, evaluasi hasil,
dan kemudahan penggunaan aplikasi.

## 2. Tahap Pengembangan Awal

Pada tahap awal, WisataFinder menggunakan dataset pariwisata
dengan 437 destinasi.

Aplikasi dikembangkan menggunakan:
- Python.
- Streamlit.
- Pandas.
- Scikit-learn.
- TF-IDF.
- Cosine Similarity.

Fitur awal aplikasi meliputi:
- Pencarian destinasi berdasarkan kata kunci.
- Filter lokasi.
- Filter kategori.
- Penampilan informasi destinasi.
- Pengurutan hasil berdasarkan kemiripan teks.

Pada tahap ini, tujuan utama adalah membangun produk awal
yang dapat digunakan untuk mencari informasi destinasi wisata.

## 3. Pengembangan Dataset Nasional

Untuk memperluas cakupan pencarian, dilakukan pengembangan
menggunakan dataset nasional dari Kaggle.

Dataset sumber memiliki 1.025 destinasi dengan 11 kolom.

Setelah proses pengolahan, disusun dataset nasional
dengan 1.025 destinasi dan 23 kolom.

Dataset nasional mencakup 37 provinsi berdasarkan
label provinsi yang tersedia.

Pengembangan ini bertujuan agar pengguna dapat mencari
destinasi wisata dari lebih banyak wilayah di Indonesia.

Dataset lama tetap dipertahankan sebagai bagian dari
riwayat pengembangan. Kedua dataset tidak langsung
digabungkan karena memiliki sumber dan struktur yang berbeda.

## 4. Audit dan Pengolahan Data

Setelah dataset nasional disiapkan, dilakukan pemeriksaan
kualitas data.

Pemeriksaan mencakup:
- Nilai kosong.
- Data duplikat.
- Kelengkapan alamat.
- Koordinat geografis.
- Kualitas deskripsi.
- URL gambar.
- Konsistensi kota dan provinsi.

Hasil audit menunjukkan beberapa data memerlukan
perhatian lebih lanjut, khususnya deskripsi yang tidak
informatif, kota yang belum tersedia, koordinat yang
berulang, dan metadata lokasi yang perlu diverifikasi.

Informasi yang tidak tersedia tidak dibuat secara
sembarangan. Nilai kosong tetap dipertahankan agar
sistem tidak menyajikan informasi yang tidak memiliki dasar.

## 5. Eksperimen Pengembangan Algoritma

Eksperimen dilakukan secara bertahap melalui tujuh versi
pengembangan, yaitu V2 sampai V7, dengan V2 sebagai
pengembangan setelah baseline pencarian awal.

### V2 – Penambahan skor kecocokan

Menambahkan skor tambahan untuk kecocokan kategori
dan lokasi agar destinasi yang sesuai memperoleh
prioritas lebih tinggi.

### V3 – Penyaringan kategori dan lokasi

Mengembangkan penyaringan berdasarkan kategori
dan lokasi destinasi.

### V4 – Pencocokan nama destinasi

Memperluas pencarian dengan mempertimbangkan
nama destinasi sebagai bagian dari pencocokan.

### V5 – Deteksi kota otomatis

Mengembangkan kemampuan sistem untuk mengenali
nama kota yang disebutkan dalam kata kunci pengguna.

### V6 – Deteksi provinsi dan kota

Memperluas pengenalan lokasi menggunakan data
provinsi dan kota yang tersedia pada dataset nasional.

### V7 – Validasi lokasi pencarian

Menambahkan validasi lokasi agar sistem tidak
mengembalikan destinasi dari wilayah lain ketika
lokasi yang diminta tidak dikenali atau tidak tersedia.

Eksperimen tersebut didokumentasikan dalam notebook:

`notebooks/pengembangan_dan_evaluasi_algoritma_nasional.ipynb`

## 6. Evaluasi Algoritma

Pengujian dilakukan untuk membandingkan kemampuan
versi algoritma V2 sampai V7.

Skenario pengujian mencakup:
- Pencarian museum di Bandung.
- Pencarian wisata di Papua Barat.
- Pencarian lokasi yang tidak dikenal seperti Atlantis.
- Pencarian Kepulauan Riau.

Selain perbandingan algoritma, dilakukan evaluasi
relevansi manual pada delapan kueri dengan total
56 hasil pencarian yang dinilai.

Hasil evaluasi V7:
- Rata-rata Precision@10: 70%.
- Hasil yang ditemukan dan dinilai relevan: 56 dari 56.
- Pengujian penyaringan provinsi: 37 provinsi.
- Total hasil pada pengujian provinsi: 289.
- Hasil yang sesuai dengan metadata provinsi: 289.

Angka tersebut berlaku pada skenario dan data pengujian
yang telah dilakukan, bukan merupakan klaim akurasi umum
untuk seluruh pertanyaan pengguna.

Hasil lengkap disimpan dalam file evaluasi CSV
dan dijelaskan pada `docs/EVALUASI.md`.

## 7. Pengembangan Aplikasi Nasional

Agar aplikasi awal tetap aman, dibuat salinan aplikasi
untuk pengembangan dataset nasional.

File aplikasi:
- `app.py`: aplikasi awal.
- `app_nasional_v7.py`: salinan aplikasi untuk pengembangan
  menggunakan dataset nasional.

Aplikasi nasional diarahkan untuk menggunakan:

`data/processed/destinations_nasional.csv`

Pada salinan aplikasi dilakukan pengembangan filter:
- Provinsi.
- Kota atau kabupaten.
- Kategori wisata.

Selain itu, dilakukan perbaikan deteksi nama lokasi
agar sistem dapat mengenali nama pendek seperti
Semarang dan Bandung serta nama administratif lengkap
seperti Kabupaten Semarang dan Kota Bandung.

Pemeriksaan sintaks Python terhadap perubahan tersebut
telah berhasil. Aplikasi Streamlit juga telah berhasil
dijalankan pada tahap pengujian sebelumnya.

Pengujian ulang terhadap seluruh perubahan terakhir
masih perlu diselesaikan sebelum aplikasi nasional
ditetapkan sebagai versi utama.

## 8. Pengelolaan Repository GitHub

Proyek menggunakan repository:

https://github.com/siskamuthoharoh/WisataFinder

Pengembangan nasional dilanjutkan pada repository
yang sama untuk mempertahankan riwayat proyek.

Beberapa commit pengembangan sebelumnya meliputi:
- `1e9d522`: preprocessing data wisata.
- `874edad`: dokumentasi eksplorasi dataset.
- `c7e33ef`: penambahan dataset mentah.
- `ef31a72`: dokumentasi hasil evaluasi query expansion.
- `a0071ff`: implementasi aplikasi pencarian WisataFinder.

Pengembangan dataset nasional selanjutnya akan disimpan
melalui commit terpisah sesuai kelompok pekerjaan,
misalnya dataset, dokumentasi, aplikasi, dan evaluasi.

File sementara, file cadangan, serta hasil eksperimen
yang tidak digunakan sebagai artefak akhir tidak
direncanakan untuk dimasukkan ke repository publik.

## 9. Rencana Pengembangan Berikutnya

Tahap pengembangan berikutnya meliputi:

1. Menyelesaikan pengujian filter provinsi, kota,
   kabupaten, dan kategori.
2. Memastikan algoritma V7 terintegrasi dengan benar
   pada aplikasi nasional.
3. Melakukan pengujian fungsional pada berbagai
   variasi kata kunci.
4. Menyelesaikan seleksi file yang akan dimasukkan
   ke repository GitHub.
5. Memeriksa lisensi dataset sebelum publikasi.
6. Memperbarui README dan dokumentasi proyek.
7. Melakukan commit dan push pengembangan nasional
   secara bertahap.
8. Mengembangkan fitur chatbot WisataBot.

## 10. Rencana Pengembangan WisataBot

WisataBot direncanakan sebagai asisten pencarian
dan rekomendasi destinasi melalui percakapan.

Versi awal direncanakan menggunakan dataset nasional
yang sama dengan WisataFinder agar hasil rekomendasi
konsisten dengan informasi yang tersedia dalam aplikasi.

Fitur yang direncanakan:
- Pencarian destinasi melalui percakapan.
- Rekomendasi berdasarkan kategori wisata.
- Pencarian berdasarkan provinsi dan kota.
- Penyajian informasi destinasi dari dataset.
- Alternatif rekomendasi ketika hasil tidak ditemukan.

Fitur ini masih dalam tahap perencanaan
dan belum menjadi bagian dari aplikasi aktif.

## 11. Kesimpulan

Pengembangan WisataFinder telah berkembang dari aplikasi
pencarian awal dengan 437 destinasi menjadi pengembangan
mesin pencari nasional dengan dataset berisi 1.025 destinasi.

Proses pengembangan mencakup pengolahan dataset,
audit kualitas data, eksperimen algoritma V2 sampai V7,
evaluasi relevansi, pengujian lokasi, serta pengembangan
fitur aplikasi.

Fokus pengembangan saat ini adalah menyelesaikan
integrasi dan pengujian aplikasi nasional serta
menata dokumentasi dan repository GitHub.

WisataBot direncanakan sebagai fitur tambahan
untuk memperluas kemampuan WisataFinder dalam
memberikan rekomendasi destinasi melalui percakapan.
"""

file_pengembangan = folder_docs / "PENGEMBANGAN.md"

file_pengembangan.write_text(
    konten_pengembangan,
    encoding="utf-8"
)

print("Dokumentasi pengembangan berhasil dibuat!")
print("Lokasi:", file_pengembangan)
print(
    "Ukuran:",
    round(file_pengembangan.stat().st_size / 1024, 2),
    "KB"
)

Dokumentasi pengembangan berhasil dibuat!
Lokasi: D:\WisataFinder\docs\PENGEMBANGAN.md
Ukuran: 8.2 KB


In [13]:
from pathlib import Path

folder_proyek = Path(r"D:\WisataFinder")
file_readme = folder_proyek / "README.md"

if file_readme.exists():
    isi_readme = file_readme.read_text(encoding="utf-8")

    print("README.md ditemukan.")
    print("Ukuran:", len(isi_readme), "karakter")
    print("\n" + "=" * 60)
    print("ISI README.md SAAT INI")
    print("=" * 60)

    print(isi_readme)

else:
    print("README.md belum ditemukan.")

README.md ditemukan.
Ukuran: 16206 karakter

ISI README.md SAAT INI
# WisataFinder — Mesin Pencari Destinasi Wisata

**Sistem Temu Kembali Informasi (STKI) menggunakan TF-IDF dan Cosine Similarity**

WisataFinder adalah aplikasi pencarian destinasi wisata di Indonesia yang dikembangkan sebagai proyek akademik mata kuliah Sistem Temu Kembali Informasi (STKI). Aplikasi ini membantu pengguna menemukan destinasi wisata berdasarkan kebutuhan pencarian melalui kata kunci, filter lokasi, dan kategori.

Sistem menerapkan metode **TF-IDF (Term Frequency–Inverse Document Frequency)** untuk merepresentasikan teks destinasi dan **Cosine Similarity** untuk menghitung tingkat kemiripan antara query pengguna dan informasi destinasi. Hasil pencarian kemudian ditampilkan berdasarkan skor kemiripan, dengan penyesuaian tertentu seperti pengurutan harga untuk query yang secara eksplisit meminta wisata murah.

---

## Daftar Isi

1. [Latar Belakang](#1-latar-belakang)
2. [Tujuan Proyek](#2-tujuan-proyek)
3

In [17]:
from pathlib import Path
import shutil

folder = Path(r"D:\WisataFinder")
file_readme = folder / "README.md"

if not file_readme.exists():
    raise FileNotFoundError("README.md tidak ditemukan.")

folder_backup = folder / "backups"
folder_backup.mkdir(exist_ok=True)

file_backup = folder_backup / "README_lama.md"

if not file_backup.exists():
    shutil.copy2(file_readme, file_backup)

# Menyiapkan bagian-bagian README baru
readme_parts = []

print("Persiapan berhasil.")
print("Cadangan README:", file_backup)

Persiapan berhasil.
Cadangan README: D:\WisataFinder\backups\README_lama.md


In [18]:
readme_parts.append(r"""
# WisataFinder
### Mesin Pencari Destinasi Wisata Indonesia

WisataFinder merupakan aplikasi mesin pencari destinasi wisata berbasis web yang dikembangkan sebagai proyek akademik mata kuliah **Sistem Temu Kembali Informasi (STKI)**.

Sistem menggunakan metode **TF-IDF (Term Frequency-Inverse Document Frequency)** dan **Cosine Similarity** untuk menemukan destinasi berdasarkan kemiripan teks antara kata kunci pengguna dan informasi destinasi.

Pengembangan dilakukan dari dataset awal sebanyak 437 destinasi menuju dataset nasional sebanyak 1.025 destinasi dari 37 provinsi.

> **Status:** Pengembangan aplikasi nasional dan validasi algoritma V7 masih berlangsung.

---

## Daftar Isi

1. [Latar Belakang](#1-latar-belakang)
2. [Tujuan Proyek](#2-tujuan-proyek)
3. [Fitur Aplikasi](#3-fitur-aplikasi)
4. [Dataset](#4-dataset)
5. [Metode Pencarian](#5-metode-pencarian)
6. [Pengembangan Algoritma](#6-pengembangan-algoritma)
7. [Hasil Evaluasi](#7-hasil-evaluasi)
8. [Teknologi](#8-teknologi)
9. [Struktur Repository](#9-struktur-repository)
10. [Instalasi dan Penggunaan](#10-instalasi-dan-penggunaan)
11. [Dokumentasi](#11-dokumentasi)
12. [Keterbatasan dan Pengembangan](#12-keterbatasan-dan-pengembangan)

---

## 1. Latar Belakang

Informasi destinasi wisata di Indonesia tersedia dalam berbagai bentuk, seperti nama tempat, deskripsi, kategori, dan lokasi. Pengguna sering memiliki kebutuhan pencarian spesifik, misalnya mencari pantai di Bali atau museum di Semarang.

WisataFinder dikembangkan untuk menerapkan konsep temu kembali informasi dalam pencarian destinasi wisata. Sistem membandingkan kata kunci pengguna dengan informasi tekstual destinasi menggunakan TF-IDF dan Cosine Similarity.

## 2. Tujuan Proyek

- Menerapkan konsep Sistem Temu Kembali Informasi pada bidang pariwisata.
- Menyiapkan dan memeriksa dataset destinasi wisata.
- Menerapkan TF-IDF untuk merepresentasikan teks.
- Menggunakan Cosine Similarity untuk menghitung kemiripan query.
- Mengembangkan pencarian berdasarkan kategori dan lokasi.
- Menguji relevansi hasil pencarian.
- Mengembangkan aplikasi pencarian wisata berbasis web.

## 3. Fitur Aplikasi

Fitur yang tersedia pada aplikasi awal dan sedang dikembangkan pada versi nasional:

- Pencarian destinasi berdasarkan kata kunci.
- Pemeringkatan hasil menggunakan skor kemiripan.
- Filter kategori wisata.
- Filter provinsi dan kota/kabupaten pada aplikasi nasional.
- Deteksi lokasi yang disebutkan dalam query.
- Tampilan informasi destinasi.
- Penyesuaian hasil pencarian berdasarkan kebutuhan tertentu.

Integrasi penuh algoritma V7 pada aplikasi nasional masih memerlukan pengujian lanjutan.

## 4. Dataset

WisataFinder menggunakan dua dataset pada tahap pengembangan yang berbeda.

### 4.1 Dataset awal

| Informasi | Keterangan |
|---|---|
| Dataset mentah | `data/raw/tourism_with_id.csv` |
| Dataset hasil pengolahan | `data/processed/destinations_clean.csv` |
| Jumlah destinasi | 437 |
| Penggunaan | Pengembangan aplikasi awal |

### 4.2 Dataset nasional

Sumber dataset: [Indonesia Tourism Dataset – Kaggle](https://www.kaggle.com/datasets/sagitasantia/indonesia-tourism).

| Informasi | Keterangan |
|---|---|
| Dataset sumber | `data/raw/wisata_indonesia_new.csv` |
| Dataset hasil pengolahan | `data/processed/destinations_nasional.csv` |
| Jumlah destinasi | 1.025 |
| Kolom sumber | 11 |
| Kolom hasil pengolahan | 23 |
| Cakupan label provinsi | 37 |

Dataset nasional dipilih sebagai dataset utama untuk pengembangan selanjutnya. Dataset lama tetap dipertahankan sebagai bagian dari riwayat proyek dan pembanding. Kedua dataset tidak langsung digabungkan.

### 4.3 Ringkasan audit dataset nasional

| Pemeriksaan | Hasil |
|---|---:|
| Jumlah destinasi | 1.025 |
| Nama wisata duplikat | 0 |
| Alamat kosong | 0 |
| Koordinat kosong | 0 |
| Kota kosong | 18 |
| Deskripsi kosong | 27 |
| Deskripsi tidak informatif | 177 |
| Deskripsi informatif | 821 |
| Koordinat dalam kelompok berulang | 164 |
| URL gambar duplikat | 36 |
| Provinsi dikoreksi berdasarkan alamat | 89 |
| Provinsi perlu verifikasi | 1 |

Kolom harga, rating, dan durasi kunjungan belum memiliki nilai terisi pada dataset nasional yang diperiksa. Nilai yang tidak tersedia tidak dibuat secara sembarangan.

**Catatan lisensi:** Ketentuan lisensi dan izin redistribusi dataset Kaggle perlu diverifikasi sebelum dataset dipublikasikan pada repository publik.
""")

print("Sel 2 berhasil ditambahkan.")

Sel 2 berhasil ditambahkan.


In [19]:
readme_parts.append(r"""
## 5. Metode Pencarian

### 5.1 TF-IDF

TF-IDF (Term Frequency-Inverse Document Frequency) mengubah teks menjadi representasi numerik berdasarkan frekuensi dan tingkat kepentingan kata dalam dokumen.

Secara sederhana:

\[
TFIDF(t,d)=TF(t,d)\times IDF(t)
\]

### 5.2 Cosine Similarity

Cosine Similarity digunakan untuk mengukur kemiripan antara vektor query pengguna dan vektor dokumen destinasi.

\[
Similarity(q,d)=\frac{q\cdot d}{\|q\|\times\|d\|}
\]

Skor yang lebih tinggi menunjukkan kemiripan pola teks yang lebih besar, bukan jaminan bahwa destinasi pasti relevan.

### 5.3 Alur pencarian

1. Memuat dataset destinasi.
2. Menyiapkan teks pencarian.
3. Mengubah teks destinasi menjadi vektor TF-IDF.
4. Menerima query dari pengguna.
5. Mengubah query menjadi vektor TF-IDF.
6. Menghitung Cosine Similarity.
7. Menerapkan filter kategori dan lokasi.
8. Mengurutkan dan menampilkan hasil.

## 6. Pengembangan Algoritma

Eksperimen dilakukan secara bertahap dari V2 sampai V7.

| Versi | Pengembangan |
|---|---|
| V2 | Penambahan skor kecocokan kategori dan lokasi |
| V3 | Penyaringan kategori dan lokasi |
| V4 | Pencocokan nama destinasi |
| V5 | Deteksi kota otomatis |
| V6 | Deteksi provinsi dan kota |
| V7 | Validasi lokasi dan penanganan lokasi yang tidak dikenal |

Eksperimen ini bertujuan meningkatkan kesesuaian hasil pencarian terhadap kategori dan lokasi yang diminta pengguna.

Kode eksperimen dan hasil pengembangan tersedia pada notebook:

`notebooks/pengembangan_dan_evaluasi_algoritma_nasional.ipynb`

## 7. Hasil Evaluasi

### 7.1 Evaluasi relevansi V7

Evaluasi dilakukan menggunakan delapan query dan 56 hasil pencarian yang dinilai secara manual.

| Query | Hasil ditemukan | Relevan | Precision@10 |
|---|---:|---:|---:|
| Gunung dan wisata alam | 10 | 10 | 100% |
| Kebun binatang di Jakarta | 1 | 1 | 10% |
| Museum | 10 | 10 | 100% |
| Museum di Bandung | 4 | 4 | 40% |
| Museum di Semarang | 1 | 1 | 10% |
| Pantai di Bali | 10 | 10 | 100% |
| Tempat wisata di Bali | 10 | 10 | 100% |
| Wisata budaya di Jawa Tengah | 10 | 10 | 100% |
| **Rata-rata** | | | **70%** |

Sebanyak 56 dari 56 hasil yang ditemukan dinilai relevan pada sampel pengujian. Nilai tersebut berbeda dari Precision@10 karena Precision@10 menggunakan pembagi tetap 10, termasuk ketika hasil ditemukan kurang dari 10.

### 7.2 Pengujian provinsi

| Aspek | Hasil |
|---|---:|
| Provinsi diuji | 37 |
| Total hasil pengujian | 289 |
| Hasil sesuai metadata provinsi | 289 |
| Kesesuaian metadata | 100% |

Hasil ini menunjukkan konsistensi penyaringan metadata provinsi dalam skenario pengujian, bukan verifikasi geografis menyeluruh atas semua destinasi.

### 7.3 Perbandingan algoritma

Pengujian V2–V7 menunjukkan perkembangan kemampuan sistem dalam mengenali lokasi dan menangani lokasi yang tidak tersedia.

Pada skenario yang diuji, V7 tidak mengembalikan hasil untuk Atlantis dan Kepulauan Riau ketika tidak terdapat hasil yang sesuai pada dataset.

Hasil tersebut merupakan evaluasi pada skenario terbatas, bukan klaim akurasi umum seluruh sistem.

## 8. Teknologi

| Teknologi | Penggunaan |
|---|---|
| Python | Bahasa pemrograman |
| Pandas | Pengolahan dan pemeriksaan data |
| Scikit-learn | TF-IDF dan Cosine Similarity |
| NumPy | Pengolahan numerik |
| Streamlit | Antarmuka aplikasi web |
| Git | Pengelolaan versi |
| GitHub | Penyimpanan kode dan dokumentasi |
| Anaconda | Pengelolaan environment |
| Jupyter Notebook | Eksperimen dan dokumentasi penelitian |
""")

print("Sel 3 berhasil ditambahkan.")

Sel 3 berhasil ditambahkan.


In [22]:
from pathlib import Path
import shutil

# Lokasi proyek
folder = Path(r"D:\WisataFinder")
file_readme = folder / "README.md"

# Pastikan README tersedia
if not file_readme.exists():
    raise FileNotFoundError("README.md tidak ditemukan.")

# Simpan cadangan
backup = folder / "backups" / "README_sebelum_update.md"
backup.parent.mkdir(exist_ok=True)

if not backup.exists():
    shutil.copy2(file_readme, backup)

# Baca README lama
isi = file_readme.read_text(encoding="utf-8")

# Informasi tambahan untuk pengembangan nasional
tambahan = """
## Pengembangan Dataset Nasional

WisataFinder telah dikembangkan lebih lanjut menggunakan dataset nasional yang berisi 1.025 destinasi wisata dari 37 provinsi.

Dataset nasional yang telah diproses disimpan pada `data/processed/destinations_nasional.csv`. Dataset awal sebanyak 437 destinasi tetap dipertahankan sebagai bagian dari riwayat pengembangan dan pembanding. Kedua dataset tidak digabungkan.

### Pengembangan Algoritma

Eksperimen algoritma dilakukan melalui versi V2 sampai V7 untuk meningkatkan pencarian berdasarkan kategori, nama destinasi, kota, dan provinsi.

Evaluasi V7 pada delapan query menghasilkan rata-rata Precision@10 sebesar 70%. Sebanyak 56 dari 56 hasil yang ditemukan dinilai relevan pada sampel pengujian manual.

Pengujian penyaringan metadata juga dilakukan terhadap 37 provinsi, dengan 289 dari 289 hasil sesuai metadata provinsi yang diminta. Hasil ini bukan klaim akurasi geografis menyeluruh.

### Dokumentasi Penelitian

Penjelasan lengkap mengenai dataset, metodologi, evaluasi, dan perkembangan proyek tersedia pada:

- [Dokumentasi Dataset](docs/DATASET.md)
- [Metodologi Algoritma](docs/METODOLOGI.md)
- [Hasil Evaluasi](docs/EVALUASI.md)
- [Riwayat Pengembangan](docs/PENGEMBANGAN.md)

**Status pengembangan:** Aplikasi nasional sedang melalui validasi fungsional dan integrasi algoritma V7. WisataBot masih menjadi rencana pengembangan berikutnya.

---

"""

# Sisipkan hanya jika belum pernah ditambahkan
if "## Pengembangan Dataset Nasional" not in isi:
    posisi = isi.find("---")

    if posisi == -1:
        raise ValueError("Bagian pemisah README tidak ditemukan.")

    isi_baru = isi[:posisi] + tambahan + isi[posisi:]

    file_readme.write_text(isi_baru, encoding="utf-8")

    print("README.md berhasil diperbarui!")
else:
    print("Informasi pengembangan nasional sudah tersedia.")

# Pemeriksaan
hasil = file_readme.read_text(encoding="utf-8")

print("Jumlah karakter:", len(hasil))
print("Dokumentasi dataset:", "docs/DATASET.md" in hasil)
print("Dokumentasi metodologi:", "docs/METODOLOGI.md" in hasil)
print("Dokumentasi evaluasi:", "docs/EVALUASI.md" in hasil)
print("Dokumentasi pengembangan:", "docs/PENGEMBANGAN.md" in hasil)
print("Cadangan README:", backup)

README.md berhasil diperbarui!
Jumlah karakter: 17606
Dokumentasi dataset: True
Dokumentasi metodologi: True
Dokumentasi evaluasi: True
Dokumentasi pengembangan: True
Cadangan README: D:\WisataFinder\backups\README_sebelum_update.md


In [3]:
from pathlib import Path
import ast
import sys
from importlib import metadata

folder = Path(r"D:\WisataFinder")

file_python = [
    folder / "app.py",
    folder / "app_nasional_v7.py",
    *sorted((folder / "src").glob("*.py"))
]

stdlib = sys.stdlib_module_names
package_mapping = metadata.packages_distributions()

hasil_import = []

for file in file_python:
    if not file.exists():
        continue

    kode = file.read_text(encoding="utf-8-sig")
    tree = ast.parse(kode)

    modul_ditemukan = set()

    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for item in node.names:
                modul_ditemukan.add(item.name.split(".")[0])

        elif isinstance(node, ast.ImportFrom):
            if node.module:
                modul_ditemukan.add(node.module.split(".")[0])

    for modul in sorted(modul_ditemukan):
        if modul in stdlib:
            jenis = "Standard Library"
            distribusi = "-"
        elif modul in package_mapping:
            jenis = "Eksternal"
            distribusi = ", ".join(package_mapping[modul])
        else:
            jenis = "Perlu diperiksa"
            distribusi = "-"

        hasil_import.append({
            "File": file.name,
            "Modul": modul,
            "Jenis": jenis,
            "Distribusi": distribusi
        })

df_import = pd.DataFrame(hasil_import)

display(df_import)

print("Jumlah file Python diperiksa:", len(file_python))
print("Jumlah modul unik:", df_import["Modul"].nunique())

,File,Modul,Jenis,Distribusi
0,app.py,pandas,Perlu diperiksa,-
1,app.py,pathlib,Standard Library,-
2,app.py,re,Standard Library,-
3,app.py,requests,Eksternal,requests
4,app.py,sklearn,Perlu diperiksa,-
5,app.py,streamlit,Eksternal,streamlit
6,app.py,urllib,Standard Library,-
7,app_nasional_v7.py,pandas,Perlu diperiksa,-
8,app_nasional_v7.py,pathlib,Standard Library,-
9,app_nasional_v7.py,re,Standard Library,-


Jumlah file Python diperiksa: 5
Jumlah modul unik: 7


In [2]:
import pandas as pd

print("Pandas berhasil diimpor.")
print("Versi Pandas:", pd.__version__)

Pandas berhasil diimpor.
Versi Pandas: 2.3.3


In [4]:
import sys
import importlib.metadata as metadata

daftar_paket = [
    "pandas",
    "scikit-learn",
    "streamlit",
    "requests"
]

print("VERSI PYTHON:", sys.version.split()[0])
print("\nVERSI PAKET TERINSTAL")

for nama in daftar_paket:
    try:
        versi = metadata.version(nama)
        print(f"{nama}: {versi}")
    except metadata.PackageNotFoundError:
        print(f"{nama}: BELUM TERINSTAL")

VERSI PYTHON: 3.10.20

VERSI PAKET TERINSTAL
pandas: 2.3.3
scikit-learn: 1.7.1
streamlit: 1.65.0
requests: 2.33.1


In [6]:
from pathlib import Path

folder = Path(r"D:\WisataFinder")
file_readme = folder / "README.md"

isi = file_readme.read_text(encoding="utf-8")
baris = isi.splitlines()

# Cari bagian README yang akan diperbarui
awal = [
    i for i, x in enumerate(baris)
    if x.strip() == "## 7. Struktur Repositori"
]

akhir = [
    i for i, x in enumerate(baris)
    if x.strip() == "## 9. Contoh Penggunaan"
]

if len(awal) != 1 or len(akhir) != 1:
    raise ValueError("Judul bagian README tidak ditemukan dengan tepat.")

if awal[0] >= akhir[0]:
    raise ValueError("Urutan bagian README tidak sesuai.")

# Susunan baru bagian struktur dan instalasi
bagian_baru = [
    "## 7. Struktur Repositori",
    "",
    "Struktur utama proyek WisataFinder:",
    "",
    "```text",
    "WisataFinder/",
    "├── app.py",
    "├── app_nasional_v7.py",
    "├── README.md",
    "├── requirements.txt",
    "├── .gitignore",
    "├── .streamlit/",
    "├── data/",
    "│   ├── raw/",
    "│   └── processed/",
    "├── src/",
    "│   ├── inspect_columns.py",
    "│   ├── preprocess_data.py",
    "│   └── validate_data.py",
    "├── notebooks/",
    "├── reports/",
    "│   ├── evaluasi_query_expansion.csv",
    "│   ├── validasi_dataset.txt",
    "│   └── figures/",
    "└── docs/",
    "    ├── DATASET.md",
    "    ├── METODOLOGI.md",
    "    ├── EVALUASI.md",
    "    └── PENGEMBANGAN.md",
    "```",
    "",
    "Keterangan:",
    "",
    "| Bagian | Fungsi |",
    "|---|---|",
    "| app.py | Aplikasi pencarian versi awal |",
    "| app_nasional_v7.py | Aplikasi pengembangan dataset nasional |",
    "| data/raw/ | Dataset sumber |",
    "| data/processed/ | Dataset hasil pengolahan dan evaluasi |",
    "| src/ | Script pemeriksaan dan preprocessing |",
    "| notebooks/ | Eksperimen dan analisis algoritma |",
    "| reports/ | Laporan dan hasil pengujian |",
    "| docs/ | Dokumentasi penelitian |",
    "",
    "Folder tests/ saat ini belum memiliki file pengujian otomatis.",
    "",
    "## 8. Instalasi dan Menjalankan Aplikasi",
    "",
    "### Persyaratan",
    "",
    "- Python 3.10.",
    "- Anaconda atau Miniconda.",
    "- Git.",
    "- Dependensi dalam requirements.txt.",
    "- Dataset yang diperlukan aplikasi.",
    "",
    "### 8.1. Clone repository",
    "",
    "```bash",
    "git clone https://github.com/siskamuthoharoh/WisataFinder.git",
    "cd WisataFinder",
    "```",
    "",
    "### 8.2. Membuat environment",
    "",
    "```bash",
    "conda create -n wisatafinder python=3.10 -y",
    "conda activate wisatafinder",
    "```",
    "",
    "### 8.3. Menginstal dependensi",
    "",
    "```bash",
    "python -m pip install -r requirements.txt",
    "```",
    "",
    "### 8.4. Menjalankan aplikasi awal",
    "",
    "```bash",
    "streamlit run app.py",
    "```",
    "",
    "Aplikasi awal menggunakan destinations_clean.csv.",
    "",
    "### 8.5. Menjalankan aplikasi nasional",
    "",
    "```bash",
    "streamlit run app_nasional_v7.py",
    "```",
    "",
    "Aplikasi nasional menggunakan destinations_nasional.csv.",
    "",
    "**Catatan:** Aplikasi nasional masih dalam tahap validasi.",
    "Pastikan dataset dan dependensi tersedia sebelum menjalankan aplikasi.",
    ""
]

# Gabungkan dengan bagian README yang tidak diubah
hasil_baru = (
    baris[:awal[0]]
    + bagian_baru
    + baris[akhir[0]:]
)

teks_baru = "\n".join(hasil_baru) + "\n"

# Periksa sebelum menyimpan
assert teks_baru.count("## 7. Struktur Repositori") == 1
assert teks_baru.count("## 8. Instalasi dan Menjalankan Aplikasi") == 1
assert teks_baru.count("## 9. Contoh Penggunaan") == 1

file_readme.write_text(teks_baru, encoding="utf-8")

print("README berhasil diperbarui.")
print("Ukuran:", round(file_readme.stat().st_size / 1024, 2), "KB")
print("Pemeriksaan struktur: LULUS")

README berhasil diperbarui.
Ukuran: 16.65 KB
Pemeriksaan struktur: LULUS


In [7]:
print(file_readme.read_text(encoding="utf-8")[
    file_readme.read_text(encoding="utf-8").find("## 7. Struktur Repositori"):
    file_readme.read_text(encoding="utf-8").find("## 9. Contoh Penggunaan")
])

## 7. Struktur Repositori

Struktur utama proyek WisataFinder:

```text
WisataFinder/
├── app.py
├── app_nasional_v7.py
├── README.md
├── requirements.txt
├── .gitignore
├── .streamlit/
├── data/
│   ├── raw/
│   └── processed/
├── src/
│   ├── inspect_columns.py
│   ├── preprocess_data.py
│   └── validate_data.py
├── notebooks/
├── reports/
│   ├── evaluasi_query_expansion.csv
│   ├── validasi_dataset.txt
│   └── figures/
└── docs/
    ├── DATASET.md
    ├── METODOLOGI.md
    ├── EVALUASI.md
    └── PENGEMBANGAN.md
```

Keterangan:

| Bagian | Fungsi |
|---|---|
| app.py | Aplikasi pencarian versi awal |
| app_nasional_v7.py | Aplikasi pengembangan dataset nasional |
| data/raw/ | Dataset sumber |
| data/processed/ | Dataset hasil pengolahan dan evaluasi |
| src/ | Script pemeriksaan dan preprocessing |
| notebooks/ | Eksperimen dan analisis algoritma |
| reports/ | Laporan dan hasil pengujian |
| docs/ | Dokumentasi penelitian |

Folder tests/ saat ini belum memiliki file pengujian o

In [8]:
from pathlib import Path

file_readme = Path(r"D:\WisataFinder\README.md")
isi = file_readme.read_text(encoding="utf-8")

print("Jumlah blok kode:", isi.count("```") // 2)
print("Jumlah pembuka blok kode:", isi.count("```"))
print("Ada karakter escape sebelum heading:", r"\##" in isi)
print("Ada karakter escape sebelum blok kode:", r"\`\`\`" in isi)

print("\nDaftar heading:")
for baris in isi.splitlines():
    if baris.startswith("#"):
        print(baris)

Jumlah blok kode: 7
Jumlah pembuka blok kode: 14
Ada karakter escape sebelum heading: False
Ada karakter escape sebelum blok kode: False

Daftar heading:
# WisataFinder — Mesin Pencari Destinasi Wisata
## Pengembangan Dataset Nasional
### Pengembangan Algoritma
### Dokumentasi Penelitian
## Daftar Isi
## 1. Latar Belakang
## 2. Tujuan Proyek
## 3. Fitur Aplikasi
### 3.1. Pencarian berdasarkan kata kunci
### 3.2. Pemeringkatan hasil pencarian
### 3.3. Filter lokasi dan kategori
### 3.4. Pengurutan berdasarkan harga
### 3.5. Tampilan hasil berbasis web
## 4. Dataset
### Peran data dalam sistem
### Sumber dan atribusi
## 5. Metode dan Alur Sistem
### 5.1. Validasi dan preprocessing data
### 5.2. Pembobotan TF-IDF
### 5.3. Cosine Similarity
### 5.4. Pemeringkatan hasil
### 5.5. Diagram alur sistem
## 6. Teknologi yang Digunakan
## 7. Struktur Repositori
## 8. Instalasi dan Menjalankan Aplikasi
### Persyaratan
### 8.1. Clone repository
### 8.2. Membuat environment
### 8.3. Menginstal depend

In [1]:
from pathlib import Path

folder = Path(r"D:\WisataFinder\data")
folder.mkdir(exist_ok=True)

isi = """# Dataset WisataFinder

Repository ini tidak menyertakan salinan dataset nasional
karena ketentuan redistribusi dari sumber dataset masih
menunggu konfirmasi.

## Dataset nasional

Sumber:
https://www.kaggle.com/datasets/sagitasantia/indonesia-tourism

Untuk menjalankan aplikasi nasional:

1. Unduh dataset dari sumber aslinya setelah memeriksa ketentuan penggunaan.
2. Simpan file sumber sebagai:
   data/raw/wisata_indonesia_new.csv
3. Jalankan tahapan pengolahan pada notebook:
   notebooks/pengembangan_dan_evaluasi_algoritma_nasional.ipynb
4. Pastikan hasil pengolahan tersedia sebagai:
   data/processed/destinations_nasional.csv
5. Jalankan aplikasi nasional menggunakan Streamlit.

Dataset awal dan dataset nasional merupakan dua sumber
berbeda dan tidak digabungkan dalam pengembangan ini.
"""

file = folder / "README.md"
file.write_text(isi, encoding="utf-8")

print("Berhasil membuat:", file)

Berhasil membuat: D:\WisataFinder\data\README.md


In [1]:
from pathlib import Path

file_app_v7 = Path(r"D:\WisataFinder\app_nasional_v7.py")
baris = file_app_v7.read_text(encoding="utf-8").splitlines()

awal = next(
    i for i, x in enumerate(baris)
    if "# Deteksi provinsi dari kata kunci." in x
)

akhir = next(
    i for i, x in enumerate(baris)
    if "# Hitung TF-IDF dan cosine similarity" in x
)

for i in range(awal, akhir):
    print(f"{i + 1}: {baris[i]}")

1194:         # Deteksi provinsi dari kata kunci.
1195:         detected_province = None
1196:         detected_province_term = None
1197: 
1198:         for province_name in sorted(
1199:             province_options,
1200:             key=len,
1201:             reverse=True,
1202:         ):
1203:             pola = rf"(?<!\w){re.escape(province_name.casefold())}(?!\w)"
1204:             cocok = re.search(pola, query_lower)
1205: 
1206:             if cocok:
1207:                 detected_province = province_name
1208:                 detected_province_term = cocok.group(0)
1209:                 break
1210: 
1211:         # Alias untuk beberapa nama provinsi.
1212:         province_aliases = {
1213:             "Daerah Istimewa Yogyakarta": [
1214:                 "diy", "jogja", "yogya"
1215:             ],
1216:             "DKI Jakarta": [
1217:                 "dki", "jakarta"
1218:             ],
1219:             "Jawa Barat": [
1220:                 "jabar"
1221:             ]

In [3]:
from pathlib import Path

file_app_v7 = Path(r"D:\WisataFinder\app_nasional_v7.py")

# Baca ulang file dari disk
kode = file_app_v7.read_text(encoding="utf-8")

penanda = "        # Hitung TF-IDF dan cosine similarity"

if kode.count(penanda) != 1:
    raise ValueError("Penanda TF-IDF tidak ditemukan tepat satu kali.")

# Susun kode validasi
validasi = [
    "        # Validasi lokasi yang disebutkan pengguna.",
    '        frasa_lokasi = re.search(r"\\bdi\\s+(.+?)\\s*$", query_lower)',
    "",
    "        if (",
    "            frasa_lokasi",
    "            and detected_province is None",
    "            and detected_city is None",
    "        ):",
    "            lokasi_diminta = frasa_lokasi.group(1).strip()",
    '            if lokasi_diminta not in {"indonesia", "nusantara"}:',
    "                st.warning(",
    '                    f"Lokasi {lokasi_diminta!r} belum dikenali atau tidak tersedia dalam dataset WisataFinder."',
    "                )",
    "                st.info(",
    '                    "Periksa kembali nama provinsi atau kota/kabupaten."',
    "                )",
    "                st.stop()",
    "",
]

blok_validasi = "\n".join(validasi) + "\n"

# Sisipkan sebelum perhitungan TF-IDF
kode_baru = kode.replace(
    penanda,
    blok_validasi + penanda,
    1
)

# Periksa sintaks sebelum menyimpan
compile(kode_baru, str(file_app_v7), "exec")

# Simpan jika pemeriksaan berhasil
file_app_v7.write_text(kode_baru, encoding="utf-8")

print("Validasi lokasi berhasil ditambahkan.")
print("Pemeriksaan sintaks: LULUS")
print("File:", file_app_v7)

Validasi lokasi berhasil ditambahkan.
Pemeriksaan sintaks: LULUS
File: D:\WisataFinder\app_nasional_v7.py


In [2]:
from pathlib import Path

file_app_v7 = Path(r"D:\WisataFinder\app_nasional_v7.py")
baris = file_app_v7.read_text(encoding="utf-8").splitlines()

for i, teks in enumerate(baris):
    if "Validasi lokasi yang disebutkan pengguna" in teks:
        print("=== BAGIAN VALIDASI ===")
        for nomor in range(i, min(i + 30, len(baris))):
            print(f"{nomor + 1}: {baris[nomor]!r}")
        break
else:
    print("Bagian validasi tidak ditemukan.")

=== BAGIAN VALIDASI ===
1428: '        # Validasi lokasi yang disebutkan pengguna.'
1429: '        frasa_lokasi = re.search(r"\\bdi\\s+(.+?)\\s*$", query_lower)'
1430: ''
1431: '        if ('
1432: '            frasa_lokasi'
1433: '            and detected_province is None'
1434: '            and detected_city is None'
1435: '        ):'
1436: '            lokasi_diminta = frasa_lokasi.group(1).strip()'
1437: '            if lokasi_diminta not in {"indonesia", "nusantara"}:'
1438: '                st.warning('
1439: '                    f"Lokasi {lokasi_diminta!r} belum dikenali atau tidak tersedia dalam dataset WisataFinder."'
1440: '                )'
1441: '                st.info('
1442: '                    "Periksa kembali nama provinsi atau kota/kabupaten."'
1443: '                )'
1444: '                st.stop()'
1445: ''
1446: '        # Hitung TF-IDF dan cosine similarity'
1447: '        query_vector = vectorizer.transform([search_query])'
1448: ''
1449: '        scores = 

In [3]:
from pathlib import Path
import shutil

file_app_v7 = Path(r"D:\WisataFinder\app_nasional_v7.py")

kode = file_app_v7.read_text(encoding="utf-8")

# Tentukan bagian validasi
awal_validasi = "# Validasi lokasi yang disebutkan pengguna."
akhir_validasi = "# Hitung TF-IDF dan cosine similarity"

if kode.count(awal_validasi) != 1 or kode.count(akhir_validasi) != 1:
    raise ValueError("Bagian validasi tidak ditemukan dengan tepat.")

pos_awal = kode.index(awal_validasi)
pos_akhir = kode.index(akhir_validasi, pos_awal)

bagian = kode[pos_awal:pos_akhir]

# Pastikan kode peringatan yang lama memang tersedia
awal_pesan = "                st.warning("
akhir_pesan = "                st.stop()"

if bagian.count(awal_pesan) != 1 or bagian.count(akhir_pesan) != 1:
    raise ValueError("Bagian pesan peringatan tidak cocok.")

pos_pesan = bagian.index(awal_pesan)
pos_stop = bagian.index(akhir_pesan, pos_pesan)

pesan_baru = [
    "                st.markdown(",
    "                    (",
    """                        '<div style="background-color:#FFF3CD; color:#000000; '""",
    """                        'font-weight:700; padding:14px 16px; '""",
    """                        'border:1px solid #FFECB5; border-radius:10px; line-height:1.6;">'""",
    """                        '<strong>Lokasi tidak ditemukan</strong><br>'""",
    """                        f'Lokasi {lokasi_diminta!r} belum dikenali atau tidak tersedia '""",
    """                        'dalam dataset WisataFinder.<br>'""",
    """                        'Periksa kembali nama provinsi atau kota/kabupaten.'""",
    """                        '</div>'""",
    "                    ),",
    "                    unsafe_allow_html=True,",
    "                )",
    "                st.stop()",
]

bagian_baru = (
    bagian[:pos_pesan]
    + "\n".join(pesan_baru)
    + "\n"
)

kode_baru = (
    kode[:pos_awal]
    + bagian_baru
    + kode[pos_akhir:]
)

# Periksa sintaks sebelum menyimpan
compile(kode_baru, str(file_app_v7), "exec")

# Simpan cadangan
backup = file_app_v7.with_name(
    "app_nasional_v7_sebelum_teks_bold_hitam.py.bak"
)

if not backup.exists():
    shutil.copy2(file_app_v7, backup)

# Simpan perubahan
file_app_v7.write_text(kode_baru, encoding="utf-8")

print("Tampilan peringatan berhasil diperbaiki.")
print("Warna teks: Hitam (#000000)")
print("Ketebalan teks: Bold")
print("Pemeriksaan sintaks: LULUS")

Tampilan peringatan berhasil diperbaiki.
Warna teks: Hitam (#000000)
Ketebalan teks: Bold
Pemeriksaan sintaks: LULUS
